# **Introduction**

This notebook presents a fully reproducible and universally applicable pipeline for quantitative **microglial** morphometric analysis.  
Although the workflow is technically compatible with any single‑cell dataset, it has been specifically optimized and documented for **microglia**, the resident immune cells of the central nervous system.  

Microglia exhibit a rich repertoire of morphological states—ranging from highly ramified homeostatic forms to compact ameboid phenotypes during activation.  
Because morphology is tightly linked to function, a robust and transparent morphometric pipeline is essential for studying microglial surveillance, activation, priming, phagocytosis, and resolution.

Emphasis is placed on **modularity** and **robustness** across heterogeneous datasets, imaging conditions, and activation states.

The analytical workflow is organized into clearly defined modules, each responsible for a specific stage of processing.  
This modular structure enables users to understand, modify, or extend individual components without disrupting the integrity of the entire pipeline.

---

### **Notebook Structure**

The notebook is organized into a sequence of self‑contained modules, each performing a distinct analytical task.  
To support both real and synthetic datasets, the workflow includes an optional preliminary module:

**Module -1 — Project Setup and Overview**  
Initial configuration of the environment, imports, and description of the workflow.

**Module 0 — Optional Synthetic Image Generation for High‑Resolution Microglial Morphometry**  
Provides a controlled, high‑quality synthetic dataset representing several canonical microglial activation states (resting, primed, activated, ameboid, resolving).  
These synthetic images serve as a pilot dataset for validating the entire morphometric pipeline.  
Researchers may freely substitute or complement them with their own experimental microglia.

**Module 1 — Dataset Loader**  
Automatically scans a user‑defined directory and loads all microglial images it contains.  
This module is fully dataset‑independent and supports any number of images.

**Module 2 — Core Image Processing Functions**  
Implements the essential preprocessing steps, including loading, thresholding, segmentation, skeletonization, bounding‑box detection, and optional scale normalization.

**Module 3 — Microglia preprocessing**  
Computes the fractal dimension of each microglial arbor, capturing global complexity and branching richness.

**Module 4 — Lacunarity Curves**  
Quantifies structural heterogeneity and gap distribution across spatial scales—particularly relevant for distinguishing ramified vs. compact microglial states.

**Module 5 — Gray‑level Co‑occurrence Matrix (GLCM) - Based Texture Metrics**  
Extracts gray‑level co‑occurrence matrix (GLCM) features to characterize local texture patterns, which often change during microglial activation.

**Module 6 — Multiscale Fractal Spectrum (MFS)**  
Computes the multiscale fractal spectrum (MFS) to capture complexity across multiple spatial scales.

**Module 7 — Graph‑Based Morphometrics**  
Converts microglial skeletons into graphs and extracts topological descriptors such as node degree, branching patterns, connectivity, and arbor geometry.

**Module 8 — Batch Sholl Analysis**  
Performs radial intersection analysis to quantify arbor complexity and spatial organization—an essential descriptor of microglial ramification.

**Module 9 — Statistical Testing**  
Evaluates group differences between microglial activation states using appropriate parametric or non‑parametric tests, including effect size estimation and FDR correction.

**Module 10 — Dimensionality Reduction & Feature Importance**  
Applies PCA and UMAP for global and nonlinear structure discovery, and uses Random Forest and SHAP to identify the most discriminative morphometric features across microglial states.

---

### **Why This Notebook Is Universal**

This notebook is intentionally designed to be:

**1. Dataset‑independent (within microglial imaging)**  
The pipeline does not assume any specific number of images, naming conventions, or activation states.  
All modules automatically adapt to the dataset size and/or groups.

**2. Modular and Extensible**  
Each analytical step is isolated in its own module.  
Users may replace or extend individual components without affecting the rest of the workflow.

**3. Reproducible**  
All preprocessing steps are deterministic and explicitly defined.  
No hidden parameters, uncontrolled randomness, or environment‑dependent behaviors are used.

**4. Transparent**  
Every transformation is visualized and documented.  
Researchers can verify each stage before proceeding to quantitative analysis.

---

### **Important Notes for Users**

To ensure smooth execution:

1. **Run the modules in order.**  
   Later modules depend on functions and variables defined earlier.

2. **Dataset path.**  
   Window pop‑up will appear to select the images folder. If this does not work, the only required user edit is, if necessary:  
   `DATASET_DIR = r"your/path/here/dataset"`  
   Internal functions and variables should not be altered.

3. **Ensure the dataset folder contains only images.**  
   Non‑image files may cause loading errors.

4. **Visual modules scale automatically.**  
   All plotting functions adapt dynamically to the number of images.

5. **Small or homogeneous datasets may trigger statistical warnings.**  
   These do not indicate errors and do not affect execution.

6. **CSV files use dot decimals and comma separators.**  
   All `.csv` files generated by the application use the standard scientific format: dot as the decimal separator (`.`) and comma as the field separator (`,`).  
   Depending on your regional settings, spreadsheet software such as Excel may display some numeric values incorrectly, especially very small p-values written in scientific notation.  
   For reliable visualization in Excel, we recommend opening the corresponding `.xlsx` files when available, or importing the `.csv` manually and selecting comma as the delimiter and dot as the decimal separator.
---

### **Goal of This Notebook**

The objective of this notebook is to provide a reproducible, transparent, and universally applicable **microglial morphometric analysis framework** that can be used by any researcher, regardless of dataset size, activation state, or computational expertise.  
Its modular design ensures that the workflow can be easily adapted to new datasets, extended with additional metrics, or integrated into larger analytical frameworks.

As a practical demonstration, the notebook includes a synthetic microglia generator (Module 0) that produces several canonical activation states.  
These synthetic images serve as a pilot dataset for executing the entire morphometric pipeline end‑to‑end.  
Although synthetic microglia are used here as an example, the workflow remains fully compatible with any experimental dataset containing isolated microglial cells acquired at appropriate resolution.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">

## **Module -1 — Project Setup and Overview**

**Notebook Initialization — Dependency Check & Conditional Install/Upgrade**

This module prepares the execution environment for the entire microglial morphometric analysis pipeline.
It verifies that the required Python packages are installed, checks whether their versions meet minimum requirements, and automatically installs or upgrades them when needed.

In addition to dependency setup, this module:

- Imports the core scientific, statistical, imaging, and visualization libraries used throughout the pipeline
- Reports package and Python version information for reproducibility
- Detects the availability of key imaging libraries such as OpenCV, scikit-image, and Pillow
- Creates the output directories required for downstream analyses, including exports/ and synthetic_microglia_40x/
- Configures figure export settings for consistent saving of generated plots
- Defines helper plotting utilities used later in the notebook

Because this notebook may be executed in heterogeneous environments, the initialization step is designed to be as robust as possible.
Rather than assuming that all dependencies are already available, the script checks each required package against a minimum version and attempts to install or upgrade it when necessary.

This module provides the computational foundation for all downstream analyses in the pipeline, including preprocessing, segmentation, skeletonization, morphometric quantification, statistical testing, dimensionality reduction, and visualization.

Run this cell before executing any other section of the notebook.

---

### **Notes on Dependency Handling**

During initialization, the notebook checks the availability and version of all required packages.
If a package is missing or outdated, the script attempts to install or upgrade it automatically using pip.

Some imaging libraries, such as OpenCV, Pillow, and scikit-image, are also checked explicitly so their availability can be reported to the user.
Depending on the execution environment, installation may fail if package management is restricted or internet access is unavailable. In such cases, the notebook may require manual installation of the missing dependencies before continuing.

This initialization step is intended to improve portability and reproducibility across local machines, institutional servers, and cloud-based environments

In [ ]:
# ============================================================
# Notebook Initialization — Dependency Check & Conditional Install/Upgrade
# ============================================================

print("[INFO] ==== Environment initialized ====")
# ============================================================
# Imports 1
# ============================================================

import platform
import os
import sys
import glob
import math
import random
import warnings
import subprocess
import tkinter as tk
from tkinter import filedialog
from collections import defaultdict

# Standard library for installed package version lookup
from importlib.metadata import version as get_installed_version, PackageNotFoundError

# ------------------------------------------------------------
# Helper function: pip install / upgrade
# ------------------------------------------------------------
def pip_install_or_upgrade(package_spec):
    print(f"[INFO] Installing/upgrading {package_spec} ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", package_spec
    ])

# ------------------------------------------------------------
# Bootstrap packaging if missing
# Needed BEFORE using packaging.version.Version
# ------------------------------------------------------------
try:
    from packaging.version import Version
except ImportError:
    print("[WARNING] 'packaging' not found. Installing packaging...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", "packaging"
    ])
    from packaging.version import Version

# ------------------------------------------------------------
# Conditional install/upgrade:
# only install if missing OR upgrade if below minimum version
# ------------------------------------------------------------
def ensure_minimum_version(pip_name, min_version):
    """
    Ensure that a package is installed and its version is >= min_version.

    Behavior:
      - if package is missing -> install
      - if installed version < min_version -> upgrade
      - if installed version >= min_version -> do nothing
    """
    try:
        installed_version = get_installed_version(pip_name)
        print(f"[INFO] {pip_name} installed version: {installed_version}")

        if Version(installed_version) < Version(min_version):
            print(f"[INFO] {pip_name} is below minimum required version ({min_version}) -> upgrading...")
            pip_install_or_upgrade(f"{pip_name}>={min_version}")
        else:
            print(f"[INFO] {pip_name} already satisfies minimum version ({min_version}). Skipping.")
    except PackageNotFoundError:
        print(f"[WARNING] {pip_name} is not installed -> installing...")
        pip_install_or_upgrade(f"{pip_name}>={min_version}")

# ------------------------------------------------------------
# Core tools with minimum versions
# ------------------------------------------------------------
CORE_TOOLS = {
    "pip": "26.0.1",
    "setuptools": "82.0.1",
    "wheel": "0.46.3",
    "packaging": "26.0",
}

print(f"[INFO] Python version: {platform.python_version()}")

print("\n[INFO] Checking core packaging tools...")
for tool_name, min_version in CORE_TOOLS.items():
    ensure_minimum_version(tool_name, min_version)

# ------------------------------------------------------------
# Required Python packages for this notebook
# module_name is informative only here
# pip = package name in pip
# min_version = minimum acceptable version
# ------------------------------------------------------------
REQUIRED_PACKAGES = {
    "numpy": {"pip": "numpy", "min_version": "1.24.0"},
    "pandas": {"pip": "pandas", "min_version": "2.0.0"},
    "matplotlib": {"pip": "matplotlib", "min_version": "3.7.0"},
    "scipy": {"pip": "scipy", "min_version": "1.10.0"},
    "skimage": {"pip": "scikit-image", "min_version": "0.21.0"},
    "sklearn": {"pip": "scikit-learn", "min_version": "1.3.0"},
    "pingouin": {"pip": "pingouin", "min_version": "0.5.3"},
    "statsmodels": {"pip": "statsmodels", "min_version": "0.14.0"},
    "networkx": {"pip": "networkx", "min_version": "3.1"},
    "seaborn": {"pip": "seaborn", "min_version": "0.12.2"},
    "tqdm": {"pip": "tqdm", "min_version": "4.66.0"},
    "scikit_posthocs": {"pip": "scikit-posthocs", "min_version": "0.7.0"},
    "umap": {"pip": "umap-learn", "min_version": "0.5.4"},
    "PIL": {"pip": "pillow", "min_version": "9.5.0"},
    "cv2": {"pip": "opencv-python", "min_version": "4.8.0.76"},
    "shap": {"pip": "shap", "min_version": "0.45.0"},
    "openpyxl": {"pip": "openpyxl", "min_version": "3.1.0"},
}

print("\n[INFO] Checking required packages against minimum versions...")
for module_name, pkg_info in REQUIRED_PACKAGES.items():
    ensure_minimum_version(
        pip_name=pkg_info["pip"],
        min_version=pkg_info["min_version"]
    )

# ------------------------------------------------------------
# Verify OpenCV  and Skimage validation
# ------------------------------------------------------------

try:
    import cv2
    OPENCV_AVAILABLE = True
except ImportError:
    OPENCV_AVAILABLE = False

# Verify scikit-image availability
try:
    import skimage
    SKIMAGE_AVAILABLE = True
except ImportError:
    SKIMAGE_AVAILABLE = False

print(f"[INFO] OpenCV available: {OPENCV_AVAILABLE}")
print(f"[INFO] scikit-image available: {SKIMAGE_AVAILABLE}")
# ============================================================
# Imports 2
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pingouin as pg
import umap.umap_ as umap
import shap

try:
    from PIL import Image
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

from scipy.ndimage import convolve
from scipy.stats import (
    shapiro,
    levene,
    ttest_ind,
    f_oneway,
    mannwhitneyu,
    kruskal
)

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

from skimage import io, exposure
from skimage.color import rgb2gray
from skimage.draw import line
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import gaussian, threshold_otsu, threshold_local
from skimage.measure import label, regionprops
from skimage.morphology import skeletonize, remove_small_objects
from skimage.transform import resize
from skimage.util import img_as_ubyte

from statsmodels.stats.multitest import multipletests
from statsmodels.stats.diagnostic import lilliefors
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import scikit_posthocs as sp

# ------------------------------------------------------------
# Environment info
# ------------------------------------------------------------
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", plt.matplotlib.__version__)
print("seaborn version:", sns.__version__)
print("scikit-image version:", skimage.__version__)
print("OpenCV version:", cv2.__version__)
print("Python executable:", sys.executable)

print("\n[INFO] ===== Starting pipeline =====")

# ------------------------------------------------------------
# Create exports folder
# ------------------------------------------------------------


# Get the directory where THIS script is located

def get_base_dir():
    if getattr(sys, "frozen", False):
        return Path(sys.executable).resolve().parent
    return Path(__file__).resolve().parent

SCRIPT_DIR = get_base_dir()

EXPORT_DIR = SCRIPT_DIR / "exports"
OUTPUT_DIR = SCRIPT_DIR / "synthetic_microglia_40x"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("[INFO] Script directory:", SCRIPT_DIR)
print("[INFO] EXPORT_DIR:", EXPORT_DIR)
print("[INFO] OUTPUT_DIR:", OUTPUT_DIR)

# ============================================================
# Universal table exporter
# ============================================================
# Every time the script writes a .csv file, it also writes a
# matching .xlsx file. The .xlsx file is recommended for Excel
# users because it avoids regional decimal/separator problems.
# ============================================================

def _safe_excel_sheet_name(name="Data"):
    """
    Excel sheet names cannot exceed 31 characters and cannot contain
    some special characters.
    """
    invalid_chars = ["\\", "/", "*", "[", "]", ":", "?"]
    name = str(name)

    for ch in invalid_chars:
        name = name.replace(ch, "_")

    if not name.strip():
        name = "Data"

    return name[:31]


def _format_excel_sheet(ws, df):
    """
    Basic formatting for Excel exports:
    - freeze header row
    - add autofilter
    - adjust column widths
    - use scientific notation for p-values
    - preserve numeric values as real numbers
    """
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    pvalue_like_columns = {
        "p_value",
        "p_adj",
        "p_adj_global",
        "p_adj_family",
        "levene_p"
    }

    for col_idx, col_name in enumerate(df.columns, start=1):
        col_letter = ws.cell(row=1, column=col_idx).column_letter
        col_name_str = str(col_name)
        col_name_lower = col_name_str.lower()

        # Estimate readable column width
        try:
            sample_values = df[col_name].astype(str).head(200).tolist()
            max_len = max([len(col_name_str)] + [len(v) for v in sample_values])
        except Exception:
            max_len = len(col_name_str)

        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 50)

        # Numeric formatting
        try:
            is_numeric = pd.api.types.is_numeric_dtype(df[col_name])
            is_integer = pd.api.types.is_integer_dtype(df[col_name])
            is_float = pd.api.types.is_float_dtype(df[col_name])
        except Exception:
            is_numeric = False
            is_integer = False
            is_float = False

        if is_numeric:
            if (
                col_name_lower in pvalue_like_columns
                or col_name_lower.endswith("_p")
                or "p_value" in col_name_lower
                or "p_adj" in col_name_lower
            ):
                number_format = "0.000E+00"
            elif is_integer:
                number_format = "0"
            elif is_float:
                number_format = "0.000000"
            else:
                number_format = "General"

            for row_idx in range(2, ws.max_row + 1):
                ws.cell(row=row_idx, column=col_idx).number_format = number_format


def _write_xlsx_copy_from_dataframe(df, csv_path):
    """
    Create an .xlsx copy next to a .csv file.
    """
    csv_path = Path(csv_path)
    xlsx_path = csv_path.with_suffix(".xlsx")

    try:
        sheet_name = _safe_excel_sheet_name(csv_path.stem)

        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )

            ws = writer.sheets[sheet_name]
            _format_excel_sheet(ws, df)

        print(f"[INFO] Excel copy saved: {xlsx_path}")

    except Exception as e:
        print(f"[WARNING] Could not create Excel copy for {csv_path}: {e}")


# ------------------------------------------------------------
# Patch pandas DataFrame.to_csv once
# ------------------------------------------------------------
# This keeps all existing df.to_csv(...) calls working normally,
# but automatically creates a matching .xlsx file.
# ------------------------------------------------------------

if not hasattr(pd.DataFrame, "_original_to_csv_for_dual_export"):
    pd.DataFrame._original_to_csv_for_dual_export = pd.DataFrame.to_csv

    def _to_csv_and_xlsx(self, path_or_buf=None, *args, **kwargs):
        # First, write the normal CSV exactly as before
        result = pd.DataFrame._original_to_csv_for_dual_export(
            self,
            path_or_buf,
            *args,
            **kwargs
        )

        # Then, if the output is a CSV file path, create an XLSX copy
        if path_or_buf is not None:
            try:
                output_path = Path(path_or_buf)

                if output_path.suffix.lower() == ".csv":
                    _write_xlsx_copy_from_dataframe(self, output_path)

            except Exception as e:
                print(f"[WARNING] Automatic XLSX export failed for {path_or_buf}: {e}")

        return result

    pd.DataFrame.to_csv = _to_csv_and_xlsx

# Automatically save every figure generated in the notebook
plt.rcParams["savefig.directory"] = os.getcwd()
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["savefig.format"] = "png"

print(f"[INFO] OpenCV available: {OPENCV_AVAILABLE}")
print(f"[INFO] scikit-image available: {SKIMAGE_AVAILABLE}")
print(f"[INFO] PIL available: {PIL_AVAILABLE}")

# ============================================================
# Plotting guide
# ============================================================

def lighten_color(color, amount=0.6):
    """
    Lighten a matplotlib color.
    amount=0 -> original color
    amount=1 -> white
    """
    import colorsys
    import matplotlib.colors as mc

    try:
        c = mc.cnames[color]
    except Exception:
        c = color

    r, g, b = mc.to_rgb(c)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = 1 - amount * (1 - l)
    return colorsys.hls_to_rgb(h, l, s)


def should_use_boxplot(df, group_col="cell_state", image_threshold=5, group_threshold=5):
    """
    Switch from per-image bars to grouped boxplot + points when:
      - number of groups > group_threshold
      - OR any group has more than image_threshold images
    """
    if df.empty or group_col not in df.columns:
        return False

    n_groups = df[group_col].nunique(dropna=True)
    max_images_in_group = df[group_col].value_counts(dropna=True).max()

    return (n_groups > group_threshold) or (max_images_in_group > image_threshold)


def build_group_color_map(groups):
    return {
        state: plt.cm.tab10(i % 10)
        for i, state in enumerate(groups)
    }


def plot_bars_or_boxplot(
    df,
    value_col,
    group_col="cell_state",
    filename_col="sample_id",
    state_order=None,
    ylabel=None,
    xlabel=None,
    title=None,
    figsize_bar=(14, 6),
    figsize_box=(10, 6),
    save_path=None,
    image_threshold=5,
    group_threshold=5,
    show_legend=True
):
    """
    Universal plotting function:
      - small dataset -> bar plot per image
      - large dataset -> boxplot by group + individual points
    """
    df_plot = df.copy().dropna(subset=[value_col, group_col])

    if df_plot.empty:
        print(f"[WARNING] No valid data to plot for: {value_col}")
        return

    # Group ordering
    if state_order is None:
        groups = list(df_plot[group_col].dropna().unique())
    else:
        groups = [g for g in state_order if g in df_plot[group_col].dropna().unique()]
        groups += [g for g in df_plot[group_col].dropna().unique() if g not in groups]

    df_plot[group_col] = pd.Categorical(df_plot[group_col], categories=groups, ordered=True)

    sort_cols = [group_col]
    if filename_col in df_plot.columns:
        sort_cols.append(filename_col)

    df_plot = df_plot.sort_values(sort_cols).reset_index(drop=True)

    color_map = build_group_color_map(groups)
    use_box = should_use_boxplot(
        df_plot,
        group_col=group_col,
        image_threshold=image_threshold,
        group_threshold=group_threshold
    )

    # --------------------------------------------------------
    # CASE 1 — bar plot per image
    # --------------------------------------------------------
    if not use_box:
        df_plot["color"] = df_plot[group_col].astype(object).map(color_map)

        plt.figure(figsize=figsize_bar)

        plt.bar(
            x=np.arange(len(df_plot)),
            height=df_plot[value_col],
            color=df_plot["color"],
            edgecolor="black",
            linewidth=0.7
        )

        if filename_col in df_plot.columns:
            plt.xticks(
                ticks=np.arange(len(df_plot)),
                labels=df_plot[filename_col],
                rotation=75,
                ha="right",
                fontsize=8
            )
        else:
            plt.xticks(np.arange(len(df_plot)))

        plt.ylabel(ylabel if ylabel else value_col)
        plt.xlabel(xlabel if xlabel else "Image Filename")
        plt.title(title if title else value_col)

        if show_legend:
            handles = [
                plt.Line2D([0], [0], color=color_map[state], lw=8, label=state)
                for state in groups
            ]
            plt.legend(handles=handles, title="Group", bbox_to_anchor=(1.05, 1), loc="upper left")

        ax = plt.gca()
        ax.grid(axis="y", linestyle="--", alpha=0.3)

        plt.tight_layout()

        if save_path is not None:
            plt.savefig(save_path, dpi=600, bbox_inches="tight")

        plt.show()
        return

    # --------------------------------------------------------
    # CASE 2 — boxplot + individual points
    # --------------------------------------------------------
    plt.figure(figsize=figsize_box)

    light_palette = {g: lighten_color(color_map[g], 0.6) for g in groups}

    sns.boxplot(
        data=df_plot,
        x=group_col,
        y=value_col,
        hue=group_col,
        order=groups,
        palette=light_palette,
        dodge=False,
        width=0.55,
        showfliers=False,
        linewidth=1.2
    )

    sns.stripplot(
        data=df_plot,
        x=group_col,
        y=value_col,
        hue=group_col,
        order=groups,
        palette=color_map,
        dodge=False,
        jitter=0.12,
        alpha=0.9,
        size=5,
        edgecolor="black",
        linewidth=0.4
    )

    ax = plt.gca()
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

    ax.grid(axis="y", linestyle="--", alpha=0.3)

    plt.ylabel(ylabel if ylabel else value_col)
    plt.xlabel(xlabel if xlabel else "Group")
    plt.title(title if title else f"{value_col} by group")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=600, bbox_inches="tight")

    plt.show()

print("[INFO] ===== Environment Created =====")

### Synthetic Image Generation

Synthetic Image Generation

This module generates a small synthetic dataset of single-cell microglial images that can be used as a controlled reference set for testing and demonstrating the downstream morphometric workflow. Rather than replacing experimental microscopy data, it provides a reproducible collection of predefined microglial-like morphologies that facilitate pipeline development, visualization, and end-to-end benchmarking.

The generator creates 25 RGB images of isolated synthetic microglia (5 images for each of 5 predefined cell states) in a fixed 512 × 512 format. Each image contains a single centrally positioned cell rendered against a black background. The synthetic signal is distributed across biologically interpretable channels:

- Processes in the green channel
- Soma in the green channel
- Nucleus in the blue channel

To emulate a fluorescence/confocal-like appearance, the images are further processed with mild Gaussian blur and additive Gaussian noise.

The module simulates five predefined microglial morphological states:

- Resting microglia (small soma, numerous long and highly ramified processes)
- Primed microglia (slightly enlarged soma, fewer and less arborized branches)
- Activated microglia (enlarged soma with shorter, thicker processes)
- Ameboid microglia (large rounded soma with minimal branching)
- Resolving microglia (intermediate morphology with partial process re-extension)

These synthetic morphologies are generated through parameterized drawing rules that control soma radius, branch number, branch length, angular dispersion, thickness, and arborization probability. Reproducibility is enforced through a fixed random seed, allowing the same synthetic dataset to be regenerated consistently across runs and environments.

In addition to image generation, this module also:

- Saves all synthetic images as .png files in the output directory
- Creates a labels.csv file linking each filename to its corresponding microglial state
- Generates a 5 × 5 visualization panel summarizing the full synthetic dataset

Within the broader structure of the notebook, these synthetic images serve as a convenient demonstration dataset for running the subsequent stages of the analysis pipeline, including preprocessing, segmentation, morphometric extraction, visualization, and classification.

Although this synthetic dataset is primarily intended for testing and demonstration, the downstream workflow can also be applied to real experimental microglial images, provided that image quality and cell isolation are sufficient for reliable morphometric analysis.

In [ ]:
# ============================================================
# Synthetic Microglia Generator — 5 Functional States
# ============================================================
# Processes = GREEN channel
# Soma = GREEN
# Nucleus = BLUE
# ============================================================
print("[INFO] ===== Module 0 - Synthetic Microglia =====")
# ============================================================
# Global configuration
# ============================================================
N_IMAGES_TOTAL = 25
IMAGE_SIZE = (512, 512)
BACKGROUND_LEVEL = 0.0
MAX_INTENSITY = 1.0
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

CELL_STATES = [
    "resting_microglia",
    "primed_microglia",
    "activated_microglia",
    "ameboid_microglia",
    "resolving_microglia"
]

IMAGES_PER_STATE = N_IMAGES_TOTAL // len(CELL_STATES)

# ============================================================
# Utility functions
# ============================================================

def create_empty_image():
    """Return a pure black RGB image."""
    return np.zeros((IMAGE_SIZE[0], IMAGE_SIZE[1], 3), dtype=np.float32)

def add_green(img, rr, cc, intensity):
    """Add intensity to GREEN channel."""
    img[rr, cc, 1] = np.clip(img[rr, cc, 1] + intensity, 0, MAX_INTENSITY)
    return img

def add_blue(img, rr, cc, intensity):
    """Add intensity to BLUE channel."""
    img[rr, cc, 2] = np.clip(img[rr, cc, 2] + intensity, 0, MAX_INTENSITY)
    return img

def draw_soma(img, center, radius, intensity=0.8):
    """Draw soma in GREEN, nucleus in BLUE."""
    yy, xx = np.ogrid[:IMAGE_SIZE[0], :IMAGE_SIZE[1]]
    mask = (yy - center[0])**2 + (xx - center[1])**2 <= radius**2

    # Soma (green)
    img[mask, 1] = np.clip(img[mask, 1] + intensity, 0, MAX_INTENSITY)

    # Nucleus (blue)
    nucleus_mask = (yy - center[0])**2 + (xx - center[1])**2 <= (radius * 0.45)**2
    img[nucleus_mask, 2] = np.clip(img[nucleus_mask, 2] + 0.9, 0, MAX_INTENSITY)

    return img

def draw_branch(img, start, length, n_segments, angle_spread, base_angle,
                thickness=1, intensity=0.7, arborization_prob=0.0):

    y, x = start
    seg_length = length / n_segments
    current_angle = base_angle

    for _ in range(n_segments):
        current_angle += np.random.uniform(-angle_spread, angle_spread)

        dy = seg_length * np.sin(current_angle)
        dx = seg_length * np.cos(current_angle)

        new_y = int(y + dy)
        new_x = int(x + dx)

        rr, cc = line(int(y), int(x), new_y, new_x)

        # Processes in GREEN
        for t in range(-thickness, thickness + 1):
            rr_shift = np.clip(rr + t, 0, IMAGE_SIZE[0] - 1)
            cc_shift = np.clip(cc + t, 0, IMAGE_SIZE[1] - 1)
            img = add_green(img, rr_shift, cc_shift, intensity)

        # Arborization
        if np.random.rand() < arborization_prob:
            side_angle = current_angle + np.random.uniform(-np.pi/3, np.pi/3)
            side_len = length * np.random.uniform(0.1, 0.3)

            img, _ = draw_branch(
                img,
                start=(new_y, new_x),
                length=side_len,
                n_segments=max(2, n_segments // 3),
                angle_spread=angle_spread,
                base_angle=side_angle,
                thickness=max(1, thickness - 1),
                intensity=intensity * 0.8,
                arborization_prob=arborization_prob * 0.7
            )

        y, x = new_y, new_x

    return img, (y, x)

def add_gaussian_noise(img, sigma=0.02):
    noise = np.random.normal(0, sigma, img.shape)
    img = img + noise
    return np.clip(img, 0, MAX_INTENSITY)

def apply_confocal_blur(img, sigma=1.0):
    """Blur all channels slightly for realism."""
    blurred = img.copy()
    for c in range(3):
        blurred[:, :, c] = gaussian(img[:, :, c], sigma=sigma, preserve_range=True)
    return blurred

# ============================================================
# Microglia Morphology Generators
# ============================================================

def generate_resting_microglia():
    img = create_empty_image()
    center = (256, 256)
    img = draw_soma(img, center, radius=10, intensity=0.8)

    n_branches = np.random.randint(8, 12)
    for _ in range(n_branches):
        angle = np.random.uniform(0, 2*np.pi)
        img, _ = draw_branch(
            img, center,
            length=np.random.uniform(120, 200),
            n_segments=12,
            angle_spread=np.deg2rad(8),
            base_angle=angle,
            thickness=1,
            intensity=0.6,
            arborization_prob=0.5
        )

    return add_gaussian_noise(apply_confocal_blur(img, sigma=1.0), sigma=0.03)

def generate_primed_microglia():
    img = create_empty_image()
    center = (256, 256)
    img = draw_soma(img, center, radius=12, intensity=0.85)

    n_branches = np.random.randint(6, 9)
    for _ in range(n_branches):
        angle = np.random.uniform(0, 2*np.pi)
        img, _ = draw_branch(
            img, center,
            length=np.random.uniform(100, 150),
            n_segments=10,
            angle_spread=np.deg2rad(12),
            base_angle=angle,
            thickness=1,
            intensity=0.7,
            arborization_prob=0.25
        )

    return add_gaussian_noise(apply_confocal_blur(img, sigma=1.1), sigma=0.035)

def generate_activated_microglia():
    img = create_empty_image()
    center = (256, 256)
    img = draw_soma(img, center, radius=14, intensity=0.9)

    n_branches = np.random.randint(3, 6)
    for _ in range(n_branches):
        angle = np.random.uniform(0, 2*np.pi)
        img, _ = draw_branch(
            img, center,
            length=np.random.uniform(40, 80),
            n_segments=5,
            angle_spread=np.deg2rad(20),
            base_angle=angle,
            thickness=2,
            intensity=0.8,
            arborization_prob=0.1
        )

    return add_gaussian_noise(apply_confocal_blur(img, sigma=1.2), sigma=0.04)

def generate_ameboid_microglia():
    img = create_empty_image()
    center = (256, 256)
    img = draw_soma(img, center, radius=18, intensity=1.0)

    for _ in range(np.random.randint(0, 3)):
        angle = np.random.uniform(0, 2*np.pi)
        img, _ = draw_branch(
            img, center,
            length=np.random.uniform(20, 40),
            n_segments=3,
            angle_spread=np.deg2rad(25),
            base_angle=angle,
            thickness=3,
            intensity=0.9,
            arborization_prob=0.0
        )

    return add_gaussian_noise(apply_confocal_blur(img, sigma=1.3), sigma=0.05)

def generate_resolving_microglia():
    img = create_empty_image()
    center = (256, 256)
    img = draw_soma(img, center, radius=12, intensity=0.85)

    n_branches = np.random.randint(4, 7)
    for _ in range(n_branches):
        angle = np.random.uniform(0, 2*np.pi)
        img, _ = draw_branch(
            img, center,
            length=np.random.uniform(80, 120),
            n_segments=8,
            angle_spread=np.deg2rad(15),
            base_angle=angle,
            thickness=1,
            intensity=0.7,
            arborization_prob=0.2
        )

    return add_gaussian_noise(apply_confocal_blur(img, sigma=1.1), sigma=0.035)

# ============================================================
# Main generation loop
# ============================================================

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("[INFO] OUTPUT_DIR:", OUTPUT_DIR)
    print("[INFO] OUTPUT_DIR abs:", OUTPUT_DIR.resolve())

    generators = {
        "resting_microglia": generate_resting_microglia,
        "primed_microglia": generate_primed_microglia,
        "activated_microglia": generate_activated_microglia,
        "ameboid_microglia": generate_ameboid_microglia,
        "resolving_microglia": generate_resolving_microglia
    }

    records = []
    counter = 0

    for cell_state in CELL_STATES:
        for _ in range(IMAGES_PER_STATE):
            img = generators[cell_state]()
            fname = f"{cell_state}_{counter:03d}.png"
            save_path = OUTPUT_DIR / fname

            plt.imsave(save_path, img)
            
            records.append({"filename": fname, "cell_state": cell_state})
            counter += 1

    labels_path = OUTPUT_DIR / "labels.csv"
    pd.DataFrame(records).to_csv(labels_path, index=False)

# ============================================================
# Visualization panel
# ============================================================

def visualize_panel():
    files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(".png")])[:25]

    fig, axes = plt.subplots(5, 5, figsize=(15, 15))

    for ax, fname in zip(axes.flatten(), files):
        img = plt.imread(OUTPUT_DIR / fname)
        ax.imshow(img)
        ax.set_title(fname, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(EXPORT_DIR / "Module 0 - synthetic_microglia_panel.png", dpi=600, bbox_inches="tight")
    plt.show()

if __name__ == "__main__":
    main()
    visualize_panel()

print("\n[INFO] ===== Module 0 Completed =====")

**Figure 1. Synthetic microglial dataset representing five canonical morphological states.**  
This panel illustrates the full set of 25 synthetic microglial images generated at 40× confocal‑like resolution, encompassing resting, primed, activated, ameboid, and resolving phenotypes. Each state exhibits characteristic morphological signatures: resting microglia display highly ramified arbors with fine distal processes; primed cells show moderate retraction and thickening of proximal branches; activated microglia exhibit short, thickened processes and enlarged somata; ameboid cells adopt a compact, minimally branched morphology; and resolving microglia partially re‑extend processes following activation. These controlled, state‑specific morphologies provide a standardized reference dataset for validating the morphometric pipeline, enabling systematic benchmarking of fractal, topological, and texture‑based descriptors under well‑defined structural conditions.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## **Module 1 — Dataset Loader**

This module implements a flexible dataset loader designed to handle common directory organizations for microglial image datasets.
Rather than enforcing a strict format, it automatically detects whether the dataset is organized into subfolders (multiclass) or stored as a flat collection of images, and adapts its behavior accordingly.

To simplify usage across different environments, the module first attempts to open a graphical folder selector. If this is not available (e.g., in headless or restricted environments), a fallback path can be manually specified.

---

### **Dataset Selection**

The dataset directory is selected through an interactive dialog:

- A GUI folder selector (via tkinter) is launched
- If no folder is selected or the GUI is unavailable, a fallback path is used

This design allows the loader to operate seamlessly across local machines, servers, and notebook environments.

---

#### **Supported Dataset Structures**  
1. Multiclass folder structure (recommended)

If the selected directory contains subfolders, each subfolder is interpreted as a separate class or group:

dataset/
├── resting_microglia/
├── primed_microglia/
├── activated_microglia/
├── ameboid_microglia/
└── resolving_microglia/

In this mode:

Each subfolder name is used as the class / group label
All compatible image files inside each folder are loaded
The number and naming of classes are fully user-defined

2. Flat folder structure (no subfolders)

If no subdirectories are detected, the loader assumes a flat dataset:

image_001.png  
activated_002.png  
resting_003.png  

In this case:

Images are still loaded automatically
Labels are inferred from the filename prefix, using the pattern:
label_identifier_###.png

For example:

resting_001.png → label: resting
activated_cell_002.png → label: activated_cell

This approach allows basic labeling without requiring subfolder organization.

---

### **Supported File Formats**

The loader searches for images using the following extensions:

.png, .jpg, .jpeg, .tif, .tiff

All matching files are collected recursively within each detected folder (single-level).

---

### **Outputs of the Loader**

After scanning the dataset, the module returns three standardized outputs:

- all_paths — list of absolute paths to all detected images
- all_cell_states — label associated with each image
- class_names — ordered list of detected classes

If no images are found, the loader returns empty lists and displays a warning.

---

### **Behavior and Design Notes**

- Class detection is automatic and based on folder names or filename patterns
- Image paths are stored as absolute paths for downstream compatibility
- The order of classes reflects their first appearance in the dataset
- No assumptions are made about image content beyond file format

---

### **Integration with the Pipeline**

The outputs of this module are directly consumed by all downstream steps, including:

- preprocessing and segmentation
- morphometric feature extraction
- fractal and texture analysis
- graph-based modeling
- dimensionality reduction (PCA, UMAP)
- statistical analysis and classification

Once the dataset is loaded, no additional configuration is required.

---

### **Usage**

1. Run the module
2. Select your dataset folder (or define a fallback path)
3. The loader automatically detects structure and labels

This ensures a simple and reproducible data ingestion step that works across different dataset organizations and computational environments.

---

### **Dataset path**  
Window pop‑up will appear to select the images folder. If this does not work, the only required user edit is, if necessary:  
`DATASET_DIR = r"your/path/here/dataset"`  
Internal functions and variables should not be altered.

In [ ]:
# ============================================================
# Universal Multiclass Dataset Loader
# ============================================================

print("\n[INFO] ===== Module 1 - Dataset Loader =====")

# ------------------------------------------------------------
# Folder selector
# ------------------------------------------------------------
def select_dataset_directory():
    root = tk.Tk()
    root.withdraw()                     
    root.attributes("-topmost", True) 
    root.update()

    selected_dir = filedialog.askdirectory(
        title="Select microglia dataset folder",
        mustexist=True
    )

    root.destroy()

    if not selected_dir:
        raise ValueError("No folder selected.")

    return selected_dir

# ------------------------------------------------------------
# Dataset loader
# ------------------------------------------------------------
def load_dataset_multiclass(base_dir, extensions=("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff")):
    base_dir = os.path.abspath(base_dir)

    image_paths = []
    cell_states = []
    display_names = []
    unique_ids = []

    # Recursive search: supports images in subfolders and sub‑subfolders
    for ext in extensions:
        pattern = os.path.join(base_dir, "**", ext)
        for img in sorted(glob.glob(pattern, recursive=True)):
            img_abs = os.path.abspath(img)
            rel_path = os.path.relpath(img_abs, base_dir).replace("\\", "/")
            parts = Path(rel_path).parts

            # If folders are present, the group will be the first folder
            if len(parts) >= 2:
                state_name = parts[0]
            else:
                fname = os.path.basename(img_abs)
                state_name = os.path.splitext(fname)[0].rsplit("_", 1)[0]

            image_paths.append(img_abs)
            cell_states.append(state_name)
            display_names.append(os.path.basename(img_abs))  # Show Short name
            unique_ids.append(rel_path)                      # Unique identifier

    if len(image_paths) == 0:
        print("[WARNING] No images found.")
        return [], [], [], [], []

    classes = list(dict.fromkeys(cell_states))

    print(f"[INFO] Detected groups: {classes}")
    print(f"[INFO] Loaded {len(image_paths)} images from: {base_dir}")

    return image_paths, cell_states, classes, display_names, unique_ids
# ============================================================
# Select folder and load dataset
# ============================================================

try:
    DATASET_DIR = select_dataset_directory()
except Exception:
    DATASET_DIR = r"your/path/here/dataset"

#
all_paths, all_cell_states, class_names, all_display_names, all_unique_ids = load_dataset_multiclass(DATASET_DIR)
state_order = class_names

print("\n[INFO] ===== Module 1 Completed =====")

<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## **Module 2 — Core Image Processing Functions**

This module defines foundational image processing utilities used in later stages of the microglial morphometric analysis pipeline.
At this stage, the focus is not on a full preprocessing workflow, but on a specific operation required for structural analysis: the extraction of a smoothed skeleton suitable for fractal dimension estimation.

'compute_skeleton_for_fd(binary)'

This function generates a clean and permissive skeleton representation from a binary image.
It is specifically designed for fractal dimension (FD) analysis, where preserving the overall structure of the signal is more important than aggressively simplifying it.

The function expects a binary input image (foreground = microglial structure, background = 0) and applies the following steps:

1. Binary normalization

The input is converted to a boolean array to ensure consistent processing:

- Non-zero values → foreground (True)
- Zero values → background (False)

If the input contains no foreground pixels, the function returns an empty skeleton.

2. Removal of very small objects

Small connected components are removed using:

'remove_small_objects(max_size=29)'

This step eliminates tiny artifacts while preserving most of the microglial structure.
The threshold is intentionally permissive to avoid removing fine processes.

3. Mild smoothing

A Gaussian filter is applied:

- Sigma = 1.0
- Followed by thresholding (> 0.2)

This operation:

- reduces pixel-level noise
- smooths irregular edges
- improves skeleton continuity

4. Skeletonization

The processed binary image is reduced to a 1-pixel-wide skeleton using:

'skimage.morphology.skeletonize'

This skeleton preserves the global topology of the structure and is suitable for:

- Fractal dimension estimation
- Structural complexity analysis
- Downstream graph-based representations

---

### **Design Philosophy**

This function is intentionally designed to be:

- Permissive — preserves as much structural information as possible
- Minimalistic — avoids aggressive filtering or pruning
- Deterministic — produces consistent outputs for identical inputs
- FD-oriented — optimized for fractal analysis rather than visual cleanliness

Notably, the function does not perform:

- Largest connected component selection
- Branch pruning or spur removal
- Morphological simplification beyond basic smoothing

These choices ensure that fine microglial processes are retained, which is critical for accurate fractal measurements.

---

### **Role in the Pipeline**

The output of this function serves as a structural representation of microglial morphology and is primarily used in:

- Fractal dimension computation
- Structural complexity metrics
- Graph-based morphometric analysis

This module establishes an initial building block for more advanced processing steps defined in subsequent modules.

In [ ]:
# ============================================================
# Core Image Processing Functions
# ============================================================

print("\n[INFO] ===== Module 2 - Core Image Processing Functions =====")

# ------------------------------------------------------------
# Skeleton for fractal dimension
# ------------------------------------------------------------
def compute_skeleton_for_fd(binary):
    """
    Simple and permissive FD skeleton:
    - keep most foreground signal
    - mild smoothing
    - no largest-component filtering
    - no spur pruning
    """
    binary = binary.astype(bool)

    if not np.any(binary):
        return np.zeros_like(binary, dtype=bool)

    binary_clean = remove_small_objects(binary, max_size=29)
    binary_smooth = gaussian(binary_clean.astype(float), sigma=1.0) > 0.2
    skel = skeletonize(binary_smooth)
    return skel

print("\n[INFO] ===== Module 2 Completed =====")

### **Notes on Skeleton Representation for Analysis**

The skeleton generated in this module is not intended for visualization purposes, but for quantitative structural analysis.

Unlike visualization-oriented skeletons, which often prioritize visual clarity, this implementation is intentionally permissive and preserves fine structural details. This is particularly important for microglial morphometrics, where thin processes and subtle branching patterns contribute significantly to measures such as fractal dimension.

As a result:

- Small branches are retained whenever possible
- No aggressive pruning is applied
- The skeleton reflects structural complexity rather than visual simplicity

This design ensures that downstream analyses capture biologically relevant morphological features, even when they appear visually noisy


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


### **Module 3 — Fractal Dimension (Box‑Counting Method)**

This module performs three key tasks within the microglial morphometric analysis pipeline:

- Loads and preprocesses microglial images,
- Generates an ordered visual inspection panel for quality control, and
- Computes fractal dimension (FD) using the classical box-counting method.

Rather than being limited to fractal analysis alone, this module establishes the processed image representations required for downstream structural quantification and provides both numerical and visual outputs for evaluating microglial morphology.

It is designed to work with both synthetic microglial images and real microscopy data, provided that the images can be loaded as grayscale-compatible inputs.

---

1. Image Loading
'load_image(path)'

This function loads an image from disk and converts it into a grayscale representation suitable for preprocessing.

The loader uses a hierarchical backend strategy:

- OpenCV, when available
- Scikit-image, as an alternative backend
- Pillow (PIL), as a secondary fallback

For multichannel images, the function converts the signal to grayscale before further processing.
In the OpenCV path, RGB-like images are converted using standard grayscale conversion.
In the scikit-image fallback, multichannel images are reduced using a weighted combination of the green and blue channels, which is appropriate for the synthetic microglial images generated earlier in the notebook.

If the image cannot be loaded by any backend, the function raises an explicit error.

2. Image Preprocessing
'preprocess_image(img, target_size=(1024, 1024), ...)'

This function standardizes each image prior to quantitative analysis.
The preprocessing workflow includes the following steps:

- Resizing

All images are resized to a fixed target resolution (default: 1024 × 1024 px), ensuring consistent scale for downstream measurements, uses adaptive local thresholding with a minimum-signal constraint, and keeps RGB/RGBA conversion to grayscale before segmentation. This improves preservation of fine microglial processes while keeping downstream metrics comparable across images.

- Optional Gaussian blur

A mild Gaussian blur can be applied before thresholding in order to reduce local noise and improve segmentation stability.

- Thresholding

The module supports two segmentation modes:

Otsu thresholding (default)
adaptive local thresholding (optional)

This step converts the grayscale image into a binary mask representing the segmented microglial structure.

- Small-component removal

Connected components below a minimum size threshold can be removed in order to suppress small artifacts.

- Small-hole filling

Small holes inside the segmented structure can be filled to produce more coherent binary objects.

- Skeleton generation

Two skeleton representations are generated:

visual skeleton — a standard skeleton used for inspection and display
FD skeleton — a smoothed and more permissive skeleton produced by compute_skeleton_for_fd, intended for fractal analysis
g. Bounding box extraction

A bounding box is computed from the binary foreground coordinates.
If no foreground is detected, the function returns a default bounding box covering the full target image.

The function returns:

- resized grayscale image
- binary mask
- visual skeleton
- FD-oriented skeleton
- bounding box

All image-like outputs are converted to uint8 for compatibility and consistency.

3. Ordered Visual Inspection Panel
'visualize_preprocessing_panel_ordered(...)'

To facilitate preprocessing quality control, the module creates a structured visualization panel showing the main processing stages for selected images.

For each displayed sample, the panel includes:

- original image
- resized image
- binary segmentation
- visual skeleton

The visualization is intentionally:

- grouped by cell state
- ordered according to the preferred class order
- limited to a maximum number of classes
- limited to a maximum number of images per class

This design makes it easier to inspect whether preprocessing behaves consistently across microglial phenotypes while keeping the notebook responsive for large datasets.

The resulting panel is exported to:

'Module 3 - processing.png'

4. Fractal Dimension (Box-Counting Method)
'fractal_dimension(binary_img)'

This function implements a classical and permissive box-counting algorithm for estimating the fractal dimension of a binary microglial structure.

The function expects a 2D binary image and proceeds as follows:

-  Boolean conversion

The input image is converted to boolean format.

-  Empty-mask handling

If no foreground pixels are present, the function returns:

- fd = 0.0
- empty lists for sizes and counts
- zero-valued coefficients

- Multiscale box counting

The image is partitioned into square boxes of decreasing size, obtained by repeated division by 2 from the maximum possible box size down to 2 pixels.

For each box size, the algorithm counts how many boxes contain at least one foreground pixel.

- Log-log regression

The fractal dimension is estimated as the slope of the linear fit between:

- log(1 / box_size)
- log(number_of_non_empty_boxes)

The function returns:

- fd (estimated fractal dimension)
- sizes (list of box sizes used)
- counts (number of non-empty boxes at each scale)
- coeffs (regression coefficients from the log-log fit)

This implementation is intentionally simple and permissive:

- no foreground cropping
- no extra filtering inside the FD function
- no morphology-specific assumptions beyond binarity

5. Preprocessing Cache and FD Summary Table

The module preprocesses all images listed in the dataset loader outputs and stores the results in a dictionary named preprocessed_data.

For each image, the cache stores:

- filename
- cell state
- resized image
- binary mask
- visual skeleton
- FD skeleton
- bounding box

The module then computes fractal dimension for each image and assembles a summary table containing:

- filename
- cell_state
- fractal_dimension

This table is sorted according to the preferred class order and exported as:

'Module 3 - fractal_dimension_summary_table.csv'

6. Visualization of Fractal Dimension Results

Finally, the module visualizes FD values across images or groups using the shared plotting utility plot_bars_or_boxplot(...).

Depending on dataset size and class structure, the plot is displayed either as:

- a bar plot per image, or
- a boxplot with individual data points by group

The resulting figure is exported as:

'Module 3 - bar_plot_of_fractal_dimension.png'

---

### **Role in the Pipeline**

This module is a bridge between raw input images and quantitative morphological analysis.
It standardizes the image representation, enables visual quality control, and computes one of the key global descriptors of microglial structural complexity: fractal dimension.

Its outputs support both immediate interpretation and later modules involving comparative morphometrics, statistical analysis, and classification.

In [ ]:
# ============================================================
# Microglia preprocessing + Visual Inspection Panel
# ============================================================

print("\n[INFO] ===== Module 3 - Microglia preprocessing =====")

# ============================================================
# Module 3 configuration
# ============================================================

# Recommended for high-resolution microglia images.
# Use (512, 512) if processing is too slow.
PROCESS_TARGET_SIZE = (1024, 1024)

# Adaptive threshold is better for fluorescence images with
# bright soma + faint processes + uneven background.
USE_ADAPTIVE_THRESHOLD = True

# En skimage threshold_local usa: threshold = local_mean - offset
# Por eso offset positivo puede hacer que el fondo negro entre como objeto.
# Usamos offset negativo para hacerlo más estricto.
ADAPTIVE_OFFSET = -5

# Evita que el fondo muy oscuro se segmente aunque el threshold local falle.
MIN_SIGNAL_INTENSITY = 10

# Object size must increase when image size increases.
# Suggested:
#   512  -> 80-150
#   1024 -> 200-400
MIN_OBJECT_SIZE = 250

# Fill only small holes inside the segmented cell.
MIN_HOLE_SIZE = 500

# Mild blur to reduce pixel noise before thresholding.
GAUSSIAN_BLUR = True

# Keep conservative fluorescence enhancement.
AUTO_ENHANCE = True

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def load_image(path):
    """
    Load image and convert RGB/RGBA to grayscale.

    This keeps your preferred behavior:
    RGB image -> grayscale image.

    The output is always uint8 in range 0-255.
    """

    # --- OpenCV backend ---
    if OPENCV_AVAILABLE:
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)

        if img is not None:
            # RGB/BGR/RGBA/BGRA -> grayscale
            if img.ndim == 3:
                if img.shape[2] == 4:
                    img = cv2.cvtColor(img, cv2.COLOR_BGRA2GRAY)
                else:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            img = exposure.rescale_intensity(
                img.astype(np.float32),
                out_range=(0, 255)
            ).astype(np.uint8)

            return img

    # --- scikit-image backend ---
    try:
        img = io.imread(path)

        # RGB/RGBA -> grayscale
        if img.ndim == 3:
            if img.shape[2] == 4:
                img = img[:, :, :3]
            img = rgb2gray(img)

        img = img.astype(np.float32)

        if img.max() > 0:
            img = exposure.rescale_intensity(
                img,
                out_range=(0, 255)
            )

        return img.astype(np.uint8)

    except Exception:
        pass

    # --- PIL backend ---
    if PIL_AVAILABLE:
        try:
            img = Image.open(path).convert("L")
            img = np.array(img).astype(np.float32)

            if img.max() > 0:
                img = exposure.rescale_intensity(
                    img,
                    out_range=(0, 255)
                )

            return img.astype(np.uint8)

        except Exception:
            pass

    raise ValueError(f"[ERROR] Could not load image: {path}")

# ------------------------------------------------------------
# Preprocess
# ------------------------------------------------------------
def _remove_small_components(binary, min_size):
    labeled_img = label(binary)
    cleaned = np.zeros_like(binary, dtype=bool)

    for region in regionprops(labeled_img):
        if region.area >= min_size:
            cleaned[labeled_img == region.label] = True

    return cleaned

def auto_enhance_fluorescence(
    img,
    p_low=1,
    p_high=99.5,
    min_p95_intensity=80,
    max_gain=2.0
):
    """
    Conservative rescue of weak fluorescence.
    Only boosts clearly dim images and caps the gain.
    Avoids homogenizing biological differences.
    """

    img = img.astype(np.float32)

    if img.max() <= 1.0:
        img *= 255.0

    if img.max() == 0:
        return img.astype(np.uint8)

    p95 = np.percentile(img, 95)

    # Only rescue underexposed images
    if p95 < min_p95_intensity and p95 > 0:
        gain = min(min_p95_intensity / p95, max_gain)
        img = img * gain

    # Mild percentile clipping only
    low, high = np.percentile(img, (p_low, p_high))

    if high > low:
        img = exposure.rescale_intensity(
            img,
            in_range=(low, high),
            out_range=(0, 255)
        )

    return np.clip(img, 0, 255).astype(np.uint8)

def _fill_small_holes(binary, min_hole_size):
    inv = ~binary
    labeled_img = label(inv)
    filled = binary.copy()

    for region in regionprops(labeled_img):
        if region.area < min_hole_size:
            coords = region.coords
            filled[coords[:, 0], coords[:, 1]] = True

    return filled

def preprocess_image(
    img,
    target_size=PROCESS_TARGET_SIZE,
    use_adaptive_threshold=USE_ADAPTIVE_THRESHOLD,
    adaptive_offset=ADAPTIVE_OFFSET,
    min_signal_intensity=MIN_SIGNAL_INTENSITY,
    gaussian_blur=GAUSSIAN_BLUR,
    min_object_size=MIN_OBJECT_SIZE,
    min_hole_size=MIN_HOLE_SIZE,
    clean_holes=True,
    auto_enhance=AUTO_ENHANCE
):
    """
    Preprocess grayscale fluorescence image:
      1. Resize to common size for comparable metrics
      2. Enhance fluorescence conservatively
      3. Mild blur
      4. Adaptive or Otsu threshold
      5. Remove small objects
      6. Fill small holes
      7. Skeletonize
    """

    # --------------------------------------------------------
    # Resize to common size
    # --------------------------------------------------------
    if target_size is None:
        img_resized = img.copy()
    else:
        if OPENCV_AVAILABLE:
            # INTER_AREA is good for downsampling.
            # INTER_CUBIC is better if an image is smaller than target_size.
            if img.shape[0] > target_size[0] or img.shape[1] > target_size[1]:
                interpolation = cv2.INTER_AREA
            else:
                interpolation = cv2.INTER_CUBIC

            img_resized = cv2.resize(
                img,
                target_size,
                interpolation=interpolation
            )
        else:
            img_resized = resize(
                img,
                target_size,
                preserve_range=True,
                anti_aliasing=True
            ).astype(np.uint8)

    # --------------------------------------------------------
    # Automatic fluorescence enhancement
    # --------------------------------------------------------
    if auto_enhance:
        img_resized = auto_enhance_fluorescence(
            img_resized,
            p_low=1,
            p_high=99.8,
            min_p95_intensity=70,
            max_gain=1.5
        )

    # --------------------------------------------------------
    # Optional blur
    # --------------------------------------------------------
    if gaussian_blur:
        if OPENCV_AVAILABLE:
            img_blur = cv2.GaussianBlur(img_resized, (3, 3), 0)
        else:
            img_blur = gaussian(
                img_resized,
                sigma=0.5,
                preserve_range=True
            ).astype(np.uint8)
    else:
        img_blur = img_resized

    # --------------------------------------------------------
    # Thresholding
    # --------------------------------------------------------
    if use_adaptive_threshold:
        # Window size adapted to image size.
        # For 1024 px this gives ~71 px.
        # For 512 px this gives ~51 px.
        block_size = int(min(img_blur.shape) * 0.07)

        # threshold_local requires odd block size
        if block_size % 2 == 0:
            block_size += 1

        block_size = max(block_size, 51)

        adaptive_thresh = threshold_local(
            img_blur,
            block_size=block_size,
            method="gaussian",
            offset=adaptive_offset
        )

        # Two conditions:
        # 1. Pixel must be above local threshold.
        # 2. Pixel must have minimum real fluorescence signal.
        binary = (img_blur > adaptive_thresh) & (img_blur > min_signal_intensity)

    else:
        thresh = threshold_otsu(img_blur)

        # Also apply minimum signal filter for safety.
        binary = (img_blur > thresh) & (img_blur > min_signal_intensity)

    # --------------------------------------------------------
    # Remove small objects
    # --------------------------------------------------------
    if min_object_size > 0:
        binary = _remove_small_components(binary, min_object_size)

    # --------------------------------------------------------
    # Fill small holes
    # --------------------------------------------------------
    if clean_holes and min_hole_size > 0:
        binary = _fill_small_holes(binary, min_hole_size)

    # --------------------------------------------------------
    # Skeletons
    # --------------------------------------------------------
    skeleton_visual = skeletonize(binary)
    skeleton_fd = compute_skeleton_for_fd(binary)

    # --------------------------------------------------------
    # Convert to uint8
    # --------------------------------------------------------
    binary_uint8 = binary.astype(np.uint8) * 255
    skeleton_visual_uint8 = skeleton_visual.astype(np.uint8) * 255
    skeleton_fd_uint8 = skeleton_fd.astype(np.uint8) * 255

    # --------------------------------------------------------
    # Bounding box
    # --------------------------------------------------------
    coords = np.column_stack(np.where(binary))

    if len(coords) > 0:
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        bbox = (y_min, x_min, y_max, x_max)
    else:
        bbox = (0, 0, img_resized.shape[0], img_resized.shape[1])

    return (
        img_resized,
        binary_uint8,
        skeleton_visual_uint8,
        skeleton_fd_uint8,
        bbox
    )

# ------------------------------------------------------------
# Visual inspection panel
# ------------------------------------------------------------
def visualize_preprocessing_panel_ordered(
    image_paths,
    cell_states,
    preprocessed_data,
    max_classes=5,
    max_images_per_class=10,
    preferred_order=None,
    save_path=None
):
    if save_path is None:
        save_path = os.path.join(EXPORT_DIR, "Module 3 - processing.png")

    class_to_images = defaultdict(list)
    for p, s in zip(image_paths, cell_states):
        class_to_images[s].append(p)

    classes = list(class_to_images.keys())

    if preferred_order is not None:
        ordered = [c for c in preferred_order if c in classes]
        ordered += [c for c in classes if c not in ordered]
    else:
        ordered = classes

    selected = []
    for cls in ordered[:max_classes]:
        for p in class_to_images[cls][:max_images_per_class]:
            selected.append((p, cls))

    fig, axes = plt.subplots(
        len(selected), 4,
        figsize=(12, 3 * len(selected)),
        squeeze=False
    )

    for i, (path, state) in enumerate(selected):
        try:
            data = preprocessed_data[path]

            # original en gris
            img_original = load_image(path)

            r = data["img_resized"]
            b = data["binary_uint8"]
            s = data["skeleton_visual_uint8"]

            axes[i, 0].imshow(img_as_ubyte(img_original), cmap="gray")
            axes[i, 0].set_title(f"{state}\n{data['sample_id']}", fontsize=8)
            axes[i, 0].axis("off")

            axes[i, 1].imshow(r, cmap="gray")
            axes[i, 1].set_title(f"Resized {PROCESS_TARGET_SIZE[0]}x{PROCESS_TARGET_SIZE[1]}")
            axes[i, 1].axis("off")

            axes[i, 2].imshow(b, cmap="gray")
            axes[i, 2].set_title("Binary")
            axes[i, 2].axis("off")

            axes[i, 3].imshow(s, cmap="gray")
            axes[i, 3].set_title("Skeleton")
            axes[i, 3].axis("off")

        except Exception as e:
            for j in range(4):
                axes[i, j].text(0.5, 0.5, str(e), ha="center", va="center")
                axes[i, j].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()
    
# ------------------------------------------------------------
# Run
# ------------------------------------------------------------
# ============================================================
# Preprocessing cache
# ============================================================

path_to_display = dict(zip(all_paths, all_display_names))
path_to_uid = dict(zip(all_paths, all_unique_ids))

preprocessed_data = {}

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        img = load_image(path)

        img_resized, binary_uint8, skeleton_visual_uint8, skeleton_fd_uint8, bbox = preprocess_image(
            img,
            target_size=PROCESS_TARGET_SIZE,
            use_adaptive_threshold=USE_ADAPTIVE_THRESHOLD,
            adaptive_offset=ADAPTIVE_OFFSET,
            min_signal_intensity=MIN_SIGNAL_INTENSITY,
            gaussian_blur=GAUSSIAN_BLUR,
            min_object_size=MIN_OBJECT_SIZE,
            min_hole_size=MIN_HOLE_SIZE,
            clean_holes=True,
            auto_enhance=AUTO_ENHANCE
        )

        preprocessed_data[path] = {
            "filename": path_to_display[path],
            "sample_id": path_to_uid[path],
            "cell_state": cell_state,
            "img_resized": img_resized,
            "binary_uint8": binary_uint8,
            "skeleton_visual_uint8": skeleton_visual_uint8,
            "skeleton_fd_uint8": skeleton_fd_uint8,
            "bbox": bbox
        }

    except Exception as e:
        print(f"[PREPROCESS ERROR] {path}: {e}")

visualize_preprocessing_panel_ordered(
    all_paths,
    all_cell_states,
    preprocessed_data=preprocessed_data,
    max_classes=5,
    max_images_per_class=10,
    preferred_order=state_order,
    save_path=os.path.join(EXPORT_DIR, "Module 3 - processing.png")
)

**Figure 2. End‑to‑end preprocessing workflow: from raw microglial imagery to skeletonized arbors.**  
This figure illustrates the complete preprocessing pipeline applied to individual microglial images, including the original grayscale input, Otsu‑based thresholding, binary mask generation, and final skeleton extraction using the Zhang–Suen thinning algorithm. Otsu’s adaptive thresholding reliably separates microglial structures from background intensity variations, producing clean binary masks suitable for downstream morphometrics. The resulting skeletons preserve the topological organization of the arbor—branch points, endpoints, and overall geometry—while removing redundant pixel thickness. This step is critical for fractal analysis, graph‑based topology, and Sholl profiling, as it provides a standardized, one‑pixel‑wide representation of microglial morphology that is robust across heterogeneous imaging conditions and activation states.


In [ ]:
# ============================================================
# Fractal Dimension (Box-Counting Method)
# ============================================================
# Universal, reproducible, and robust implementation.
#
# Key design principles:
#   - Works with ANY microglial activation state
#   - Uses boolean masks (binary or skeleton) safely
#   - Avoids superfluous outputs
#   - Handles empty or degenerate masks gracefully
#   - Uses powers of two for reproducibility across datasets
#
# Returned values:
#   fd       → estimated fractal dimension
#   sizes    → box sizes used
#   counts   → number of non-empty boxes at each scale
#   coeffs   → (slope, intercept) from log–log regression
#
# Researchers may plug ANY microglial binary structure here:
#   - binary mask
#   - FD skeleton
#   - thresholded channels
# ============================================================

def fractal_dimension(binary_img):
    """
    Classical and permissive box-counting FD.
    No foreground cropping, no extra filtering.
    """
    binary = binary_img.astype(bool)

    if not np.any(binary):
        return 0.0, [], [], (0.0, 0.0)

    h, w = binary.shape
    max_size = min(h, w)

    sizes = []
    size = max_size

    while size >= 2:
        sizes.append(size)
        size //= 2

    counts = []

    for size in sizes:
        n_h = h // size
        n_w = w // size

        count = 0
        for i in range(n_h):
            for j in range(n_w):
                block = binary[i * size:(i + 1) * size, j * size:(j + 1) * size]
                if np.any(block):
                    count += 1

        counts.append(count)

    sizes_log = np.log(1 / np.array(sizes))
    counts_log = np.log(np.array(counts))

    coeffs = np.polyfit(sizes_log, counts_log, 1)
    fd = coeffs[0]

    return fd, sizes, counts, coeffs

# ============================================================
# Fractal Dimension Summary Table
# ============================================================

preferred_order = state_order
fd_rows = []

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        data = preprocessed_data[path]

        binary_uint8 = data["binary_uint8"]
        binary_bool = binary_uint8.astype(bool)

        if np.any(binary_bool):
            fd, _, _, _ = fractal_dimension(binary_bool)
        else:
            fd = None

        fd_rows.append({
            "filename": data["filename"],
            "sample_id": data["sample_id"],
            "cell_state": data["cell_state"],
            "fractal_dimension": fd
        })

    except Exception as e:
        fd_rows.append({
            "filename": os.path.basename(path),
            "sample_id": path,
            "cell_state": cell_state,
            "fractal_dimension": None,
            "error": str(e)
        })

df_fd = pd.DataFrame(fd_rows)

if preferred_order is not None:
    df_fd["order_key"] = df_fd["cell_state"].apply(
        lambda x: state_order.index(x) if x in state_order else len(state_order)
    )
    df_fd = df_fd.sort_values(by=["order_key", "filename"]).drop(columns=["order_key"])
else:
    df_fd = df_fd.sort_values(by=["cell_state", "filename"])

df_fd = df_fd[["filename", "sample_id", "cell_state", "fractal_dimension"]]

# Uses dot decimal and comma separator.
df_fd.to_csv(os.path.join(EXPORT_DIR, "Module 3 - fractal_dimension_summary_table.csv"), index=False)

df_fd

# ------------------------------------------------------------
# Apply ordering if provided
# ------------------------------------------------------------
if preferred_order is not None:
    df_fd["order_key"] = df_fd["cell_state"].apply(
        lambda x: state_order.index(x) if x in state_order else len(state_order)
    )

    df_fd = df_fd.sort_values(by=["order_key", "filename"]).drop(columns=["order_key"])

else:
    df_fd = df_fd.sort_values(by=["cell_state", "filename"])

### **Fractal Dimension Summary Table**

This block computes the fractal dimension (FD) for all images in the dataset using the box-counting method applied to the binary mask generated during preprocessing.

For each image:

- the precomputed binary mask is converted to boolean format
- the fractal dimension is estimated using the fractal_dimension function
- if no foreground structure is present, the FD value is set to None

To maintain clarity and usability, the results are stored in a compact summary table containing:

- the image filename
- the microglial activation state (cell_state)
- the estimated fractal dimension

Although the underlying FD computation produces additional intermediate outputs (box sizes, counts, and regression coefficients), only the final FD value is retained in this table to provide a concise and interpretable overview of structural complexity.

The table is optionally ordered according to the predefined class order to ensure consistent grouping across microglial activation states.

All results are exported as a CSV file:

'Module 3 - fractal_dimension_summary_table.csv'

This summary table provides a clean and reproducible representation of morphological complexity across images and conditions, and can be directly used for downstream statistical analysis, visualization, or machine learning workflows.

In [ ]:
# ------------------------------------------------------------
# Final clean table
# ------------------------------------------------------------
df_fd = df_fd[["filename", "sample_id", "cell_state", "fractal_dimension"]]

# ------------------------------------------------------------
# EXPORT TABLE (Fractal Dimension Summary Table)
# ------------------------------------------------------------
df_fd.to_csv(os.path.join(EXPORT_DIR, "Module 3 - fractal_dimension_summary_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_fd

**Table 1. Fractal dimension values across five canonical microglial states.**  
This table summarizes the box‑counting fractal dimension (FD) computed for each synthetic microglial image. FD captures global branching complexity and is highly sensitive to morphological state. Resting microglia exhibit the highest FD values (≈1.35–1.44), reflecting their dense, highly ramified arbors. Primed microglia show intermediate FD values (≈1.18–1.23), consistent with partial process retraction. Resolving microglia display moderately reduced complexity (≈0.91–1.02), aligning with their transitional morphology. Activated and ameboid microglia present the lowest FD values (≈0.72–0.93), indicative of compact, minimally branched structures. These results demonstrate that fractal dimension robustly discriminates microglial activation states and provides a quantitative, scale‑invariant descriptor of morphological complexity.


In [ ]:
# ============================================================
# Plot of Fractal Dimension
# ============================================================

plot_bars_or_boxplot(
    df=df_fd,
    value_col="fractal_dimension",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="Fractal Dimension (FD)",
    xlabel="Image Filename / Group",
    title="Fractal Dimension Summary",
    save_path=os.path.join(EXPORT_DIR, "Module 3 - bar_plot_of_fractal_dimension.png")
)

print("\n[INFO] ===== Module 3 Completed =====")

**Figure 3. Fractal dimension across microglial activation states.**  
This bar plot summarizes the fractal dimension (FD) of all microglial images, ordered by activation state. A clear gradient of morphological complexity emerges: resting microglia exhibit the highest FD values, reflecting their dense, highly ramified arbors; primed cells show intermediate complexity consistent with partial process retraction; resolving microglia occupy a transitional range; and both activated and ameboid cells display markedly reduced FD values, indicative of compact, minimally branched morphologies. The monotonic decrease in FD from resting to ameboid states demonstrates the sensitivity of fractal analysis to structural remodeling and highlights FD as a robust, scale‑invariant descriptor capable of discriminating microglial phenotypes with high fidelity.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">

## **Module 4 — Lacunarity Analysis (Gliding‑Box Method)**

This module computes lacunarity (Λ) as a measure of spatial heterogeneity in microglial structures using the classical gliding-box method.
It operates on the binary masks generated during preprocessing and provides both multiscale lacunarity curves and a compact summary metric for each image.

The module is designed to integrate directly with the outputs of the preprocessing stage and can be applied to both synthetic and real microglial datasets.

---

1. Lacunarity Computation
'compute_lacunarity(binary_img, box_sizes=[2, 4, 8, 16, 32, 64, 128, 256])'

This function computes lacunarity across multiple spatial scales using a sliding-window (gliding-box) approach.

- a. Input handling

- The input image is converted to uint8 format
- The function expects a binary structure (typically the preprocessing mask)

If the input contains no foreground pixels:

- lacunarity values are returned as NaN
- no valid box sizes are reported

- b. Box sizes

A predefined list of box sizes is used:

[2, 4, 8, 16, 32, 64, 128, 256]
- Box sizes larger than the image dimensions are automatically skipped
- The list can be modified depending on resolution or biological scale

- c. Gliding-window sampling

For each valid box size r:
- an 𝑟 × 𝑟 window is slid across the image
- the sum of foreground pixels is computed for every position
- these values form the distribution St

- d. Lacunarity formula

Lacunarity is computed as:

Λ(r) = ( Var(St) / (Mean(St))^2 ) + 1

where St represents the distribution of window sums.
- High Λ → heterogeneous, irregular structures
- Low Λ → compact, homogeneous structures

The function returns:

- lac_values (lacunarity values per box size)
- valid_sizes (box sizes actually used)

2. Lacunarity Curves for All Images

For each image in the dataset:

- the binary mask from preprocessing is used
- lacunarity is computed across all valid scales
- a lacunarity curve (Λ vs. box size) is generated

All curves are plotted in a grid layout:

- one subplot per image
- labeled by filename
- automatically adjusted to dataset size

The resulting figure is exported as:

'Module 4 - lacunarity_curves_for_all_images.png'

3. Lacunarity Summary Table

For downstream analysis, the module computes a single summary metric per image:

- mean lacunarity across all valid box sizes
'(np.nanmean(lac_values))'

A summary table is constructed containing:

- filename
- cell_state
- lacunarity_mean

The table is:

- ordered according to the predefined class order (state_order)
- cleaned to remove auxiliary columns

and exported as:

'Module 4 - lacunarity_summary_table.csv'

4. Visualization of Lacunarity Summary

The distribution of mean lacunarity values is visualized using the shared plotting utility:

'plot_bars_or_boxplot(...)'

Depending on dataset size, the results are displayed as:

- per-image bar plots, or
- grouped boxplots with individual data points

The figure is exported as:

'Module 4 - bar_plot_of_mean_lacunarity_values.png'

---

### **Interpretation in Microglial Morphology**

Lacunarity complements fractal dimension by capturing spatial heterogeneity rather than global scaling behavior.

In microglial analysis:

- highly ramified structures often exhibit higher lacunarity
- compact or ameboid morphologies tend to show lower lacunarity
- intermediate states may display transitional patterns

These trends reflect differences in branch distribution, gaps, and local density, although exact values depend on preprocessing and dataset characteristics.

---

### **Role in the Pipeline**

This module extends the structural analysis introduced by fractal dimension by adding a multiscale heterogeneity descriptor.

Together, fractal dimension and lacunarity provide complementary information about:

global complexity (FD)
spatial distribution and irregularity (Λ)

enabling a more complete characterization of microglial morphology across activation states and experimental conditions.

In [ ]:
# ============================================================
# Lacunarity Analysis (Gliding-Box Method)
# ============================================================
#
# Key design principles:
#   - Works with any microglial activation state
#   - Accepts any binary structure (mask or skeleton)
#   - Avoids unnecessary outputs
#   - Handles empty or degenerate masks safely
#   - Uses deterministic box sizes for reproducibility
#
# Returned values:
#   lac_values  → lacunarity values for each valid box size
#   valid_sizes → box sizes actually used in the computation
#
# Researchers may freely modify `box_sizes` depending on resolution, magnification, or biological scale of interest.
# ============================================================

print("\n[INFO] ===== Module 4 - Lacunarity Curves =====")

def compute_lacunarity(binary_img, box_sizes=[4, 8, 16, 32, 64, 128, 256]):
    """
    Fast lacunarity using an integral image.

    This avoids the very slow nested Python loops over every window.
    Recommended when images are resized to 1024x1024.
    """

    binary = binary_img.astype(np.uint8)
    h, w = binary.shape

    if not np.any(binary):
        return [np.nan] * len(box_sizes), []

    lac_values = []
    valid_sizes = []

    # Integral image:
    # integral[y, x] contains the sum of pixels above and left of (y, x)
    integral = np.pad(binary, ((1, 0), (1, 0)), mode="constant")
    integral = integral.cumsum(axis=0).cumsum(axis=1)

    for r in box_sizes:
        if r > h or r > w:
            continue

        # Fast sum of every r x r window
        sums = (
            integral[r:, r:]
            - integral[:-r, r:]
            - integral[r:, :-r]
            + integral[:-r, :-r]
        ).astype(np.float64)

        sums = sums.ravel()

        mean = sums.mean()
        var = sums.var()

        if mean == 0:
            lac = np.nan
        else:
            lac = var / (mean ** 2) + 1

        lac_values.append(lac)
        valid_sizes.append(r)

    return lac_values, valid_sizes

# ============================================================
# Lacunarity Curves for All Images
# ============================================================
# This block computes and plots lacunarity curves (Λ vs. box size) for each microglial image in the dataset.
#
# Design principles:
#   - Universal: works with any microglial activation state
#   - Reproducible: deterministic ordering and processing
#   - Minimalistic: only essential information is shown
#   - Robust: handles empty masks and processing errors safely
#
# Each subplot shows:
#   - box sizes used
#   - corresponding lacunarity values
#   - filename for identification
#
# Lacunarity helps distinguish microglial states:
#   - Resting microglia → higher Λ (heterogeneous, ramified)
#   - Activated / ameboid → lower Λ (compact, homogeneous)
#   - Primed / resolving → intermediate Λ
# ============================================================

lac_rows = []

n = len(all_paths)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for i, (path, cell_state) in enumerate(zip(all_paths, all_cell_states)):

    try:
        data = preprocessed_data[path]

        binary_uint8 = data["binary_uint8"]
        binary_bool = binary_uint8.astype(bool)

        lac_values, sizes = compute_lacunarity(binary_bool)

        lac_rows.append({
            "filename": data["filename"],
            "sample_id": data["sample_id"],
            "cell_state": data["cell_state"],
            "lacunarity_mean": np.nanmean(lac_values)
        })

        ax = axes[i]
        ax.plot(sizes, lac_values, marker="o")
        ax.set_title(data["sample_id"], fontsize=9)
        ax.set_xlabel("Box size (pixels)")
        ax.set_ylabel("Lacunarity (Λ)")

    except Exception as e:
        ax = axes[i]
        ax.text(0.5, 0.5, f"Error:\n{os.path.basename(path)}", ha="center")
        ax.axis("off")

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Lacunarity Curves for All Images)
# ------------------------------------------------------------

plt.savefig(os.path.join(EXPORT_DIR, "Module 4 - lacunarity_curves_for_all_images.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show() 

**Figure 4. Lacunarity profiles across microglial activation states.**  
This panel displays lacunarity (λ) as a function of box size for all microglial samples, revealing consistent state‑dependent differences in spatial heterogeneity. Lacunarity decreases monotonically with increasing box size across all cells, reflecting the expected reduction in gap sensitivity at larger spatial scales. Resting microglia exhibit the lowest lacunarity values, consistent with their dense, uniformly distributed branching patterns. Primed and resolving microglia show intermediate profiles, indicating partial structural heterogeneity associated with transitional morphologies. Activated and ameboid microglia display the highest lacunarity, reflecting large contiguous gaps and reduced arborization. Together, these curves demonstrate that lacunarity provides a sensitive multiscale descriptor of microglial structural organization, complementing fractal dimension by capturing differences in local irregularity and gap distribution across activation states.


In [ ]:
# ============================================================
# Lacunarity Summary Table
# ============================================================
# This table summarizes lacunarity for each microglial image using:
#   - filename
#   - group / microglial activation state
#   - mean lacunarity across valid box sizes
#
# The table is ordered according to a user-defined biological progression (state_order).
# ============================================================

df_lac = pd.DataFrame(lac_rows)

# ------------------------------------------------------------
# Apply ordering
# ------------------------------------------------------------
df_lac["order_key"] = df_lac["cell_state"].apply(
    lambda x: state_order.index(x) if x in state_order else len(state_order)
)

df_lac = df_lac.sort_values(by=["order_key", "filename"]).drop(columns=["order_key"])

# ------------------------------------------------------------
# Final clean table
# ------------------------------------------------------------
df_lac = df_lac[["filename", "sample_id", "cell_state", "lacunarity_mean"]]

# ------------------------------------------------------------
# EXPORT TABLE (Lacunarity Summary Table)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_lac.to_csv(os.path.join(EXPORT_DIR, "Module 4 - lacunarity_summary_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_lac  

**Table 2. Mean lacunarity values across microglial activation states.**  
This table reports the mean lacunarity (λ̄) for each microglial image, capturing differences in spatial heterogeneity and gap distribution across morphological states. Resting microglia show the lowest lacunarity values (≈7–10), reflecting their dense, uniformly ramified arbors with minimal large‑scale gaps. Primed microglia exhibit slightly higher λ̄ (≈17–28), consistent with early process retraction and emerging structural irregularity. Resolving microglia display intermediate lacunarity (≈36–64), reflecting partial recovery of arborization following activation. Activated and ameboid microglia show the highest λ̄ values (≈61–131), indicative of compact morphologies with large contiguous voids and reduced branching. These results highlight lacunarity as a sensitive multiscale descriptor that complements fractal dimension by quantifying local heterogeneity and structural discontinuity across microglial states.


In [ ]:
# ============================================================
# Plot of Mean Lacunarity Values
# ============================================================
plot_bars_or_boxplot(
    df=df_lac,
    value_col="lacunarity_mean",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="Mean Lacunarity (Λ)",
    xlabel="Image Filename / Group",
    title="Mean Lacunarity Across Groups",
    save_path=os.path.join(EXPORT_DIR, "Module 4 - bar_plot_of_mean_lacunarity_values.png")
)

print("\n[INFO] ===== Module 4 Completed =====")

**Figure 5. Mean lacunarity across microglial activation states.**  
This bar plot summarizes the mean lacunarity (λ̄) for all microglial samples, ordered by activation state. A clear stratification emerges: resting microglia exhibit the lowest lacunarity values, reflecting their dense, uniformly ramified arbors with minimal large‑scale gaps. Primed microglia show slightly elevated λ̄, consistent with early structural irregularities associated with partial process withdrawal. Resolving microglia occupy an intermediate range, reflecting transitional morphologies during recovery. Activated and ameboid microglia display the highest lacunarity values, indicative of compact morphologies with large contiguous voids and reduced branching. These results highlight lacunarity as a sensitive multiscale descriptor of spatial heterogeneity, complementing fractal dimension by capturing differences in gap distribution and local irregularity across microglial states.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## **Module 5 — Texture Metrics via GLCM Analysis**

This module computes texture descriptors from microglial images using the Gray-Level Co-occurrence Matrix (GLCM) and integrates them into the broader morphometric analysis pipeline.

In addition to feature extraction, the module:

- computes GLCM-based metrics for each image
- organizes results into a structured summary table
- generates diagnostic and group-level visualizations
- produces a final multi-metric panel combining fractal dimension, lacunarity, and texture descriptors

The analysis is applied to preprocessed grayscale images and their corresponding binary masks, ensuring that texture is evaluated within the segmented microglial structure.

---

1. Conceptual Role in Microglial Analysis

GLCM-based texture metrics capture fine-scale intensity organization and local spatial relationships, complementing:

- fractal dimension (global complexity)
- lacunarity (spatial heterogeneity)

These features are particularly useful for detecting subtle morphological transitions that are not fully captured by global structural descriptors.

Typical trends observed in microglial datasets include:

- ramified morphologies → higher contrast, lower homogeneity
- compact morphologies → lower contrast, higher homogeneity
- intermediate states → transitional texture profiles

These associations depend on image quality and preprocessing and should be interpreted in context.

2. GLCM Feature Computation
'compute_glcm_features(gray_img, mask=None, ...)'

This function computes GLCM features from a grayscale image, optionally restricted to a segmented region of interest.

- Region of Interest (ROI)

For each image, the analysis is performed on a cropped region defined by the bounding box of the binary mask:

- reduces background influence
- focuses the analysis on the microglial structure
- improves robustness of texture estimation

A binary mask is applied to further suppress non-relevant pixels.

- Normalization

The grayscale image is:

- normalized to the range [0, 1]
- re-normalized within the masked region (if a mask is provided)

This ensures consistent intensity scaling across images and datasets.

- Quantization

The normalized image is discretized into a fixed number of gray levels by default: 32 levels

This step is required for GLCM computation and controls the resolution of texture analysis.

- GLCM computation

The gray-level co-occurrence matrix is computed using:

- multiple distances: [1, 2, 4]
- multiple angles: [0, π/4, π/2, 3π/4]

This multi-directional, multi-scale approach ensures:

- robustness to orientation
- reduced directional bias

- Feature extraction

The following features are computed and averaged across all distances and angles:

contrast (local intensity variation)
homogeneity (smoothness and uniformity)
energy (textural regularity)
correlation (linear dependency of pixel intensities)

- Edge-case handling

The function explicitly handles degenerate cases:

empty images or masks → all values set to NaN
constant-intensity regions:
- contrast = 0
- homogeneity = 1
- energy = 1
- correlation = 0

This ensures numerical stability and reproducibility.

3. GLCM Summary Table

For each image, the computed features are stored in a structured table:

- filename
- cell_state
- contrast
- homogeneity
- energy
- correlation

The table is:

- ordered according to the predefined biological state order (state_order)
- cleaned to remove auxiliary columns

and exported as:

'Module 5 - glcm_summary_table.csv'

4. Diagnostic Visualization

The module generates diagnostic plots showing each GLCM metric across all images:

- one subplot per metric
- values plotted against filenames
- useful for quick inspection of variability and outliers

Exported as:

'Module 5 - diagnostic_plots_for_glcm_metrics.png'

5. Group-Level Visualization

Each GLCM metric is further visualized using ordered plots:

- bar plots or boxplots depending on dataset size
- grouped by microglial activation state
- consistent color mapping across groups

These plots facilitate comparison of texture properties across biological conditions.

6. Integrated Morphometric Panel

The module concludes with a 2×2 summary panel combining key descriptors:

A — Fractal Dimension
B — Mean Lacunarity
C — GLCM Contrast
D — GLCM Homogeneity

This panel provides a compact, multidimensional view of microglial morphology, integrating:

- global complexity
- spatial heterogeneity
- fine-scale texture

Exported as:

'Module 5 - final_figure_panel_fd_lacunarity_glcm_metrics.png'

---

### **Role in the Pipeline**

This module extends the morphometric framework by incorporating intensity-based texture descriptors into the analysis.

Together with fractal dimension and lacunarity, GLCM features enable a richer and more comprehensive characterization of microglial morphology across activation states and experimental conditions.


In [ ]:
# ============================================================
# GLCM Feature Extraction
# ============================================================
# This function computes GLCM-based texture metrics for a
# grayscale microglial image. Features are averaged across
# multiple directions and distances to ensure rotational
# invariance and robustness across activation states.
#
# Design principles:
#   - Universal: works with any microglial activation state
#   - Reproducible: fixed distances and angles
#   - Minimalistic: only essential features are returned
#   - Robust: handles empty or low-contrast images safely
#
# Researchers may adjust distances or angles depending on
# resolution or biological scale of interest.
# ============================================================

print("\n[INFO] ===== Module 5 - GLCM - Based Texture Metrics =====")

def compute_glcm_features(
    gray_img,
    mask=None,
    distances=[1, 2, 4],
    angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
    levels=32
):
    """
    Compute GLCM features on a cropped grayscale ROI.
    Optionally uses a binary mask to suppress background influence.
    """
    gray = gray_img.astype(np.float32)

    if gray.size == 0:
        return {
            "contrast": np.nan,
            "homogeneity": np.nan,
            "energy": np.nan,
            "correlation": np.nan
        }

    # Normalize to [0, 1]
    gray = gray - gray.min()
    if gray.max() > 0:
        gray = gray / gray.max()

    # Apply mask if provided
    if mask is not None:
        mask = mask.astype(bool)
        if np.any(mask):
            vals = gray[mask]
            vmin, vmax = vals.min(), vals.max()

            if vmax > vmin:
                gray = (gray - vmin) / (vmax - vmin)
                gray = np.clip(gray, 0, 1)
            else:
                gray = np.zeros_like(gray)

            gray = gray * mask.astype(np.float32)
        else:
            return {
                "contrast": np.nan,
                "homogeneity": np.nan,
                "energy": np.nan,
                "correlation": np.nan
            }

    # Quantize
    gray_q = np.floor(gray * (levels - 1)).astype(np.uint8)

    if np.std(gray_q) == 0:
        return {
            "contrast": 0.0,
            "homogeneity": 1.0,
            "energy": 1.0,
            "correlation": 0.0
        }

    glcm = graycomatrix(
        gray_q,
        distances=distances,
        angles=angles,
        levels=levels,
        symmetric=True,
        normed=True
    )

    return {
        "contrast": float(np.mean(graycoprops(glcm, "contrast"))),
        "homogeneity": float(np.mean(graycoprops(glcm, "homogeneity"))),
        "energy": float(np.mean(graycoprops(glcm, "energy"))),
        "correlation": float(np.mean(graycoprops(glcm, "correlation")))
    }

# ============================================================
# GLCM Summary Table
# ============================================================
# This block computes GLCM texture metrics for all microglial images and stores them in a summary table.
#
# Columns:
#   filename | cell_state | contrast | homogeneity | energy | correlation
# ============================================================

glcm_rows = []

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        data = preprocessed_data[path]

        img_resized = data["img_resized"]
        binary_uint8 = data["binary_uint8"]
        bbox = data["bbox"]

        r0, c0, r1, c1 = bbox
        gray = img_resized[r0:r1, c0:c1]
        binary_roi = binary_uint8[r0:r1, c0:c1].astype(bool)

        feats = compute_glcm_features(gray, mask=binary_roi)

        glcm_rows.append({
            "filename": data["filename"],
            "sample_id": data["sample_id"],
            "cell_state": data["cell_state"],
            **feats
        })

    except Exception as e:
        glcm_rows.append({
            "filename": os.path.basename(path),
            "sample_id": path,
            "cell_state": cell_state,
            "contrast": None,
            "homogeneity": None,
            "energy": None,
            "correlation": None,
            "error": str(e)
        })

df_glcm = pd.DataFrame(glcm_rows)

# ------------------------------------------------------------
# Apply ordering
# ------------------------------------------------------------

df_glcm["order_key"] = df_glcm["cell_state"].apply(
    lambda x: state_order.index(x) if x in state_order else len(state_order)
)

df_glcm = df_glcm.sort_values(by=["order_key", "filename"]).drop(columns=["order_key"])

# ------------------------------------------------------------
# Final clean table
# ------------------------------------------------------------
df_glcm = df_glcm[[
    "filename",
    "sample_id",
    "cell_state",
    "contrast",
    "homogeneity",
    "energy",
    "correlation"
]]

# ------------------------------------------------------------
# EXPORT TABLE (GLCM Summary Table)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_glcm.to_csv(os.path.join(EXPORT_DIR, "Module 5 - glcm_summary_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_glcm  

**Table 3. GLCM‑derived texture metrics across microglial activation states.**  
This table summarizes four second‑order texture features—contrast, homogeneity, energy, and correlation—computed from gray‑level co‑occurrence matrices (GLCMs) for each microglial sample. Resting microglia exhibit the highest contrast and energy values, reflecting the fine‑grained, highly ramified texture of their arborized processes. Primed and resolving microglia show intermediate texture signatures, consistent with partial process retraction and moderate structural irregularity. Activated and ameboid microglia display lower homogeneity and energy alongside elevated correlation, indicative of thicker, more uniform processes and reduced local intensity variation. These texture metrics capture activation‑dependent changes that are not fully reflected in geometric descriptors alone, highlighting the value of GLCM features as complementary indicators of microglial structural remodeling.


In [ ]:
# ============================================================
# Diagnostic Plots for GLCM Metrics
# ============================================================
# This block visualizes GLCM texture metrics across all microglial samples in the dataset.
#
# Each subplot shows:
#   - the metric values for each image
#   - filenames for identification
#
# These plots help reveal texture differences between
# microglial activation states:
#   - Resting microglia → higher contrast, lower homogeneity
#   - Activated / ameboid → high homogeneity, high energy
#   - Primed / resolving → intermediate profiles
#
# The goal is to provide a quick, interpretable overview of fine-scale intensity patterns across groups.
# ============================================================

metrics = ["contrast", "homogeneity", "energy", "correlation"]

df_diag = df_glcm.copy().replace({None: np.nan})

# Añadir sample_id único desde preprocessed_data
filename_to_sample = {}
for p in all_paths:
    d = preprocessed_data[p]
    filename_to_sample[(d["filename"], d["cell_state"])] = d["sample_id"]

df_diag["sample_id"] = [
    filename_to_sample.get((row["filename"], row["cell_state"]), row["filename"])
    for _, row in df_diag.iterrows()
]

# Orden consistente
df_diag["order_key"] = df_diag["cell_state"].apply(
    lambda x: state_order.index(x) if x in state_order else len(state_order)
)
df_diag = df_diag.sort_values(["order_key", "cell_state", "sample_id"]).reset_index(drop=True)
df_diag["xpos"] = np.arange(len(df_diag))

color_map = build_group_color_map(state_order)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    ax = axes[i]

    for state in state_order:
        sub = df_diag[df_diag["cell_state"] == state].dropna(subset=[metric])
        if sub.empty:
            continue

        ax.plot(
            sub["xpos"],
            sub[metric],
            marker="o",
            linestyle="-",
            linewidth=1.4,
            markersize=4.5,
            color=color_map[state],
            label=state,
            alpha=0.9
        )

    ax.set_xticks(df_diag["xpos"])
    ax.set_xticklabels(df_diag["sample_id"], rotation=75, ha="right", fontsize=7)
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f"{metric.capitalize()} across samples")
    ax.grid(axis="y", linestyle="--", alpha=0.3)

handles = [
    plt.Line2D([0], [0], color=color_map[state], marker="o", lw=2, label=state)
    for state in state_order if state in df_diag["cell_state"].unique()
]
fig.legend(handles=handles, title="Group", loc="upper right")

plt.tight_layout()
# ------------------------------------------------------------
# EXPORT FIGURE (Diagnostic Plots for GLCM Metrics)
# ------------------------------------------------------------
plt.savefig(os.path.join(EXPORT_DIR, "Module 5 - diagnostic_plots_for_glcm_metrics.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show()

**Figure 6. GLCM‑derived texture features across microglial activation states.**  
This multi‑panel figure shows four second‑order texture metrics—contrast, homogeneity, energy, and correlation—computed from gray‑level co‑occurrence matrices (GLCMs) for each microglial sample. Resting microglia exhibit the highest contrast and energy values, reflecting the fine‑grained, highly ramified texture characteristic of their complex arborization. Primed and resolving microglia display intermediate patterns, consistent with partial process retraction and moderate structural irregularity. Activated and ameboid microglia show lower homogeneity and energy alongside elevated correlation, indicative of thicker, more uniform processes and reduced local intensity variation. Together, these texture signatures reveal activation‑dependent shifts in microglial structural organization that complement geometric descriptors such as fractal dimension and lacunarity, providing an additional layer of sensitivity to subtle morphological remodeling.


In [ ]:
# ============================================================
# Ordered Bar Plots for GLCM Metrics
# ============================================================
# This version ensures full robustness by:
#   - replacing None values with NaN
#   - filtering invalid rows before plotting
#   - preserving microglial biological ordering
#   - keeping the visualization minimalistic and reproducible
#
# These plots help compare fine‑scale texture differences across:
#   - resting_microglia
#   - primed_microglia
#   - activated_microglia
#   - ameboid_microglia
#   - resolving_microglia
#
# GLCM metrics capture subtle intensity‑based signatures that complement fractal dimension and lacunarity.
# ============================================================

# Replace None with NaN for safe numeric plotting
df_plot = df_glcm.replace({None: np.nan})

# Metrics to visualize
metrics = ["contrast", "homogeneity", "energy", "correlation"]

# Deterministic color map per group
unique_states = df_plot["cell_state"].unique()
color_map = {
    state: plt.cm.tab10(i % 10)
    for i, state in enumerate(unique_states)
}

df_plot["color"] = df_plot["cell_state"].map(color_map)

metrics = ["contrast", "homogeneity", "energy", "correlation"]

for metric in metrics:
    plot_bars_or_boxplot(
        df=df_plot,
        value_col=metric,
        group_col="cell_state",
        filename_col="sample_id",
        state_order=state_order,
        ylabel=metric.capitalize(),
        xlabel="Image Filename / Group",
        title=f"{metric.capitalize()} Across Groups",
        save_path=os.path.join(EXPORT_DIR, f"Module 5 - ordered_bar_plot_glcm_{metric}.png")
    )

**Figure 7. GLCM texture metrics across microglial activation states: contrast, homogeneity, energy, and correlation.**  
This multi‑panel figure summarizes four second‑order texture descriptors derived from gray‑level co‑occurrence matrices (GLCMs), revealing complementary aspects of microglial structural organization across activation states.  
**Top panel – Contrast:** Resting microglia exhibit the highest contrast values, reflecting the fine‑grained, high‑frequency intensity variations produced by their dense and highly ramified processes. Contrast progressively decreases through primed and resolving states and reaches its lowest levels in activated and ameboid microglia, whose thick, compact processes generate smoother local intensity transitions.  
**Second panel – Homogeneity:** Homogeneity shows the inverse trend: resting microglia display the highest values, consistent with the smooth, continuous distribution of intensities generated by their arborized texture. Activated and ameboid microglia exhibit the lowest homogeneity, reflecting abrupt transitions and reduced structural regularity.  
**Third panel – Energy:** Energy, a measure of textural uniformity, is highest in resting microglia due to the repetitive fine‑scale patterns created by their extensive branching. Primed and resolving microglia show intermediate energy values, while activated and ameboid cells display the lowest energy, consistent with simplified, less repetitive intensity patterns.  
**Bottom panel – Correlation:** Correlation captures linear dependency between neighboring pixel intensities and is highest in ameboid and activated microglia, whose thick, homogeneous processes produce strong local intensity continuity. Resting microglia show the lowest correlation values, driven by the irregular, fine‑scale fluctuations characteristic of highly ramified arbors.  
Together, these four texture metrics provide a multidimensional view of microglial structural remodeling, capturing activation‑dependent changes that complement geometric descriptors such as fractal dimension and lacunarity.


In [ ]:
# ============================================================
# Final Figure Panel — FD, Lacunarity, and GLCM Metrics
# ============================================================
# This block generates a 2×2 panel summarizing the key morphometric descriptors for microglial morphology:
#
#   A — Fractal Dimension (ramification / complexity)
#   B — Mean Lacunarity (heterogeneity / gap distribution)
#   C — GLCM Contrast (fine‑scale intensity variation)
#   D — GLCM Homogeneity (texture smoothness / compactness)
#
# Design principles:
#   - Microglia‑specific: reflects activation‑dependent remodeling
#   - Reproducible: deterministic ordering and colors
#   - Minimalistic: only essential visual elements
#   - Publication‑ready: suitable for high‑impact journals
#
# This panel provides a compact, multidimensional summary of microglial structural organization across groups.
# ============================================================

def plot_on_axis_bars_or_boxplot(
    ax,
    df,
    value_col,
    group_col="cell_state",
    filename_col="sample_id",
    state_order=None,
    ylabel=None,
    title=None,
    image_threshold=5,
    group_threshold=5
):
    """
    Same logic as plot_bars_or_boxplot, but draws inside an existing axis.
    """
    df_plot = df.copy().dropna(subset=[value_col, group_col])

    if df_plot.empty:
        ax.set_title(title if title else value_col)
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center")
        ax.axis("off")
        return

    # Group ordering
    if state_order is None:
        groups = list(df_plot[group_col].dropna().unique())
    else:
        groups = [g for g in state_order if g in df_plot[group_col].dropna().unique()]
        groups += [g for g in df_plot[group_col].dropna().unique() if g not in groups]

    df_plot[group_col] = pd.Categorical(df_plot[group_col], categories=groups, ordered=True)

    sort_cols = [group_col]
    if filename_col in df_plot.columns:
        sort_cols.append(filename_col)

    df_plot = df_plot.sort_values(sort_cols).reset_index(drop=True)

    color_map = build_group_color_map(groups)
    use_box = should_use_boxplot(
        df_plot,
        group_col=group_col,
        image_threshold=image_threshold,
        group_threshold=group_threshold
    )

    # --------------------------------------------------------
    # CASE 1 — bar plot per image
    # --------------------------------------------------------
    if not use_box:
        df_plot["color"] = df_plot[group_col].astype(object).map(color_map)

        ax.bar(
            x=np.arange(len(df_plot)),
            height=df_plot[value_col],
            color=df_plot["color"],
            edgecolor="black",
            linewidth=0.7
        )

        ax.set_xticks(np.arange(len(df_plot)))
        if filename_col in df_plot.columns:
            ax.set_xticklabels(df_plot[filename_col], rotation=75, ha="right", fontsize=7)

        ax.grid(axis="y", linestyle="--", alpha=0.3)

    # --------------------------------------------------------
    # CASE 2 — boxplot + individual points
    # --------------------------------------------------------
    else:
        light_palette = {g: lighten_color(color_map[g], 0.6) for g in groups}

        sns.boxplot(
            data=df_plot,
            x=group_col,
            y=value_col,
            hue=group_col,
            order=groups,
            palette=light_palette,
            dodge=False,
            width=0.55,
            showfliers=False,
            linewidth=1.2,
            ax=ax
        )

        sns.stripplot(
            data=df_plot,
            x=group_col,
            y=value_col,
            hue=group_col,
            order=groups,
            palette=color_map,
            dodge=False,
            jitter=0.12,
            alpha=0.9,
            size=4.5,
            edgecolor="black",
            linewidth=0.4,
            ax=ax
        )

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", linestyle="--", alpha=0.3)

    ax.set_title(title if title else value_col, fontsize=12)
    ax.set_ylabel(ylabel if ylabel else value_col)

# ------------------------------------------------------------
# Prepare ordered dataframes
# ------------------------------------------------------------
df_fd_plot = df_fd.copy().dropna(subset=["fractal_dimension"])
df_lac_plot = df_lac.copy().dropna(subset=["lacunarity_mean"])
df_glcm_plot = df_glcm.copy().dropna(subset=["contrast", "homogeneity"])

# ------------------------------------------------------------
# Create the 2×2 panel
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

# ------------------------------------------------------------
# Panel A — Fractal Dimension
# ------------------------------------------------------------
plot_on_axis_bars_or_boxplot(
    ax=axes[0],
    df=df_fd_plot,
    value_col="fractal_dimension",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="FD",
    title="A — Fractal Dimension"
)

# ------------------------------------------------------------
# Panel B — Mean Lacunarity
# ------------------------------------------------------------
plot_on_axis_bars_or_boxplot(
    ax=axes[1],
    df=df_lac_plot,
    value_col="lacunarity_mean",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="Λ",
    title="B — Mean Lacunarity"
)

# ------------------------------------------------------------
# Panel C — GLCM Contrast
# ------------------------------------------------------------
plot_on_axis_bars_or_boxplot(
    ax=axes[2],
    df=df_glcm_plot,
    value_col="contrast",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="Contrast",
    title="C — GLCM Contrast"
)

# ------------------------------------------------------------
# Panel D — GLCM Homogeneity
# ------------------------------------------------------------
plot_on_axis_bars_or_boxplot(
    ax=axes[3],
    df=df_glcm_plot,
    value_col="homogeneity",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="Homogeneity",
    title="D — GLCM Homogeneity"
)

# ------------------------------------------------------------
# Legend (shared across all panels)
# ------------------------------------------------------------
present_states = []
for df_tmp in [df_fd_plot, df_lac_plot, df_glcm_plot]:
    present_states.extend(df_tmp["cell_state"].dropna().unique().tolist())

unique_states = []
for s in state_order:
    if s in present_states and s not in unique_states:
        unique_states.append(s)

color_map = {state: plt.cm.tab10(i % 10) for i, state in enumerate(unique_states)}

handles = [
    plt.Line2D([0], [0], color=color_map[state], lw=8, label=state)
    for state in unique_states
]
fig.legend(handles=handles, title="Group", loc="upper right")

plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE
# ------------------------------------------------------------
plt.savefig(
    os.path.join(EXPORT_DIR, "Module 5 - final_figure_panel_fd_lacunarity_glcm_metrics.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("\n[INFO] ===== Module 5 Completed =====")

**Figure 8. Multidomain morphometric descriptors reveal consistent activation‑dependent gradients in microglial structure.**  
This four‑panel figure integrates geometric, fractal, and texture‑based metrics to characterize microglial morphology across activation states.  
**(A) Fractal Dimension (FD):** Resting microglia exhibit the highest FD values, reflecting their dense, highly ramified arbors. FD progressively decreases through primed and resolving states and reaches its lowest levels in activated and ameboid microglia, whose compact morphologies lack fine‑scale branching.  
**(B) Mean Lacunarity (Λ):** Lacunarity shows the inverse trend: ameboid and activated microglia display the highest Λ values, indicative of large structural gaps and reduced arborization. Resting microglia show minimal lacunarity, consistent with their uniform, space‑filling branching patterns.  
**(C) GLCM Contrast:** Resting microglia exhibit the highest contrast values, driven by the fine‑grained intensity variations produced by their complex arborization. Activated and ameboid microglia show markedly lower contrast, reflecting smoother, more homogeneous textures associated with simplified morphologies.  
**(D) GLCM Homogeneity:** Homogeneity mirrors the structural regularity of each state: resting microglia show the highest values, while ameboid microglia exhibit the lowest, consistent with abrupt intensity transitions and reduced structural continuity.  
Together, these complementary metrics reveal a coherent multiscale signature of microglial activation, demonstrating that geometric complexity, spatial heterogeneity, and texture organization shift in parallel as microglia transition from resting to activated and ameboid states.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## **Module 6 — Multiscale Fractal Spectrum (MFS)**

This module computes the Multiscale Fractal Spectrum (MFS), a scale-resolved descriptor of microglial structural complexity that extends the information provided by a single fractal dimension.
Instead of reducing morphology to one global value, the MFS quantifies how structural occupancy changes across multiple spatial scales.

Within this pipeline, the MFS is computed from the FD-oriented skeleton generated during preprocessing, providing a simplified but structurally informative representation of each microglial cell.

---

### **Conceptual relevance in microglia**

A single fractal dimension captures global complexity, but it may overlook how structural organization differs between fine, intermediate, and coarse spatial scales.
The MFS addresses this limitation by estimating a scale-wise fractal descriptor across a sequence of box sizes.

In microglial morphology, this is useful because different activation states often differ not only in total complexity, but also in how that complexity is distributed across scales. Typical tendencies may include:

- Resting microglia (broader multiscale complexity)
- Primed microglia (reduced fine-scale complexity)
- Activated microglia (flatter spectra associated with more compact morphologies)
- Ameboid microglia (minimal multiscale variation)
- Resolving microglia (partial re-expansion of multiscale structure)

These trends should be interpreted as biologically meaningful tendencies rather than fixed rules.

---

### **Computational method**

The function multiscale_fractal_spectrum(binary_img, scales=[2, 4, 8, 16, 32, 64, 128, 256]) computes the MFS from a binary structure using a simple and deterministic multiscale procedure.

For each scale 𝑠:

- The input is converted to boolean format.
- Scales larger than the image dimensions are skipped automatically.
- The image is cropped so that its height and width are divisible by 𝑠.
- The cropped image is reshaped into non-overlapping 𝑠 × 𝑠 blocks.
- Each block is reduced by taking its maximum value, producing a coarse-grained binary representation.
- The number of occupied coarse blocks is counted.
- The scale-wise descriptor is computed as:

FD_s = frac = log(count) \ log(1/s)

If no occupied blocks are present at a given scale, the corresponding MFS value is set to NaN.

The function returns:

- spectrum (the list of MFS values across valid scales)
- valid_scales (the subset of scales actually used)

---

## **Input used in this pipeline**

Although the function can operate on any binary image, in this notebook it is applied specifically to:

- the FD-oriented skeleton produced by preprocess_image()
- stored in preprocessed_data[path]["skeleton_fd_uint8"]

This choice provides:

- a compact structural representation
- stable multiscale behavior
- consistency with the preceding fractal analysis

---

### **Outputs**

This module generates several complementary outputs:

- MFS table for all images, with one column per scale
- MFS curves per image
- comparative group-level MFS curves
- heatmap of the multiscale spectrum across all images

The exported table has the structure:

- filename
- cell_state
- mfs_scale_2
- mfs_scale_4
- mfs_scale_8
...

and is saved as:

'Module 6 - mfs_table.csv'

---

### **Visualizations produced**

The module generates three main visual summaries:

1. MFS curves per image

Each image is displayed in its own subplot, showing the MFS values as a function of scale.
This provides an image-level view of multiscale complexity.

Saved as:

'Module 6 - mfs_curves_per_image.png'

2. Comparative MFS by group

The MFS table is converted to long format, and the mean spectrum is computed for each cell_state × scale combination.
This enables direct comparison of average multiscale profiles across groups.

Saved as:

'Module 6 - comparative_mfs_by_group.png'

3. Heatmap of the MFS

A heatmap is generated using the scale-wise MFS values for all images, with:

- rows = images
- columns = MFS descriptors by scale

This provides a compact overview of multiscale variation across the dataset.

Saved as:

'Module 6 - heatmap_of_the_MFS.png'

---

### **Role in the pipeline**

The MFS complements classical fractal dimension by adding scale-dependent structural information.
Whereas single-value FD summarizes overall complexity, the MFS describes how complexity is distributed across different spatial resolutions.

Together with lacunarity, GLCM texture metrics, and the other descriptors in the pipeline, the MFS contributes to a more complete quantitative characterization of microglial morphology across activation states and experimental conditions

In [ ]:
# ============================================================
#  Multiscale Fractal Spectrum (MFS)
# ============================================================
# This version is robust, minimalistic, and works with any skeletonized microglial morphology.
#
# Microglial relevance:
#   - Resting microglia → broad multiscale complexity
#   - Primed microglia → reduced fine-scale complexity
#   - Activated microglia → flatter spectra (compact morphology)
#   - Ameboid microglia → minimal multiscale variation
#   - Resolving microglia → re-expansion of multiscale structure
#
# Researchers may adjust `scales` depending on image resolution.
# ============================================================

print("\n[INFO] ===== Module 6 - Multiscale Fractal Spectrum (MFS) =====")

def multiscale_fractal_spectrum(binary_img, scales=[2, 4, 8, 16, 32, 64, 128, 256]):
    """
    Classical simple multiscale fractal spectrum.
    """
    binary = binary_img.astype(bool)
    h, w = binary.shape

    spectrum = []
    valid_scales = []

    for s in scales:
        if s > h or s > w:
            continue

        H = h // s
        W = w // s

        Z = binary[:H * s, :W * s]
        Z = Z.reshape(H, s, W, s).max(axis=(1, 3))

        count = np.sum(Z)

        if count > 0:
            fd_s = np.log(count) / np.log(1 / s)
        else:
            fd_s = np.nan

        spectrum.append(fd_s)
        valid_scales.append(s)

    return spectrum, valid_scales

# ============================================================
# Multiscale Fractal Spectrum (MFS) Table
# ============================================================
# Computes the MFS for each microglial image using the skeleton produced by preprocess_image().
#  Skeletons ensure:
#   - non‑empty structure
#   - stable multiscale behavior
#   - reproducibility across activation states
#
# Output:
#   filename | cell_state | mfs_scale_2 | mfs_scale_4 | ...
#
# This table provides a scale‑resolved signature of microglial complexity, capturing differences between:
#   - resting_microglia (broad spectra)
#   - primed_microglia (reduced fine‑scale complexity)
#   - activated_microglia (flattened spectra)
#   - ameboid_microglia (minimal multiscale variation)
#   - resolving_microglia (re‑emerging multiscale structure)
# ============================================================

mfs_rows = []

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        data = preprocessed_data[path]

        skel = data["skeleton_fd_uint8"].astype(bool)
        spectrum, scales = multiscale_fractal_spectrum(skel)

        mfs_rows.append({
            "filename": data["filename"],
            "cell_state": data["cell_state"],
            **{f"mfs_scale_{s}": spectrum[j] for j, s in enumerate(scales)}
        })

    except Exception as e:
        print("Error processing:", path, e)
        continue

df_mfs = pd.DataFrame(mfs_rows)

# ------------------------------------------------------------
# EXPORT TABLE (MFS Table)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_mfs.to_csv(os.path.join(EXPORT_DIR, "Module 6 - mfs_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_mfs 

**Table 4. Multiscale Fractal Spectrum (MFS) across microglial activation states.**  
This table reports the multiscale fractal spectrum (MFS) values computed at increasing box sizes (2–64 px), providing a scale‑dependent characterization of microglial structural complexity. Resting microglia exhibit the most negative MFS values across all scales, reflecting their rich multiscale branching architecture and high structural variability. Primed and resolving microglia show intermediate MFS profiles, consistent with partial process retraction and transitional remodeling. Activated and ameboid microglia display the least negative values, indicating reduced multiscale complexity and simplified arborization. The progressive flattening of the MFS curve from resting to ameboid states highlights the sensitivity of multiscale fractal analysis to hierarchical structural organization, capturing differences that extend beyond single‑scale fractal dimension and complement other morphometric descriptors such as lacunarity and GLCM texture metrics.


In [ ]:
# ============================================================
# MFS Curves per Image
# ============================================================
# This block visualizes the Multiscale Fractal Spectrum for each microglial image.
#
# Each subplot shows:
#   - the scales used (box sizes)
#   - the corresponding multiscale fractal descriptors
#   - the filename for identification
#
# Biological interpretation:
#   - Resting microglia → broad, high‑variance spectra
#   - Primed microglia → reduced fine‑scale complexity
#   - Activated microglia → flatter spectra (compact morphology)
#   - Ameboid microglia → minimal multiscale variation
#   - Resolving microglia → re‑emergence of multiscale structure
#
# These curves provide a scale-resolved view of microglial complexity that complements FD, lacunarity, and GLCM metrics.
# ============================================================

n = len(df_mfs)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for i, row in df_mfs.iterrows():
    ax = axes[i]

    scales = [int(col.split("_")[-1]) for col in df_mfs.columns if col.startswith("mfs_scale_")]
    values = [row[f"mfs_scale_{s}"] for s in scales]

    ax.plot(scales, values, marker="o")
    ax.set_title(row["filename"], fontsize=9)
    ax.set_xlabel("Scale (box size)")
    ax.set_ylabel("MFS value")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR, "Module 6 - mfs_curves_per_image.png"), dpi=300, bbox_inches="tight")
plt.show()

**Figure 9. Multiscale Fractal Spectrum (MFS) reveals hierarchical differences in microglial structural complexity across activation states.**  
This figure displays the multiscale fractal spectrum (MFS) for all microglial samples, showing how log(P) values evolve across increasing box sizes. Each curve captures the scale‑dependent distribution of structural detail within a single cell. Resting microglia exhibit the steepest and most negative MFS profiles, reflecting their rich hierarchical branching and high complexity across spatial scales. Primed and resolving microglia show intermediate slopes, consistent with partial process retraction and transitional remodeling. Activated and ameboid microglia display the flattest curves, indicating reduced multiscale complexity and simplified arborization dominated by coarse structural features. The systematic flattening of MFS curves from resting to ameboid states demonstrates the sensitivity of multiscale fractal analysis to hierarchical organization, capturing differences that extend beyond single‑scale fractal dimension and providing a powerful descriptor of microglial activation‑dependent morphological remodeling.


In [ ]:
# ============================================================
# Comparative Multiscale Fractal Spectrum by Group
# ============================================================
# This block compares the Multiscale Fractal Spectrum across groups.
#
# Interpretation:
#   - Resting microglia → broad, high‑variance spectra
#   - Primed microglia → reduced fine‑scale complexity
#   - Activated microglia → flatter spectra (compact morphology)
#   - Ameboid microglia → minimal multiscale variation
#   - Resolving microglia → re‑emergence of multiscale structure
#
# This visualization highlights how microglial complexity
# evolves across spatial scales and differs between states.
# ============================================================

# Melt table into long format
df_long = df_mfs.melt(
    id_vars=["filename", "cell_state"],
    var_name="scale_label",
    value_name="MFS_s"
)

# Extract numeric scale
df_long["scale"] = df_long["scale_label"].str.extract(r"(\d+)").astype(int)

# Compute mean per cell_state × scale
df_grouped = df_long.groupby(["cell_state", "scale"], as_index=False)["MFS_s"].mean()

# Deterministic colors per mgroup
unique_states = df_grouped["cell_state"].unique()
color_map = {state: plt.cm.tab10(i % 10) for i, state in enumerate(unique_states)}

plt.figure(figsize=(8, 6))

for state in unique_states:
    sub = df_grouped[df_grouped["cell_state"] == state]
    plt.plot(sub["scale"], sub["MFS_s"], marker="o", color=color_map[state], label=state)

plt.xlabel("Scale (box size)")
plt.ylabel("Mean MFS value")
plt.title("Multiscale Fractal Spectrum by Group")
plt.legend(title="Group")
plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Comparative MFS by group)
# ------------------------------------------------------------
plt.savefig(os.path.join(EXPORT_DIR, "Module 6 - comparative_mfs_by_group.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show()

**Figure 10. Multiscale Fractal Spectrum (MFS) averaged by microglial activation state.**  
This figure shows the mean multiscale fractal spectrum (MFS) for each microglial state, illustrating how structural complexity evolves across spatial scales. Resting microglia display the most negative and steeply declining MFS curve, reflecting their rich hierarchical branching and high complexity across fine and coarse scales. Primed and resolving microglia exhibit intermediate profiles, consistent with partial process retraction and transitional remodeling during activation or recovery. Activated and ameboid microglia show the flattest curves, indicating reduced multiscale complexity and simplified arborization dominated by coarse structural features. The clear separation between curves demonstrates that MFS captures hierarchical organization beyond single‑scale fractal dimension, providing a powerful multiscale descriptor of microglial activation‑dependent morphological remodeling.


In [ ]:
# ============================================================
# Heatmap of the Multiscale Fractal Spectrum
# ============================================================
# This heatmap provides a compact visualization of the MFS across all microglial images. 
# Each row corresponds to one image, and each column corresponds to a multiscale descriptor (FD_s at a given box size).
#
# Biological interpretation:
#   - Resting microglia → high values at fine scales, broad profile
#   - Primed microglia → reduced fine-scale complexity
#   - Activated microglia → flatter, lower MFS across scales
#   - Ameboid microglia → minimal multiscale variation
#   - Resolving microglia → re-expansion of multiscale structure
#
# Design principles:
#   - Microglia-specific but universally applicable
#   - Reproducible: deterministic ordering and color mapping
#   - Minimalistic: only essential information is shown
#   - Customizable: colormap, ordering, normalization
# ============================================================

# ------------------------------------------------------------
# Select only numeric MFS columns
# (all columns starting with "mfs_scale_")
# ------------------------------------------------------------
mfs_cols = [col for col in df_mfs.columns if col.startswith("mfs_scale_")]
numeric_data = df_mfs[mfs_cols].values

# ------------------------------------------------------------
# Create the heatmap
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))

plt.imshow(numeric_data, cmap="viridis", aspect="auto")
plt.colorbar(label="MFS value")

# Row labels: filenames
plt.yticks(
    ticks=np.arange(len(df_mfs)),
    labels=df_mfs["filename"],
    fontsize=7
)

# Column labels: scale descriptors
plt.xticks(
    ticks=np.arange(len(mfs_cols)),
    labels=mfs_cols,
    rotation=45,
    ha="right",
    fontsize=8
)

plt.title("MFS — Heatmap")
plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Heatmap of the MFS)
# ------------------------------------------------------------
plt.savefig(os.path.join(EXPORT_DIR, "Module 6 - heatmap_of_the_MFS.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show()  

print("\n[INFO] ===== Module 6 Completed =====")

**Figure 11. Multiscale Fractal Spectrum (MFS) heatmap reveals state‑dependent gradients of hierarchical complexity in microglia.**  
This heatmap visualizes the multiscale fractal spectrum (MFS) across all microglial samples and spatial scales, providing a compact representation of hierarchical structural complexity. Each row corresponds to a single microglial cell, and each column represents an MFS scale (2–64 px). Resting microglia exhibit the most negative MFS values across all scales, forming a distinct dark‑purple band that reflects their rich multiscale branching architecture. Primed and resolving microglia show intermediate patterns, with moderately negative values indicating partial loss or recovery of hierarchical structure. Activated and ameboid microglia display the least negative values, forming bright yellow–green regions that correspond to simplified, coarse‑scale morphologies with minimal fine‑scale detail. The clear stratification of color patterns across states demonstrates that MFS captures robust, scale‑dependent signatures of microglial activation, complementing single‑scale fractal dimension and enhancing the multiscale characterization of microglial morphology.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## Module 7 — Graph‑Based Morphometrics from Skeletonized Microglial Structures

This module extracts graph-theoretic morphometric descriptors from the skeletonized morphology of each microglial image.
In this representation, each skeleton pixel is treated as a graph node, and edges are created between neighboring pixels according to a deterministic 8-connectivity rule.

This graph-based representation captures the branching architecture and topological organization of microglia and complements the structural, spatial, and textural descriptors computed in previous modules, including fractal dimension, lacunarity, GLCM metrics, and the multiscale fractal spectrum.

---

### **Conceptual relevance in microglia**

Graph-based morphometrics provide a topological view of microglial morphology.
Whereas fractal and lacunarity metrics describe global complexity and spatial heterogeneity, graph descriptors quantify how a microglial cell branches, connects, and extends as a skeletonized network.

Typical morphological tendencies may include:

- Resting microglia (many endpoints, many bifurcations, larger graph extent)
- Primed microglia (reduced distal branching)
- Activated microglia (simplified arbor and shorter topological extent)
- Ameboid microglia (minimal graph structure)
- Resolving microglia (partial re-emergence of branching complexity)

These trends are biologically interpretable but depend on the quality of preprocessing and skeleton extraction.

---

### **Input used in this pipeline**

In this notebook, graph metrics are computed from:

'preprocessed_data[path]["skeleton_visual_uint8"]'

This means the module uses the visual skeleton generated during preprocessing, not the FD-oriented skeleton used in the multiscale fractal analysis.

Computational steps

1. Neighbor counting

A local 8-neighbor counting scheme is applied using convolution with a dedicated kernel.
This provides the number of neighboring skeleton pixels for each foreground pixel.

This step is used to identify:

- endpoints — skeleton pixels with exactly 1 neighbor
- bifurcations — skeleton pixels with 3 or more neighbors

2. Graph construction

The function skeleton_to_graph(skel) converts the skeleton into a NetworkX graph:

- each skeleton pixel becomes a node
- edges are added between connected neighboring pixels
- edge construction is performed deterministically while avoiding duplicate edges

This yields an undirected graph representation of the microglial skeleton.

3. Graph morphometric computation

The function graph_morphometrics(skel) computes a compact set of graph descriptors:

- endpoints (number of terminal skeleton nodes)
- bifurcations (number of branching nodes)
- total_nodes (total number of graph nodes)
- total_edges (total number of graph edges)
- total_length (simple graph-length proxy equal to the number of edges)
- diameter (graph diameter of the largest connected component)
- avg_degree (average node degree across the full graph)

If the skeleton is empty, the function returns zeros for count-based metrics and NaN for diameter and average degree.

For robustness:

- the diameter is computed only on the largest connected component
- if that component contains more than 2000 nodes, the diameter is set to NaN to avoid excessive computation
- Outputs generated

This module produces several outputs:

- Graph morphometrics table

For each image, the extracted graph metrics are stored in a summary table containing:

- filename
- cell_state
- endpoints
- bifurcations
- total_nodes
- total_edges
- total_length
- diameter
- avg_degree

This table is exported as:

'Module 7 - graph_based_morphometrics_table.csv'

- Graph overlay visualization

A visualization panel is generated in which:

- the skeleton is shown in grayscale
- graph nodes are overlaid in red

This provides a direct visual check of the graph representation extracted from each microglial skeleton.

Exported as:

'Module 7 - graph_overlay_on_skeletonized_microglial_structures.png'

- Master morphometrics table

This module also builds a unified dataframe, df_master_basic, by merging the previously computed morphometric tables:

- fractal dimension
- lacunarity
- GLCM metrics
- multiscale fractal spectrum
- graph-based morphometrics

This integrated table is indexed by:

- filename
- cell_state

and exported as:

'Module 7 - build_master_dataframe_df_master_basic.csv'

This merged dataframe serves as the base structure for later comparative analysis, statistics, and machine learning.

- Comparative plots of graph morphometrics

The module generates a multi-panel figure comparing graph-based descriptors across samples and groups.
Each metric is visualized using the shared plotting logic, which automatically chooses between per-image bar plots and grouped boxplots depending on dataset size.

Metrics visualized:

- endpoints
- bifurcations
- total_nodes
- total_edges
- total_length
- diameter
- avg_degree

Exported as:

'Module 7 - comparative_plots_of_graph_morphometrics_microglia.png'

- Group-level graph profile comparison

A final group-level comparison plot is generated by computing the mean value of each graph metric within each microglial state and plotting these profiles across metrics.

This provides a compact summary of topological differences between groups.

Exported as:

'Module 7 - group_level_comparison_of_graph_morphometrics.png'

---

### **Role in the pipeline**

This module adds a network-theoretic layer to the morphometric analysis of microglia.
By converting skeletonized cells into graphs, it quantifies branching topology, connectivity, and structural extent in a way that complements the geometric and intensity-based descriptors computed earlier.

Together with the merged master dataframe generated in this module, these features contribute to a multidimensional and analysis-ready representation of microglial morphology across activation states and experimental conditions.

In [ ]:
# ============================================================
# Graph-Based Morphometrics
# ============================================================
# This module extracts graph-theoretic descriptors from the skeleton produced in the preprocessing pipeline.
#
# Microglial relevance:
#   - Resting microglia → many endpoints, many bifurcations,
#     long diameters, high connectivity
#   - Primed microglia → reduced distal branching
#   - Activated microglia → simplified arbor, short diameter
#   - Ameboid microglia → minimal graph structure
#   - Resolving microglia → re-expansion of branching
#
# Design principles:
#   - Universal: works with any microglial activation state
#   - Reproducible: deterministic 8-connectivity rules
#   - Minimalistic: only essential morphometric descriptors
#   - Accessible: clear English annotations for researchers
#
# Researchers may extend this module by adding:
#   - subtree size distributions
#   - cycle detection
#   - geodesic path statistics
# ============================================================

print("\n[INFO] ===== Module 7 - Graph-Based Morphometrics =====")

# ------------------------------------------------------------
# 8-connectivity kernel for neighbor counting
# ------------------------------------------------------------
kernel = np.array([
    [1, 1, 1],
    [1,10, 1],
    [1, 1, 1]
])


def neighbor_count(skel):
    conv = convolve(skel.astype(np.uint8), kernel, mode="constant", cval=0)
    return conv - 10


def skeleton_to_graph(skel):
    G = nx.Graph()
    skel = skel.astype(bool)

    ys, xs = np.where(skel)
    nodes = list(zip(ys, xs))
    G.add_nodes_from(nodes)

    node_set = set(nodes)

    # Only 4 duplicates "to the front" to avoid duplicate edges
    forward_neighbors = [
        (-1,  1),
        ( 0,  1),
        ( 1,  1),
        ( 1,  0),
    ]

    for y, x in nodes:
        for dy, dx in forward_neighbors:
            nb = (y + dy, x + dx)
            if nb in node_set:
                G.add_edge((y, x), nb)

    return G


def fast_largest_component_diameter(G):
    """
    Fast approximate graph diameter using two BFS passes.

    For skeleton-like graphs this is much faster than nx.diameter().
    It avoids returning NaN when the largest component has >2000 nodes.
    """

    if G.number_of_nodes() == 0:
        return np.nan

    try:
        largest_cc = max(nx.connected_components(G), key=len)
        subG = G.subgraph(largest_cc).copy()

        if subG.number_of_nodes() == 0:
            return np.nan

        if subG.number_of_nodes() == 1:
            return 0

        # First BFS: arbitrary node -> farthest node
        start = next(iter(subG.nodes))
        lengths = nx.single_source_shortest_path_length(subG, start)
        farthest = max(lengths, key=lengths.get)

        # Second BFS: farthest node -> farthest distance
        lengths = nx.single_source_shortest_path_length(subG, farthest)
        diameter_approx = max(lengths.values())

        return int(diameter_approx)

    except Exception:
        return np.nan


def graph_morphometrics(skel):
    skel = skel.astype(bool)

    if not np.any(skel):
        return dict(
            endpoints=0,
            bifurcations=0,
            total_nodes=0,
            total_edges=0,
            total_length=0,
            diameter=np.nan,
            avg_degree=np.nan,
            avg_junction_degree=np.nan,
            endpoint_density=np.nan,
            bifurcation_density=np.nan,
            branching_index=np.nan,
            largest_component_nodes=0,
            n_components=0
        )

    neigh = neighbor_count(skel)

    endpoints = int(np.sum(skel & (neigh == 1)))
    bifurcations = int(np.sum(skel & (neigh >= 3)))

    G = skeleton_to_graph(skel)

    total_nodes = G.number_of_nodes()
    total_edges = G.number_of_edges()
    total_length = total_edges

    # Connected components
    try:
        components = list(nx.connected_components(G))
        n_components = len(components)
        largest_component_nodes = len(max(components, key=len)) if components else 0
    except Exception:
        n_components = np.nan
        largest_component_nodes = np.nan

    # Fast diameter instead of nx.diameter cutoff at 2000 nodes
    diameter = fast_largest_component_diameter(G)

    # Degree metrics
    if total_nodes > 0:
        degrees = np.array([deg for _, deg in G.degree()], dtype=float)

        # This will usually be close to 2 in skeleton graphs.
        avg_degree = float(np.mean(degrees))

        # More biologically useful: average degree only at branching nodes.
        junction_degrees = degrees[degrees >= 3]
        if len(junction_degrees) > 0:
            avg_junction_degree = float(np.mean(junction_degrees))
        else:
            avg_junction_degree = 0.0

        endpoint_density = endpoints / total_nodes
        bifurcation_density = bifurcations / total_nodes

        if endpoints > 0:
            branching_index = bifurcations / endpoints
        else:
            branching_index = np.nan

    else:
        avg_degree = np.nan
        avg_junction_degree = np.nan
        endpoint_density = np.nan
        bifurcation_density = np.nan
        branching_index = np.nan

    return dict(
        endpoints=endpoints,
        bifurcations=bifurcations,
        total_nodes=total_nodes,
        total_edges=total_edges,
        total_length=total_length,
        diameter=diameter,
        avg_degree=avg_degree,
        avg_junction_degree=avg_junction_degree,
        endpoint_density=endpoint_density,
        bifurcation_density=bifurcation_density,
        branching_index=branching_index,
        largest_component_nodes=largest_component_nodes,
        n_components=n_components
    )

# ============================================================
# Apply graph metrics to all group images
# ============================================================

graph_rows = []

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        data = preprocessed_data[path]

        skel = data["skeleton_visual_uint8"].astype(bool)

        metrics = graph_morphometrics(skel)

        graph_rows.append({
            "filename": data["filename"],
            "cell_state": data["cell_state"],
            **metrics
        })

    except Exception as e:
        print("Error processing:", path, e)
        continue

df_graph = pd.DataFrame(graph_rows)

# ------------------------------------------------------------
# EXPORT TABLE (Graph-Based Morphometrics)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_graph.to_csv(os.path.join(EXPORT_DIR, "Module 7 - graph_based_morphometrics_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_graph

**Table 5. Graph‑theoretic descriptors of microglial arbor architecture across activation states.**  
This table summarizes topological and geometric properties extracted from skeleton‑based graph representations of each microglial cell. Resting microglia exhibit the highest numbers of endpoints, bifurcations, nodes, edges, and total branch length, reflecting their dense, highly ramified arborization. Their large diameters and moderately elevated average degree further indicate extensive branching distributed across long spatial extents. Primed and resolving microglia show intermediate graph complexity, consistent with partial process retraction and transitional remodeling during activation or recovery. Activated microglia display reduced branching and shorter overall arbor length, while ameboid microglia show the most simplified graphs, with minimal nodes, edges, and near‑linear structures. The systematic reduction in graph complexity from resting to ameboid states highlights the sensitivity of graph‑theoretic metrics to activation‑dependent morphological remodeling and complements fractal, lacunarity, and texture‑based descriptors.


In [ ]:
# ============================================================
# Graph Overlay on Skeletonized Microglial Structures
# ============================================================
# This block visualizes the graph representation of each microglial skeleton by overlaying graph nodes on top of the skeleton.
#
# Biological interpretation:
#   - Resting microglia → dense arbor, many nodes, rich topology
#   - Primed microglia → reduced distal branching
#   - Activated microglia → compact, simplified graph
#   - Ameboid microglia → minimal or near-zero graph structure
#   - Resolving microglia → re-expansion of graph connectivity
#
# Design principles:
#   - Microglia-specific but universally applicable
#   - Reproducible: deterministic skeleton + graph extraction
#   - Minimalistic: nodes only (edges optional for clarity)
# ============================================================

# Number of images
n = len(all_paths)

# Dynamic grid: 4 columns, as many rows as needed
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for i, (path, cell_state) in enumerate(zip(all_paths, all_cell_states)):

    try:
        data = preprocessed_data[path]

        skel = data["skeleton_visual_uint8"].astype(bool)
        G = skeleton_to_graph(skel)

        ax = axes[i]
        ax.imshow(skel, cmap="gray")
        ax.set_title(f"{data['filename']} ({data['cell_state']})", fontsize=9)
        ax.axis("off")

        ys = [node[0] for node in G.nodes()]
        xs = [node[1] for node in G.nodes()]

        ax.scatter(xs, ys, s=1, c="red")

    except Exception as e:
        print("Error processing:", path, e)
        continue

# Hide unused subplots (if any)
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Graph Overlay on Skeletonized Microglial Structures)
# ------------------------------------------------------------
plt.savefig(os.path.join(EXPORT_DIR, "Module 7 - graph_overlay_on_skeletonized_microglial_structures.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show()  

**Figure 12. Graph‑based representation of microglial arbor architecture overlaid on skeletonized structures.**  
This figure shows the graph reconstruction of each microglial cell obtained by overlaying graph nodes and edges directly onto the corresponding skeletonized morphology. Endpoints, bifurcations, and connecting segments are rendered as a topological graph that preserves the geometry of the underlying arbor. Resting microglia display dense, highly branched networks with numerous bifurcations and long‑range connections, reflecting their complex surveillance morphology. Primed and resolving microglia exhibit intermediate graph density and branching, consistent with partial process retraction or recovery. Activated microglia show reduced arborization with fewer nodes and shorter paths, while ameboid microglia present the most simplified graphs, often reduced to minimal linear or near‑linear structures. This visualization highlights how graph‑theoretic representations capture activation‑dependent remodeling of microglial topology and complement fractal, lacunarity, and texture‑based descriptors.


In [ ]:
# ============================================================
# Build Master DataFrame (df_master_basic)
# ============================================================
# This merges all morphometric tables into a single unified
# dataframe indexed by filename + cell_state.
# ============================================================

# Start with fractal dimension table
df_master_basic = df_fd.copy()

# Merge lacunarity
df_master_basic = df_master_basic.merge(df_lac, on=["filename", "cell_state"], how="left")

# Merge GLCM metrics
df_master_basic = df_master_basic.merge(df_glcm, on=["filename", "cell_state"], how="left")

# Merge Multiscale Fractal Spectrum
df_master_basic = df_master_basic.merge(df_mfs, on=["filename", "cell_state"], how="left")

# Merge graph-based morphometrics
df_master_basic = df_master_basic.merge(df_graph, on=["filename", "cell_state"], how="left")

# ------------------------------------------------------------
# EXPORT TABLE (Build Master DataFrame — df_master_basic)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_master_basic.to_csv(os.path.join(EXPORT_DIR, "Module 7 - build_master_dataframe_df_master_basic.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_master_basic 

**Table 6. Integrated multimodal feature matrix combining geometric, fractal, textural, multiscale, and graph‑theoretic descriptors of microglial morphology.**  
This table consolidates all quantitative descriptors extracted from each microglial sample, providing a unified feature matrix spanning multiple morphological domains. Fractal dimension and multiscale fractal spectrum (MFS) capture geometric and hierarchical complexity, with resting microglia showing the highest FD and most negative MFS values, reflecting dense, multiscale arborization. Lacunarity quantifies spatial heterogeneity, peaking in ameboid and activated states and reaching its lowest values in resting cells. GLCM texture metrics (contrast, homogeneity, energy, correlation) reveal complementary intensity‑based signatures, with resting microglia exhibiting high contrast and energy, while ameboid cells show elevated correlation and reduced homogeneity. Graph‑theoretic metrics derived from skeletons (endpoints, bifurcations, nodes, edges, total length, diameter, average degree) further distinguish states: resting microglia form large, highly branched networks, whereas ameboid cells reduce to minimal, near‑linear structures. Together, these multimodal features provide a comprehensive quantitative fingerprint of microglial activation, capturing structural remodeling across geometric, textural, and topological scales.


In [ ]:
# Use the master dataframe for graph-based barplots
df_graph_diag = df_master_basic.copy()

# Ensure a valid cell_id exists
if "cell_id" not in df_graph_diag.columns:
    if "filename" in df_graph_diag.columns:
        df_graph_diag["cell_id"] = df_graph_diag["filename"]
    else:
        df_graph_diag["cell_id"] = df_graph_diag.index.astype(str)

# ============================================================
# Comparative Plots of Graph Morphometrics
# ============================================================

metrics = [
    "endpoints",
    "bifurcations",
    "endpoint_density",
    "bifurcation_density",
    "branching_index",
    "avg_junction_degree",
    "diameter",
    "total_nodes",
    "total_length"
]

# Ensure consistent naming
if "stage" in df_graph_diag.columns and "cell_state" not in df_graph_diag.columns:
    df_graph_diag["cell_state"] = df_graph_diag["stage"]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    plot_on_axis_bars_or_boxplot(
        ax=axes[i],
        df=df_graph_diag,
        value_col=metric,
        group_col="cell_state",
        filename_col="cell_id",
        state_order=state_order,
        ylabel="Value",
        title=metric.replace("_", " ").capitalize()
    )

# Hide unused subplots
for j in range(len(metrics), len(axes)):
    axes[j].axis("off")

plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Comparative Plots of Graph Morphometrics)
# ------------------------------------------------------------
plt.savefig(
    os.path.join(EXPORT_DIR, "Module 7 - comparative_plots_of_graph_morphometrics_microglia.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

**Figure 13. Comparative graph‑theoretic morphometrics across microglial activation states.**  
This figure summarizes seven graph‑based descriptors extracted from skeletonized microglial arbors, enabling direct comparison of topological and geometric organization across activation states. Resting microglia consistently exhibit the highest values for endpoints, bifurcations, total nodes, total edges, total branch length, and diameter, reflecting their dense, highly ramified surveillance morphology. Primed and resolving microglia occupy an intermediate range, showing partial reductions in branching complexity and arbor extent consistent with transitional remodeling. Activated microglia display markedly reduced graph complexity, while ameboid microglia show the most simplified structures, often reduced to minimal node–edge configurations with short diameters and low average degree. Together, these barplots highlight the progressive collapse of microglial arbor topology during activation and the sensitivity of graph‑theoretic metrics to structural remodeling across functional states.


In [ ]:
# Use the master dataframe for group-level graph morphometrics
df_graph_diag = df_master_basic.copy()

# ============================================================
# Group-Level Comparison of Graph Morphometrics
# Universal version: works with individual images or groups
# ============================================================

metrics = [
    "endpoints",
    "bifurcations",
    "total_nodes",
    "total_edges",
    "total_length",
    "diameter",
    "avg_degree"
]

# Ensure consistent naming
if "stage" in df_graph_diag.columns and "cell_state" not in df_graph_diag.columns:
    df_graph_diag["cell_state"] = df_graph_diag["stage"]

# Keep only metrics that exist in the dataframe
metrics_existing = [m for m in metrics if m in df_graph_diag.columns]

# Remove metrics that are completely empty
metrics_existing = [
    m for m in metrics_existing
    if not df_graph_diag[m].dropna().empty
]

# Identify groups in the same order as the dataset loader
groups = [g for g in state_order if g in df_graph_diag["cell_state"].unique()]
groups += [g for g in df_graph_diag["cell_state"].unique() if g not in groups]

# Deterministic color map per group
color_map = {state: plt.cm.tab10(i % 10) for i, state in enumerate(groups)}

plt.figure(figsize=(12, 6))

for state in groups:
    subset = df_graph_diag[df_graph_diag["cell_state"] == state]

    if subset.empty:
        continue

    mean_vals = subset[metrics_existing].mean(numeric_only=True)

    plt.plot(
        metrics_existing,
        mean_vals[metrics_existing],
        marker="o",
        label=state,
        color=color_map[state]
    )

# Decide automatically whether log scale is useful
all_values = df_graph_diag[metrics_existing].values.flatten()
all_values = all_values[np.isfinite(all_values)]
all_values = all_values[all_values > 0]

if len(all_values) > 0:
    value_ratio = all_values.max() / all_values.min()

    if value_ratio > 100:
        plt.yscale("log")
        ylabel = "Mean value (log scale)"
    else:
        ylabel = "Mean value"
else:
    ylabel = "Mean value"

plt.xticks(rotation=45, ha="right")
plt.ylabel(ylabel)
plt.title("Graph Morphometrics — Group-Level Comparison")
plt.legend(title="Group", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(EXPORT_DIR, "Module 7 - group_level_comparison_of_graph_morphometrics.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("\n[INFO] ===== Module 7 Completed =====")

**Figure 14. Group‑level comparison of graph‑theoretic morphometrics across microglial activation states.**  
This figure summarizes the mean values of seven graph‑based descriptors—endpoints, bifurcations, total nodes, total edges, total branch length, diameter, and average degree—computed for each microglial activation state. Resting microglia exhibit the highest values across nearly all metrics, reflecting their dense, extensively branched surveillance morphology. Primed and resolving microglia occupy an intermediate position, showing partial reductions in branching complexity and arbor extent consistent with transitional remodeling. Activated microglia display further reductions in graph complexity, while ameboid microglia show the lowest values across all descriptors, indicating a collapse of arbor structure into minimal, compact forms. The clear separation between states demonstrates that graph‑theoretic metrics capture robust, state‑dependent differences in microglial topology and complement fractal, lacunarity, and texture‑based analyses.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## Module 8 — Sholl Analysis

This module performs Sholl analysis on skeletonized microglial morphologies to quantify how branching complexity varies with radial distance from a central reference point.
Within this pipeline, the analysis is applied to the visual skeleton generated during preprocessing, providing a consistent topological representation across all images.

Because soma segmentation may be unreliable or unavailable in some datasets, the module uses the centroid of the skeleton as a universal and reproducible reference point for radial sampling.

---

### **Conceptual relevance in microglia**

Sholl analysis captures the radial distribution of branching complexity, complementing the descriptors computed in previous modules:

- fractal dimension
- lacunarity
- GLCM texture metrics
- multiscale fractal spectrum
- graph-based morphometrics

In microglial morphology, this is particularly useful because activation-dependent remodeling often changes not only total complexity, but also how that complexity is distributed around the cell center.

Typical tendencies may include:

- Resting microglia (higher intersection counts across broader radial ranges)
- Primed microglia (reduced distal complexity with a more inward profile)
- Activated microglia (compact curves with lower AUC and steeper decline)
- Ameboid microglia (minimal radial structure and near-flat Sholl profiles)
- Resolving microglia (partial re-emergence of distal intersections)

These should be interpreted as biologically meaningful trends rather than fixed signatures.

---

### **Centroid as a universal reference point**

The function skeleton_centroid(skel) computes the centroid of the skeleton by averaging the coordinates of all foreground skeleton pixels.

This strategy provides:

- independence from explicit soma segmentation
- robustness across imaging conditions
- consistency across activation states and datasets

If the skeleton is empty, the centroid is returned as (None, None) and downstream Sholl outputs are safely set to default empty or null values.

---

### **Sholl computation**

The function sholl_analysis(skel, step=5, max_radius=None) performs the radial analysis.

For each image:

- The skeleton is converted to boolean format.
- The skeleton centroid is computed.
- A sequence of radii is generated with a default step size of 5 pixels.
- If max_radius is not provided, the maximum radius is set to the image diagonal.
- For each radius, a thin circular band is defined around the centroid.
- The number of skeleton pixels lying on that circular band is counted.

This produces the Sholl curve, i.e. intersections as a function of radius.

In computational terms, intersections are identified by selecting skeleton pixels satisfying:

∣ 𝑑(𝑥,𝑦)−𝑟 ∣ <0.5

where d(x,y) is the Euclidean distance from the centroid and r is the current radius.

---

### **Summary descriptors extracted**

For each skeleton, the module extracts:

max_intersections — the maximum value of the Sholl curve
radius_of_max — the radius at which the maximum occurs
auc — area under the Sholl curve, computed by trapezoidal accumulation
decay_rate — slope of a linear fit between radius and intersections
center_x, center_y — centroid coordinates used as the radial anchor

If the skeleton is empty or no valid intersections are found, safe default values are returned.

---

### **Outputs generated**

This module generates several outputs.

1. Full Sholl table

A dataframe is built containing, for each image:

- filename
- cell_state
- max_intersections
- radius_of_max
- auc
- decay_rate
- radii
- intersections

This full table preserves both the summary descriptors and the complete Sholl curve data.

It is exported as:

'Module 8 - batch_sholl_analysis_across_all_microglial_samples.csv'

2. Sholl summary table

A reduced summary table is then created by removing the full curve arrays (radii, intersections) and keeping only the scalar descriptors.

It is exported as:

'Module 8 - sholl_summary_table.csv'

3. Metric plots

For each summary descriptor:

- max_intersections
- radius_of_max
- auc
- decay_rate

the module generates ordered barplots or boxplots using the shared plotting utility.

These plots provide a compact comparison of Sholl-derived metrics across microglial states.

4. Sholl overlays

The module also creates visual overlays for all samples showing:

- the skeleton in grayscale
- the centroid in yellow
- the sampled concentric circles in cyan
- detected intersection pixels in red

This visualization helps verify that the radial sampling is behaving as expected.

The overlay figure is exported as:

'Module 8 - sholl_overlay_for_all_microglial_samples.png'

---

### **Universal master morphometric dataframe**

In addition to Sholl analysis itself, this module builds a unified master morphometric dataframe by merging all feature tables available in the workspace, including:

- Sholl summary metrics
- fractal dimension
- lacunarity
- GLCM metrics
- multiscale fractal spectrum
- graph-based morphometrics

To ensure compatibility across modules, identifier and group columns are automatically standardized to:

- cell_id
- cell_state

The merged table is exported as:

'Module 8 - build_master_morphometric_dataframe_universal.csv'

This dataframe provides a unified feature matrix for downstream statistical analysis, visualization, and machine learning.

---

### **Role in the pipeline**

This module adds a radial perspective to the morphometric workflow.
While previous modules quantify global complexity, texture, topology, and multiscale organization, Sholl analysis captures how branching density is distributed as a function of distance from the morphological center.

Together with the master dataframe generated here, this module helps consolidate the full quantitative representation of microglial morphology across activation states and experimental conditions.

In [ ]:
# ============================================================
# Skeleton Centroid
# ============================================================
# Computes the centroid of a skeletonized morphology.
# Used as the universal anchor for Sholl analysis.
# ============================================================

print("\n[INFO] ===== Module 8 - Batch Sholl Analysis =====")

def skeleton_centroid(skel):
    skel = skel.astype(bool)

    ys, xs = np.where(skel)

    if len(xs) == 0:
        return None, None

    cx = xs.mean()
    cy = ys.mean()

    return cx, cy

# ============================================================
# Sholl Analysis
# ============================================================

def sholl_analysis(skel, step=10, max_radius=None):
    """
    Perform Sholl analysis on a skeletonized microglial morphology.
    """
    skel = skel.astype(bool)
    h, w = skel.shape

    cx, cy = skeleton_centroid(skel)

    if cx is None or cy is None:
        return {
            "radii": np.array([]),
            "intersections": np.array([]),
            "max_intersections": 0,
            "radius_of_max": np.nan,
            "auc": 0.0,
            "decay_rate": np.nan,
            "center_x": np.nan,
            "center_y": np.nan
        }

    ys, xs = np.where(skel)

    if max_radius is None:
        distances_to_skeleton = np.sqrt((xs - cx)**2 + (ys - cy)**2)
        max_radius = int(np.ceil(distances_to_skeleton.max()))

    radii = np.arange(step, max_radius, step)
    intersections = []

    yy, xx = np.indices(skel.shape)

    for r in radii:
        circle = np.abs(np.sqrt((xx - cx)**2 + (yy - cy)**2) - r) < 0.5
        intersec = np.logical_and(circle, skel).sum()
        intersections.append(intersec)

    intersections = np.array(intersections)

    if len(intersections) == 0:
        return {
            "radii": radii,
            "intersections": intersections,
            "max_intersections": 0,
            "radius_of_max": np.nan,
            "auc": 0.0,
            "decay_rate": np.nan,
            "center_x": cx,
            "center_y": cy
        }

    max_intersections = intersections.max()
    radius_of_max = radii[intersections.argmax()]
    auc = float(np.sum((intersections[1:] + intersections[:-1]) * np.diff(radii) / 2))

    if len(radii) >= 2:
        coeffs = np.polyfit(radii, intersections, 1)
        decay_rate = coeffs[0]
    else:
        decay_rate = np.nan

    return dict(
        radii=radii,
        intersections=intersections,
        max_intersections=max_intersections,
        radius_of_max=radius_of_max,
        auc=auc,
        decay_rate=decay_rate,
        center_x=cx,
        center_y=cy
    )

# ============================================================
# Batch Sholl Analysis Across All Microglial Samples
# ============================================================

sholl_rows = []

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        data = preprocessed_data[path]

        skel = data["skeleton_visual_uint8"].astype(bool)

        sh = sholl_analysis(skel)

        sholl_rows.append({
            "filename": data["filename"],
            "sample_id": data["sample_id"],
            "cell_state": data["cell_state"],
            "max_intersections": sh["max_intersections"],
            "radius_of_max": sh["radius_of_max"],
            "auc": sh["auc"],
            "decay_rate": sh["decay_rate"],
            "radii": sh["radii"],
            "intersections": sh["intersections"]
        })

    except Exception as e:
        print("Error processing:", path, e)
        continue

df_sholl = pd.DataFrame(sholl_rows)

# ============================================================
# Sholl Wide Table (one column per radius)
# Required for Mixed ANOVA in Module 9
# ============================================================

# Collect all radii found across all cells
all_sholl_radii = sorted({
    int(r)
    for radii in df_sholl["radii"]
    for r in radii
})

sholl_wide_rows = []

for _, row in df_sholl.iterrows():
    entry = {
        "filename": row["filename"],
        "cell_state": row["cell_state"]
    }

    # Create a dictionary radius -> intersections
    radius_to_value = {
        int(r): float(v)
        for r, v in zip(row["radii"], row["intersections"])
    }

    # Use the same radius columns for every cell
    # Missing radii are filled with 0 intersections.
    for r in all_sholl_radii:
        entry[f"sholl_{r}"] = radius_to_value.get(r, 0.0)

    sholl_wide_rows.append(entry)

df_sholl_wide = pd.DataFrame(sholl_wide_rows)

# Uses dot decimal and comma separator.
df_sholl_wide.to_csv(
    os.path.join(EXPORT_DIR, "Module 8 - sholl_wide_table.csv"),
    index=False
)

df_sholl_wide

# ------------------------------------------------------------
# EXPORT TABLE (Batch Sholl Analysis Across All Microglial Samples)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_sholl.to_csv(os.path.join(EXPORT_DIR, "Module 8 - batch_sholl_analysis_across_all_microglial_samples.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_sholl

**Table 7. Sholl‑based branching metrics quantify radial arbor complexity across microglial activation states.**  
This table summarizes Sholl analysis descriptors for each microglial sample, capturing how branching complexity is distributed across increasing radial distances from the soma. Resting microglia exhibit the highest maximum number of intersections, the largest radii of maximal branching, and the greatest area under the curve (AUC), reflecting their expansive, highly ramified arbors that extend across wide spatial domains. Their strongly negative decay rates further indicate a gradual decline in branching density with distance, characteristic of complex surveillance morphologies. Primed and resolving microglia show intermediate Sholl profiles, with moderate peak intersections and AUC values consistent with partial process retraction or recovery. Activated microglia display reduced branching concentrated near the soma, while ameboid microglia show minimal or absent intersections across all radii, reflecting their collapsed, non‑ramified morphology. Together, these Sholl‑derived metrics provide a sensitive radial signature of activation‑dependent structural remodeling and complement fractal, texture, and graph‑theoretic descriptors.


In [ ]:
# ============================================================
# Sholl Summary Table
# ============================================================

df_sholl_summary = df_sholl.drop(columns=["radii", "intersections"])

# ------------------------------------------------------------
# EXPORT TABLE (Sholl Summary Table)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_sholl_summary.to_csv(os.path.join(EXPORT_DIR, "Module 8 - sholl_summary_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_sholl_summary

**Table 8. Summary Sholl metrics quantify radial branching complexity across microglial activation states.**  
This table presents four key descriptors derived from Sholl analysis—maximum number of intersections, radius of maximal branching, area under the curve (AUC), and decay rate—for each microglial sample. Resting microglia exhibit the highest peak intersections, the largest radii of maximal branching, and the greatest AUC values, reflecting their expansive, highly ramified arbors that extend across wide spatial domains. Their strongly negative decay rates indicate a gradual decline in branching density with distance, characteristic of complex surveillance morphologies. Primed and resolving microglia show intermediate Sholl signatures, with moderate peak intersections and AUC values consistent with partial process retraction or recovery. Activated microglia display reduced branching concentrated near the soma, while ameboid microglia show minimal or absent intersections and near‑zero AUC, reflecting their collapsed, non‑ramified morphology. Together, these summary metrics provide a compact radial fingerprint of activation‑dependent structural remodeling and complement fractal, texture, and graph‑theoretic descriptors.


In [ ]:
metrics = ["max_intersections", "radius_of_max", "auc", "decay_rate"]

for metric in metrics:
    plot_bars_or_boxplot(
        df=df_sholl_summary,
        value_col=metric,
        group_col="cell_state",
        filename_col="sample_id",
        state_order=state_order,
        ylabel=metric.replace("_", " ").capitalize(),
        xlabel="Image Filename / Group",
        title=f"Sholl Metric — {metric.replace('_', ' ').capitalize()}",
        save_path=os.path.join(EXPORT_DIR, f"Module 8 - sholl_{metric}.png")
    )

**Figure 15. Group‑level comparison of Sholl‑derived branching metrics across microglial activation states.**  
This figure summarizes four key Sholl descriptors—maximum number of intersections, radius of maximal branching, area under the curve (AUC), and decay rate—averaged across microglial activation states. Resting microglia exhibit the highest values for max intersections, radius of maximal branching, and AUC, reflecting their expansive, highly ramified arbors that extend across wide radial domains. Primed microglia show intermediate but elevated values, consistent with partially retracted yet still complex branching patterns. Resolving and activated microglia display reduced radial complexity, with lower peak intersections and smaller AUC values. Ameboid microglia show minimal or near‑zero branching across all metrics, indicating a collapsed, non‑ramified morphology. Decay rate follows the same gradient: resting and primed cells show the steepest negative slopes, while ameboid microglia approach zero, reflecting the absence of radial branching. Together, these group‑level Sholl metrics provide a compact radial signature of activation‑dependent morphological remodeling.


In [ ]:
# ============================================================
# Sholl Overlay for All Microglial Samples
# ============================================================

# Number of samples
n = len(all_paths)

# Dynamic grid layout
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for i, (path, cell_state) in enumerate(zip(all_paths, all_cell_states)):

    try:
        data = preprocessed_data[path]

        skel = data["skeleton_visual_uint8"].astype(bool)

        sh = sholl_analysis(skel)

        radii = sh["radii"]

        cx, cy = sh["center_x"], sh["center_y"]

        if np.isnan(cx) or np.isnan(cy):
            continue

        yy, xx = np.indices(skel.shape)
        intersections_mask = np.zeros_like(skel, dtype=bool)

        radii_to_plot = radii[::5]

        for r in radii_to_plot:
            circle = np.abs(np.sqrt((xx - cx)**2 + (yy - cy)**2) - r) < 0.5
            intersections_mask |= np.logical_and(circle, skel)

        ax = axes[i]
        ax.imshow(skel, cmap="gray")

        # Zoom around the skeleton so the cell is visible
        ys_skel, xs_skel = np.where(skel)

        if len(xs_skel) > 0:
            margin = 80

            x_min = max(xs_skel.min() - margin, 0)
            x_max = min(xs_skel.max() + margin, skel.shape[1])

            y_min = max(ys_skel.min() - margin, 0)
            y_max = min(ys_skel.max() + margin, skel.shape[0])

            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_max, y_min)

        ax.scatter([cx], [cy], c="yellow", s=40)

        theta = np.linspace(0, 2 * np.pi, 400)
        for r in radii_to_plot:
            x = cx + r * np.cos(theta)
            y = cy + r * np.sin(theta)
            ax.plot(x, y, color="cyan", alpha=0.15, linewidth=0.6)

        ys, xs = np.where(intersections_mask)
        ax.scatter(xs, ys, c="red", s=8)

        ax.set_title(f"{data['filename']} ({data['cell_state']})", fontsize=9)
        ax.axis("off")

    except Exception as e:
        print("Error processing:", path, e)
        continue

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Sholl Overlay for All Microglial Samples)
# ------------------------------------------------------------
plt.savefig(os.path.join(EXPORT_DIR, "Module 8 - sholl_overlay_for_all_microglial_samples.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show() 

**Figure 16. Visual overview of microglial morphological diversity across activation states.**  
This figure presents a gallery of representative microglial cells spanning the five activation states analyzed in this study: activated, ameboid, primed, resolving, and resting. Each circular thumbnail displays the processed morphology of a single cell, allowing rapid visual comparison of structural complexity across states. Resting microglia exhibit dense, highly ramified arbors characteristic of their surveillance phenotype. Primed and resolving microglia show intermediate branching patterns, reflecting transitional remodeling during activation or recovery. Activated microglia display reduced arborization with shorter, thicker processes, while ameboid microglia appear as compact, non‑ramified structures typical of fully activated, phagocytic states. This visual panel provides an intuitive qualitative complement to the quantitative descriptors presented in earlier figures and tables, highlighting the pronounced morphological gradients that define microglial activation.


In [ ]:
# ============================================================
# Build Master Morphometric DataFrame
# ============================================================

# ============================================================
# Pandas display settings
# ============================================================
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# ============================================================
# Identify all feature dataframes available in the workspace
# ============================================================

feature_dfs = []

feature_candidates = [
    ("df_sholl_summary", globals().get("df_sholl_summary")),
    ("df_sholl_wide", globals().get("df_sholl_wide")),
    ("df_fd", globals().get("df_fd")),
    ("df_lac", globals().get("df_lac")),
    ("df_glcm", globals().get("df_glcm")),
    ("df_mfs", globals().get("df_mfs")),
    ("df_graph", globals().get("df_graph")),
]

for name, df in feature_candidates:
    if df is not None and not df.empty:
        feature_dfs.append(df.copy())

if len(feature_dfs) == 0:
    raise ValueError("No feature dataframes found. Please run Modules 1–8 first.")

# ============================================================
# Utility: detect ID and state columns
# ============================================================

def detect_id_column(df):
    # Prefer filename because it is shared by all feature tables.
    # sample_id is useful for display, but causes duplicate columns during merges.
    candidates = ["cell_id", "filename", "image_id", "image_name", "id", "sample_id"]

    for c in candidates:
        if c in df.columns:
            return c

    raise KeyError("No ID column found in dataframe.")


def detect_state_column(df):
    candidates = ["cell_state", "state", "group", "label", "phenotype"]

    for c in candidates:
        if c in df.columns:
            return c

    raise KeyError("No cell_state column found in dataframe.")

# ============================================================
# Standardize column names and remove auxiliary duplicate IDs
# ============================================================

clean_dfs = []
sample_id_maps = []

for df in feature_dfs:
    df = df.copy()

    id_col = detect_id_column(df)
    state_col = detect_state_column(df)

    df.rename(columns={id_col: "cell_id", state_col: "cell_state"}, inplace=True)

    # Save sample_id once, only for final display
    if "sample_id" in df.columns:
        tmp_sample = df[["cell_id", "cell_state", "sample_id"]].copy()
        tmp_sample = tmp_sample.dropna(subset=["cell_id", "cell_state"])
        tmp_sample = tmp_sample.drop_duplicates(subset=["cell_id", "cell_state"])
        sample_id_maps.append(tmp_sample)

    # Remove auxiliary ID/display columns before merging
    # They cause sample_id_x / sample_id_y conflicts.
    auxiliary_id_cols = ["sample_id", "filename", "image_id", "image_name", "id"]

    for col in auxiliary_id_cols:
        if col in df.columns and col not in ["cell_id", "cell_state"]:
            df.drop(columns=[col], inplace=True)

    # Remove duplicated columns inside the same dataframe
    df = df.loc[:, ~df.columns.duplicated()]

    clean_dfs.append(df)

# ============================================================
# Merge all feature tables on cell_id + cell_state
# ============================================================

df_master = clean_dfs[0]

for df in clean_dfs[1:]:

    # Safety: remove repeated non-key columns before merging
    overlapping_cols = [
        c for c in df.columns
        if c in df_master.columns and c not in ["cell_id", "cell_state"]
    ]

    if overlapping_cols:
        df = df.drop(columns=overlapping_cols)

    df_master = df_master.merge(
        df,
        on=["cell_id", "cell_state"],
        how="outer"
    )

# ============================================================
# Add one clean sample_id column back, if available
# ============================================================

if len(sample_id_maps) > 0:
    df_sample_id = pd.concat(sample_id_maps, ignore_index=True)
    df_sample_id = df_sample_id.drop_duplicates(subset=["cell_id", "cell_state"])

    df_master = df_master.merge(
        df_sample_id,
        on=["cell_id", "cell_state"],
        how="left"
    )

# Remove duplicate columns as final safety
df_master = df_master.loc[:, ~df_master.columns.duplicated()]

# Sort columns
priority_cols = ["cell_id", "sample_id", "cell_state"]

cols = [c for c in priority_cols if c in df_master.columns] + \
       [c for c in df_master.columns if c not in priority_cols]

df_master = df_master[cols]

# ============================================================
# EXPORT
# ============================================================

# Uses dot decimal and comma separator.
df_master.to_csv(os.path.join(EXPORT_DIR, "Module 8 - build_master_morphometric_dataframe_universal.csv"), index=False)

df_master

print("\n[INFO] ===== Module 8 Completed =====")

**Table 9. Integrated multimodal feature set combining Sholl, texture, multiscale fractal, and graph‑theoretic descriptors of microglial morphology.**  
This table consolidates a comprehensive set of morphological descriptors for each microglial cell, integrating radial branching metrics (Sholl analysis), GLCM texture features, multiscale fractal spectrum (MFS) values, and graph‑theoretic properties derived from skeletonized arbors. Resting microglia exhibit the highest values for max intersections, radius of maximal branching, and AUC, alongside high contrast and strongly negative MFS values, reflecting their dense, multiscale arborization and rich textural variability. Primed and resolving microglia show intermediate signatures across all modalities, consistent with transitional remodeling. Activated microglia display reduced branching, lower MFS magnitude, and simplified graph structure, while ameboid microglia show minimal intersections, near‑zero AUC, shallow MFS curves, and extremely sparse graph topology. By integrating radial, textural, fractal, and topological descriptors into a single matrix, this feature set provides a multidimensional fingerprint of microglial activation state and captures structural remodeling across complementary spatial scales.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## Module 9 — Statistical Testing

This module performs the main inferential statistical analysis of the morphometric features extracted in the previous modules.
Using the unified feature table df_master, it evaluates whether the available quantitative descriptors differ across microglial activation states through an adaptive testing framework that accounts for distributional assumptions, variance structure, repeated-measures patterns, effect sizes, and multiple testing correction.

In addition to global statistical comparisons, the module also performs post-hoc pairwise testing for globally significant metrics when appropriate.

---

### **Scope of the analysis**

The statistical workflow operates on the numeric columns available in df_master, excluding identifier variables and separating metrics according to their structure.

Two main analysis paths are used:

- Between-group testing for standard scalar descriptors
- Repeated-measures testing for eligible Sholl-style radial variables

Metrics whose names match repeated-measures patterns such as:

- sholl_...
- mfs_scale_...

are excluded from the standard between-group pipeline and treated separately when the required data structure is available.

---

1. Between-group statistical decision tree

For each eligible numeric metric, the module first organizes the data by cell_state and removes missing values.
At least two groups with valid data are required for a metric to be tested.

- Normality testing

Normality is assessed separately within each group using the following rules:

- n < 3 → distribution cannot be reliably tested (insufficient_n)
- zero variance → normality is treated as non-evaluable (zero_variance)
- 3 ≤ n < 5000 → Shapiro–Wilk
- n ≥ 5000 → Lilliefors test for normality

A metric is considered normal only if all groups:

- have testable sample sizes
- do not have zero variance
- pass the corresponding normality test at the chosen alpha level

If any group fails or cannot be evaluated, the metric is routed to the non-parametric branch.

- Homogeneity of variances

If the metric is classified as normal, the module evaluates variance homogeneity using Levene’s test (centered on the median).

This step determines whether the parametric comparison should assume equal variances.

- Global group comparison

The appropriate global test is selected automatically according to:

- number of valid groups
- normality status
- equality of variances
- Two groups
- Student t-test → normal + equal variances
- Welch t-test → normal + unequal variances
- Mann–Whitney U → non-normal or non-evaluable normality
- More than two groups
- ANOVA → normal + equal variances
- Welch ANOVA → normal + unequal variances
- Kruskal–Wallis → non-normal or non-evaluable normality

Metrics with zero global variance or no usable values are skipped.

2. Effect size estimation

For each successful global test, the module also computes an effect size whenever possible:

- Student / Welch t-test → Cohen’s d
- Mann–Whitney U → rank-biserial correlation (RBC)
- ANOVA → eta squared
- Welch ANOVA → partial eta squared (np2) if available from pingouin
- Kruskal–Wallis → epsilon squared

This provides a complementary measure of effect magnitude beyond the p-value alone.

3. Repeated-measures branch for Sholl-type data

The module includes a dedicated repeated-measures branch for Sholl-style radial variables when these are available as multiple radius-specific columns (e.g., sholl_5, sholl_10, etc.).

If at least three such columns are present, the module:

- reshapes the data into long format
- runs a mixed ANOVA with:
- between-subject factor: cell_state
- within-subject factor: radius
- subject identifier: cell_id
- attempts Mauchly’s test of sphericity
- computes Greenhouse–Geisser and Huynh–Feldt epsilon values

The resulting repeated-measures summary is added to the statistical results table as a dedicated Sholl entry.

Because this branch depends on the presence of radius-wise Sholl columns, it is only executed when those variables are available in df_master.

4. Multiple testing correction

After global testing, the module applies Benjamini–Hochberg FDR correction to the set of valid global p-values.

This correction is applied only to metrics with non-missing p-values.
Metrics with NaN p-values are excluded from the correction procedure and retain missing adjusted values.

The corrected p-values are stored in the output column:

- p_adj

5. Output of the global statistical testing

The module assembles a consolidated results table containing, for each tested metric:

- metric name
- number of groups and observations
- groups included in the comparison
- normality decision
- normality test methods by group
- normality p-values by group
- Levene p-value
- equal-variance decision
- global statistical test applied
- test statistic
- raw p-value
- effect size
- effect size name
- FDR-adjusted p-value

For repeated-measures Sholl testing, additional fields may include:

- sphericity_p
- GG_epsilon
- HF_epsilon

This table is exported as:

'Module 9 - microglia_statistical_testing_results.csv'

6. Post-hoc pairwise comparisons

For metrics that satisfy all of the following:

- present in df_master
- not classified as repeated-measures metrics
- tested across more than two groups
- globally significant after BH/FDR correction

the module performs post-hoc pairwise comparisons using the test appropriate to the global result:

- ANOVA → Tukey HSD
- Welch ANOVA → Games–Howell
- Kruskal–Wallis → Dunn test with BH correction

For each pairwise comparison, the module records:

- metric
- global test
- group 1
- group 2
- mean difference
- post-hoc test applied
- unadjusted p-value when available
- adjusted p-value
- significance flag

These results are exported as:

'Module 9 - posthoc_pairwise_comparisons.csv'

---

### **Role in the pipeline**

This module provides the inferential statistical layer of the microglial morphometric workflow.
Where earlier modules quantify structural, topological, radial, textural, and multiscale properties, this module evaluates whether those descriptors differ significantly across activation states and identifies which group contrasts drive the observed effects.

By combining:

- assumption-aware global testing
- effect sizes
- FDR correction
- optional repeated-measures analysis
- post-hoc pairwise comparisons

the module establishes a rigorous statistical framework for interpreting the multidimensional morphometric differences observed across microglial phenotypes.

In [ ]:
# ============================================================
# Module 9 — Statistical Testing
# ============================================================
# This module performs:
#   - Between-group ANOVA / Welch / Kruskal–Wallis
#   - Mixed ANOVA (between × within) for Sholl metrics
#   - Mauchly’s test of sphericity
#   - Greenhouse–Geisser / Huynh–Feldt corrections
#
# SPECIALIZED FOR MICROGLIA:
#   - Uses cell_state as grouping factor
#   - Optimized for small datasets
#   - Compatible with df_master (universal feature table)
# ============================================================
# BETWEEN-GROUP PIPELINE
#   Normality:
#       - n < 3     -> insufficient_n -> treated as non-parametric
#       - 3 <= n < 5000  -> Shapiro-Wilk
#       - n >= 5000      -> Lilliefors
#       - zero variance  -> zero_variance -> treated as non-parametric
#
#   If normal:
#       - Levene for variance homogeneity
#       - 2 groups:
#           * equal variances    -> Student t-test
#           * unequal variances  -> Welch t-test
#       - >2 groups:
#           * equal variances    -> ANOVA
#           * unequal variances  -> Welch ANOVA
#
#   If non-normal:
#       - 2 groups  -> Mann–Whitney U
#       - >2 groups -> Kruskal–Wallis
#
# EFFECT SIZES
#   - Student/Welch t-test -> Cohen's d
#   - Mann–Whitney U       -> rank-biserial correlation (RBC)
#   - ANOVA                -> eta squared
#   - Welch ANOVA          -> np2 if available from pingouin
#   - Kruskal–Wallis       -> epsilon squared
#
# MULTIPLE TESTING
#   - Global p-values across metrics -> FDR Benjamini-Hochberg
#
# REPEATED MEASURES
#   - Sholl metrics handled separately with mixed ANOVA
# ============================================================

print("\n[INFO] ===== Module 9 - Statistical Testing =====")

warnings.filterwarnings("default")

# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------

if "df_master" not in globals():
    raise NameError("df_master not found. Please run the Master Builder module first.")

df_stats = df_master.copy()

GROUP_COL = "cell_state"
if GROUP_COL not in df_stats.columns:
    raise KeyError("cell_state column not found in df_master.")

if "cell_id" not in df_stats.columns:
    raise KeyError("cell_id column not found in df_master.")

# ------------------------------------------------------------
# Numeric metrics
# ------------------------------------------------------------
numeric_cols = [
    c for c in df_stats.select_dtypes(include=[np.number]).columns
    if c not in ["cell_id"]
]

# Repeated-measures patterns excluded from between-group tests
rm_patterns = ["sholl_", "mfs_scale_"]

def is_repeated_measures_metric(metric):
    return any(p in metric.lower() for p in rm_patterns)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def clean_group_data(df, metric, group_col):
    """
    Returns:
        valid_groups: list of group labels with at least 1 non-NA value
        data_by_group: dict[group] -> 1D numpy array
        n_groups: number of valid groups
        n_total: total non-missing observations
    """
    sub = df[[group_col, metric]].dropna().copy()
    valid_groups = []
    data_by_group = {}

    for g in sub[group_col].dropna().unique():
        vals = sub.loc[sub[group_col] == g, metric].dropna().values.astype(float)
        if len(vals) > 0:
            valid_groups.append(g)
            data_by_group[g] = vals

    return valid_groups, data_by_group, len(valid_groups), len(sub)

def group_normality_test(x):
    """
    Normality per group:
      - n < 3: cannot reliably test
      - zero variance: not evaluable
      - 3 <= n < 5000: Shapiro-Wilk
      - n >= 5000: Lilliefors
    """
    n = len(x)

    if n < 3:
        return np.nan, "insufficient_n"

    if np.nanstd(x) == 0:
        return np.nan, "zero_variance"

    if n < 5000:
        stat, p = shapiro(x)
        return p, "Shapiro-Wilk"
    else:
        stat, p = lilliefors(x, dist="norm")
        return p, "Lilliefors"

def evaluate_normality(data_by_group, alpha=0.05):
    """
    A metric is considered normal only if ALL groups:
      - have testable sample size (>= 3)
      - do not have zero variance
      - pass the relevant normality test
    """
    pvals = {}
    methods = {}
    normal = True

    for g, x in data_by_group.items():
        p, method = group_normality_test(x)
        pvals[g] = p
        methods[g] = method

        if np.isnan(p):
            normal = False
        elif p <= alpha:
            normal = False

    return normal, pvals, methods

def safe_eta_squared_from_anova(F_value, k, n_total):
    """
    eta^2 from one-way ANOVA F statistic
    """
    try:
        df_between = k - 1
        df_within = n_total - k
        if df_within <= 0:
            return np.nan
        return (F_value * df_between) / ((F_value * df_between) + df_within)
    except Exception:
        return np.nan

def safe_epsilon_squared_from_kruskal(H_value, k, n_total):
    """
    epsilon^2 for Kruskal-Wallis
    """
    try:
        denom = n_total - k
        if denom <= 0:
            return np.nan
        return (H_value - k + 1) / denom
    except Exception:
        return np.nan

# ------------------------------------------------------------
# 1. BETWEEN-GROUP TESTS
# ------------------------------------------------------------
results = []

for metric in numeric_cols:

    if is_repeated_measures_metric(metric):
        continue

    valid_groups, data_by_group, k, n_total = clean_group_data(df_stats, metric, GROUP_COL)

    # Need at least 2 groups with data
    if k < 2:
        warnings.warn(f"[{metric}] skipped: fewer than 2 groups with data.")
        continue

    # Avoid zero-variance total metrics that cannot be tested meaningfully
    try:
        all_values = np.concatenate([data_by_group[g] for g in valid_groups])
    except Exception as e:
        warnings.warn(f"[{metric}] skipped: could not concatenate values ({e}).")
        continue

    if len(all_values) == 0 or np.nanstd(all_values) == 0:
        warnings.warn(f"[{metric}] skipped: zero global variance or no valid values.")
        continue

    # ----------------------------
    # Normality
    # ----------------------------
    normal, norm_pvals, norm_methods = evaluate_normality(data_by_group, alpha=0.05)

    normality_method_summary = "; ".join(
        [f"{g}:{norm_methods[g]}" for g in valid_groups]
    )
    normality_p_summary = "; ".join(
        [f"{g}:{'nan' if pd.isna(norm_pvals[g]) else round(norm_pvals[g], 6)}" for g in valid_groups]
    )

    # ----------------------------
    # If normal -> Levene
    # ----------------------------
    lev_p = np.nan
    equal_var = np.nan

    if normal:
        try:
            lev_stat, lev_p = levene(*[data_by_group[g] for g in valid_groups], center="median")
            equal_var = bool(lev_p > 0.05)
        except Exception as e:
            warnings.warn(f"[{metric}] Levene test failed: {e}")
            lev_p = np.nan
            equal_var = False

    # ----------------------------
    # Choose test based on k
    # ----------------------------
    test_name = None
    stat = np.nan
    p = np.nan
    effect_size = np.nan
    effect_size_name = np.nan

    try:
        # ====================================================
        # TWO GROUPS
        # ====================================================
        if k == 2:
            g1, g2 = valid_groups
            x1 = data_by_group[g1]
            x2 = data_by_group[g2]

            if normal:
                if equal_var:
                    test_name = "Student t-test"
                    stat, p = ttest_ind(x1, x2, equal_var=True, nan_policy="omit")
                else:
                    test_name = "Welch t-test"
                    stat, p = ttest_ind(x1, x2, equal_var=False, nan_policy="omit")

                try:
                    effect_size = pg.compute_effsize(x1, x2, eftype="cohen")
                    effect_size_name = "cohen_d"
                except Exception as e:
                    warnings.warn(f"[{metric}] effect size computation failed: {e}")
                    effect_size = np.nan
                    effect_size_name = np.nan

            else:
                test_name = "Mann–Whitney U"
                stat, p = mannwhitneyu(x1, x2, alternative="two-sided")

                try:
                    mw = pg.mwu(x1, x2, alternative="two-sided")
                    if "RBC" in mw.columns:
                        effect_size = mw["RBC"].iloc[0]
                        effect_size_name = "rank_biserial_correlation"
                    else:
                        warnings.warn(
                            f"[{metric}] Mann–Whitney effect size column RBC not found. "
                            f"Available columns: {list(mw.columns)}"
                        )
                        effect_size = np.nan
                        effect_size_name = np.nan
                except Exception as e:
                    warnings.warn(f"[{metric}] Mann–Whitney effect size failed: {e}")
                    effect_size = np.nan
                    effect_size_name = np.nan

        # ====================================================
        # MORE THAN TWO GROUPS
        # ====================================================
        elif k > 2:
            arrays = [data_by_group[g] for g in valid_groups]

            if normal:
                if equal_var:
                    test_name = "ANOVA"
                    stat, p = f_oneway(*arrays)

                    try:
                        effect_size = safe_eta_squared_from_anova(stat, k, n_total)
                        effect_size_name = "eta_squared"
                    except Exception as e:
                        warnings.warn(f"[{metric}] ANOVA effect size failed: {e}")
                        effect_size = np.nan
                        effect_size_name = np.nan

                else:
                    test_name = "Welch ANOVA"
                    tmp = df_stats[[GROUP_COL, metric]].dropna().copy()
                    welch = pg.welch_anova(dv=metric, between=GROUP_COL, data=tmp)

                    # Normalize column names across pingouin versions
                    welch_cols_map = {c: c.lower().replace("-", "_") for c in welch.columns}

                    # Statistic
                    stat_found = False
                    for original_col, norm_col in welch_cols_map.items():
                        if norm_col in ["f", "f_val"]:
                            stat = welch[original_col].iloc[0]
                            stat_found = True
                            break

                    if not stat_found:
                        warnings.warn(
                            f"[{metric}] Welch ANOVA statistic extraction failed. "
                            f"Available columns: {list(welch.columns)}"
                        )

                    # p-value
                    p_found = False
                    for original_col, norm_col in welch_cols_map.items():
                        if norm_col in ["p_unc", "p", "pval", "p_value"]:
                            p = welch[original_col].iloc[0]
                            p_found = True
                            break

                    if not p_found or pd.isna(p):
                        warnings.warn(
                            f"[{metric}] Welch ANOVA p-value extraction failed. "
                            f"Available columns: {list(welch.columns)}"
                        )

                    # effect size
                    es_found = False
                    for original_col, norm_col in welch_cols_map.items():
                        if norm_col in ["np2", "eta2", "eta_squared"]:
                            effect_size = welch[original_col].iloc[0]
                            effect_size_name = norm_col
                            es_found = True
                            break

                    if not es_found:
                        warnings.warn(
                            f"[{metric}] Welch ANOVA effect size not found. "
                            f"Available columns: {list(welch.columns)}"
                        )
                        effect_size = np.nan
                        effect_size_name = np.nan

            else:
                test_name = "Kruskal–Wallis"
                stat, p = kruskal(*arrays)

                try:
                    effect_size = safe_epsilon_squared_from_kruskal(stat, k, n_total)
                    effect_size_name = "epsilon_squared"
                except Exception as e:
                    warnings.warn(f"[{metric}] Kruskal effect size failed: {e}")
                    effect_size = np.nan
                    effect_size_name = np.nan

        # General warning if any test ran but p is still NaN
        if test_name is not None and pd.isna(p):
            warnings.warn(
                f"[{metric}] {test_name} returned p_value = NaN. "
                f"Check input data, group sizes, zero variance within groups, or library output format."
            )

    except Exception as e:
        test_name = "FAILED"
        stat = np.nan
        p = np.nan
        effect_size = np.nan
        effect_size_name = np.nan
        warnings.warn(f"[{metric}] test failed: {e}")

    results.append({
        "metric": metric,
        "n_groups": k,
        "n_total": n_total,
        "groups": " | ".join(map(str, valid_groups)),
        "normal": normal,
        "normality_method": normality_method_summary,
        "normality_p_by_group": normality_p_summary,
        "levene_p": lev_p,
        "equal_variance": equal_var,
        "test": test_name,
        "statistic": stat,
        "p_value": p,
        "effect_size": effect_size,
        "effect_size_name": effect_size_name
    })

# ------------------------------------------------------------
# 2. MIXED ANOVA FOR SHOLL
# ------------------------------------------------------------
sholl_cols = [c for c in numeric_cols if "sholl_" in c.lower()]

if len(sholl_cols) >= 3:
    try:
        df_sholl_rm = df_stats[["cell_id", GROUP_COL] + sholl_cols].copy()

        df_long = df_sholl_rm.melt(
            id_vars=["cell_id", GROUP_COL],
            value_vars=sholl_cols,
            var_name="radius",
            value_name="value"
        ).dropna()

        df_long["radius_num"] = df_long["radius"].str.extract(r"(\d+)").astype(float)

        mix = pg.mixed_anova(
            dv="value",
            within="radius_num",
            between=GROUP_COL,
            subject="cell_id",
            data=df_long
        )

        if "Source" in mix.columns:
            interaction_rows = mix[mix["Source"].astype(str).str.contains(r"\*", regex=True, na=False)]
            if not interaction_rows.empty:
                mix_row = interaction_rows.iloc[0]
            else:
                mix_row = mix.iloc[0]
        else:
            mix_row = mix.iloc[0]

        try:
            mauchly = pg.sphericity(df_long, dv="value", subject="cell_id", within="radius_num")
            sphericity_p = mauchly[1] if isinstance(mauchly, (tuple, list)) and len(mauchly) > 1 else np.nan
        except Exception as e:
            warnings.warn(f"[Sholl] Mauchly sphericity test failed: {e}")
            sphericity_p = np.nan

        try:
            eps_raw = pg.epsilon(df_long, dv="value", subject="cell_id", within="radius_num")
            gg = np.nan
            hf = np.nan

            if isinstance(eps_raw, dict):
                gg = eps_raw.get("gg", np.nan)
                hf = eps_raw.get("hf", np.nan)
            elif isinstance(eps_raw, (list, tuple)) and len(eps_raw) >= 2:
                gg = eps_raw[0]
                hf = eps_raw[1]
        except Exception as e:
            warnings.warn(f"[Sholl] epsilon calculation failed: {e}")
            gg = np.nan
            hf = np.nan

        p_mix = (
            mix_row["p-unc"] if "p-unc" in mix_row.index else
            mix_row["p_unc"] if "p_unc" in mix_row.index else
            mix_row["p"] if "p" in mix_row.index else
            mix_row["pval"] if "pval" in mix_row.index else
            np.nan
        )

        np2_mix = mix_row["np2"] if "np2" in mix_row.index else np.nan
        F_mix = mix_row["F"] if "F" in mix_row.index else np.nan

        if pd.isna(p_mix):
            warnings.warn("[Sholl] Mixed ANOVA returned p_value = NaN.")

        results.append({
            "metric": "Sholl",
            "n_groups": df_long[GROUP_COL].nunique(),
            "n_total": len(df_long),
            "groups": " | ".join(map(str, df_long[GROUP_COL].dropna().unique())),
            "normal": np.nan,
            "normality_method": "repeated-measures module",
            "normality_p_by_group": np.nan,
            "levene_p": np.nan,
            "equal_variance": np.nan,
            "test": "Mixed ANOVA (between × within)",
            "statistic": F_mix,
            "p_value": p_mix,
            "effect_size": np2_mix,
            "effect_size_name": "np2",
            "sphericity_p": sphericity_p,
            "GG_epsilon": gg,
            "HF_epsilon": hf
        })

    except Exception as e:
        warnings.warn(f"[Sholl] Mixed ANOVA failed: {e}")
        results.append({
            "metric": "Sholl",
            "n_groups": np.nan,
            "n_total": np.nan,
            "groups": np.nan,
            "normal": np.nan,
            "normality_method": "repeated-measures module",
            "normality_p_by_group": np.nan,
            "levene_p": np.nan,
            "equal_variance": np.nan,
            "test": f"Mixed ANOVA FAILED: {str(e)}",
            "statistic": np.nan,
            "p_value": np.nan,
            "effect_size": np.nan,
            "effect_size_name": np.nan,
            "sphericity_p": np.nan,
            "GG_epsilon": np.nan,
            "HF_epsilon": np.nan
        })

# ------------------------------------------------------------
# 3. MULTIPLE TESTING CORRECTION BY METRIC FAMILY (BH/FDR)
# ------------------------------------------------------------
df_stats_results = pd.DataFrame(results)

def assign_metric_family(metric):
    """
    Assign each metric to a biologically interpretable family.
    FDR correction will be performed separately within each family.
    """
    m = str(metric).lower()

    # Sholl summary metrics
    if (
        m == "sholl"
        or "sholl" in m
        or m in ["max_intersections", "radius_of_max", "auc", "decay_rate"]
    ):
        return "sholl"

    # Fractal / lacunarity morphology
    if m in ["fractal_dimension", "lacunarity_mean"]:
        return "fractal_lacunarity"

    # Texture features
    if m in ["contrast", "homogeneity", "energy", "correlation"]:
        return "glcm_texture"

    # Skeleton / graph morphology
    if m in [
        "endpoints",
        "bifurcations",
        "total_nodes",
        "total_edges",
        "total_length",
        "diameter",
        "avg_degree",
        "avg_junction_degree",
        "endpoint_density",
        "bifurcation_density",
        "branching_index",
        "largest_component_nodes",
        "n_components"
    ]:
        return "graph_morphology"

    # Multiscale fractal spectrum
    if m.startswith("mfs_scale_"):
        return "mfs"

    # Fallback
    return "other"


if df_stats_results.empty:
    warnings.warn("No statistical results were generated.")
    df_stats_results["metric_family"] = np.nan
    df_stats_results["p_adj_global"] = np.nan
    df_stats_results["p_adj_family"] = np.nan
    df_stats_results["significant_global_fdr"] = False
    df_stats_results["significant_family_fdr"] = False

else:
    # Assign metric family
    df_stats_results["metric_family"] = df_stats_results["metric"].apply(assign_metric_family)

    valid_mask = df_stats_results["p_value"].notna()

    # --------------------------------------------------------
    # Optional: keep global FDR for reference
    # --------------------------------------------------------
    df_stats_results["p_adj_global"] = np.nan

    if valid_mask.any():
        df_stats_results.loc[valid_mask, "p_adj_global"] = multipletests(
            df_stats_results.loc[valid_mask, "p_value"],
            method="fdr_bh"
        )[1]
    else:
        warnings.warn("No valid p-values available for global BH/FDR correction.")

    # --------------------------------------------------------
    # Main correction: FDR within each metric family
    # --------------------------------------------------------
    df_stats_results["p_adj_family"] = np.nan

    for family, idx in df_stats_results[valid_mask].groupby("metric_family").groups.items():
        idx = list(idx)

        df_stats_results.loc[idx, "p_adj_family"] = multipletests(
            df_stats_results.loc[idx, "p_value"],
            method="fdr_bh"
        )[1]

    # Keep p_adj as the main column used downstream
    # Now p_adj means family-wise FDR-adjusted p-value
    df_stats_results["p_adj"] = df_stats_results["p_adj_family"]

    # Significance flags
    df_stats_results["significant_raw"] = df_stats_results["p_value"] < 0.05
    df_stats_results["significant_global_fdr"] = df_stats_results["p_adj_global"] < 0.05
    df_stats_results["significant_family_fdr"] = df_stats_results["p_adj_family"] < 0.05

    n_nan = (~valid_mask).sum()
    if n_nan > 0:
        warnings.warn(
            f"{n_nan} metrics had NaN p-values and were excluded from BH/FDR correction."
        )

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------
df_stats_results.to_csv(os.path.join(EXPORT_DIR, "Module 9 - microglia_statistical_testing_results.csv"), index=False)

pd.set_option("display.max_rows", None)
df_stats_results

**Table 10. Statistical tests identifying morphology metrics that differ significantly across microglial activation states.**  
This table reports the results of omnibus statistical tests (ANOVA, Welch ANOVA, and Kruskal–Wallis) applied to all extracted morphological descriptors to assess whether each metric varies across microglial activation states. Several features show highly significant differences after multiple‑comparison correction, including multiscale fractal spectrum values (notably mfs_scale_4 and mfs_scale_2), graph‑theoretic descriptors such as diameter, total nodes, total edges, and total length, as well as Sholl‑derived metrics including AUC and decay rate. Texture features (homogeneity, energy, correlation, contrast) also exhibit significant state‑dependent variation. Resting microglia consistently drive the strongest effects due to their high arbor complexity, while ameboid cells anchor the opposite extreme with minimal structure. Together, these results demonstrate that microglial activation states differ robustly across geometric, textural, multiscale, and topological dimensions, validating the discriminative power of the multimodal feature set.


In [ ]:
# ============================================================
# Post-hoc Pairwise Comparisons
# ============================================================

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

if "df_master" not in globals():
    raise NameError("df_master not found. Please run Module 8 or the Master Builder first.")

if "df_stats_results" not in globals():
    raise NameError("df_stats_results not found. Please run the previous statistics module first.")

df_stats = df_master.copy()

GROUP_COL = "cell_state"
if GROUP_COL not in df_stats.columns:
    raise KeyError("cell_state column not found in df_master.")

def is_repeated_measures_metric(metric):
    return any(p in str(metric).lower() for p in ["sholl_", "mfs_scale_"]) or str(metric) == "Sholl"

def normalize_test_name(x):
    return str(x).strip().lower().replace("–", "-").replace("—", "-")

pairwise_rows = []

for _, row in df_stats_results.iterrows():

    metric = row["metric"]
    global_test = row["test"]
    p_adj_family = row.get("p_adj_family", row.get("p_adj", np.nan))
    metric_family = row.get("metric_family", "unknown")
    n_groups = row.get("n_groups", np.nan)

    # Skip repeated-measures metrics
    if is_repeated_measures_metric(metric):
        continue

    # Metric must exist in df_master
    if metric not in df_stats.columns:
        continue

    # Post-hoc only makes sense for >2 groups
    if pd.isna(n_groups) or int(n_groups) <= 2:
        continue

    # Only metrics significant after BH/FDR correction within their metric family
    if pd.isna(p_adj_family) or p_adj_family >= 0.05:
        continue

    sub = df_stats[[GROUP_COL, metric]].dropna().copy()
    if sub.empty:
        continue

    if sub[GROUP_COL].nunique() < 2:
        continue

    values = sub[metric].values
    labels = sub[GROUP_COL].values
    test_clean = normalize_test_name(global_test)

    # --------------------------------------------------------
    # ANOVA -> Tukey HSD
    # --------------------------------------------------------
    if test_clean == "anova":
        try:
            tukey = pairwise_tukeyhsd(endog=values, groups=labels, alpha=0.05)

            for res in tukey.summary().data[1:]:
                g1, g2, meandiff, p_adj, lower, upper, reject = res

                pairwise_rows.append({
                    "metric": metric,
                    "metric_family": metric_family,
                    "global_test": global_test,
                    "group1": g1,
                    "group2": g2,
                    "mean_difference": meandiff,
                    "test": "Tukey HSD",
                    "p_unc": np.nan,
                    "p_adj": p_adj,
                    "significant": bool(reject)
                })

        except Exception as e:
            print(f"[POSTHOC ERROR] metric={metric} | test={global_test} | Tukey failed: {e}")

    # --------------------------------------------------------
    # Welch ANOVA -> Games-Howell
    # --------------------------------------------------------
    elif test_clean == "welch anova":
        try:
            gh = pg.pairwise_gameshowell(data=sub, dv=metric, between=GROUP_COL)

            pcol = "pval" if "pval" in gh.columns else ("p-unc" if "p-unc" in gh.columns else None)
            diffcol = "diff" if "diff" in gh.columns else None

            for _, r in gh.iterrows():
                pval = r[pcol] if pcol is not None else np.nan
                mdiff = r[diffcol] if diffcol is not None else np.nan

                pairwise_rows.append({
                    "metric": metric,
                    "metric_family": metric_family,
                    "global_test": global_test,
                    "group1": r["A"],
                    "group2": r["B"],
                    "mean_difference": mdiff,
                    "test": "Games-Howell",
                    "p_unc": np.nan,
                    "p_adj": pval,
                    "significant": bool(pval < 0.05) if pd.notna(pval) else False
                })

        except Exception as e:
            print(f"[POSTHOC ERROR] metric={metric} | test={global_test} | Games-Howell failed: {e}")

    # --------------------------------------------------------
    # Kruskal-Wallis -> Dunn + BH
    # --------------------------------------------------------
    elif test_clean == "kruskal-wallis":
        try:
            # Unadjusted Dunn
            dunn_unc = sp.posthoc_dunn(
                sub,
                val_col=metric,
                group_col=GROUP_COL,
                p_adjust=None
            )

            # BH-adjusted Dunn
            dunn_adj = sp.posthoc_dunn(
                sub,
                val_col=metric,
                group_col=GROUP_COL,
                p_adjust="fdr_bh"
            )

            groups = list(dunn_adj.index)

            for i in range(len(groups)):
                for j in range(i + 1, len(groups)):
                    g1 = groups[i]
                    g2 = groups[j]

                    p_unc = dunn_unc.loc[g1, g2]
                    p_adj = dunn_adj.loc[g1, g2]

                    mean_diff = (
                        sub.loc[sub[GROUP_COL] == g1, metric].mean()
                        - sub.loc[sub[GROUP_COL] == g2, metric].mean()
                    )

                    pairwise_rows.append({
                        "metric": metric,
                        "metric_family": metric_family,
                        "global_test": global_test,
                        "group1": g1,
                        "group2": g2,
                        "mean_difference": mean_diff,
                        "test": "Dunn (BH)",
                        "p_unc": p_unc,
                        "p_adj": p_adj,
                        "significant": bool(p_adj < 0.05) if pd.notna(p_adj) else False
                    })

        except Exception as e:
            print(f"[POSTHOC ERROR] metric={metric} | test={global_test} | Dunn failed: {e}")

df_posthoc = pd.DataFrame(pairwise_rows)

df_posthoc.to_csv(os.path.join(EXPORT_DIR, "Module 9 - posthoc_pairwise_comparisons.csv"), index=False)

pd.set_option("display.max_rows", None)

df_posthoc

print("\n[INFO] ===== Module 9 Completed =====")

**Table 11. Post‑hoc pairwise comparisons reveal specific morphological differences between microglial activation states.**  
This table reports pairwise contrasts between microglial activation states for all morphological metrics that showed significant omnibus effects. For each comparison, the mean difference, adjusted p‑value, and significance status are provided. Multiscale fractal spectrum values (e.g., mfs_scale_4, mfs_scale_2) show consistent and robust differences across nearly all state pairs, with resting microglia exhibiting markedly more negative values than activated and ameboid cells, reflecting their higher multiscale complexity. Graph‑theoretic descriptors such as diameter, total length, and total edges show strong contrasts particularly between resting and ameboid or activated states, highlighting the collapse of arbor topology during activation. Texture metrics (contrast, homogeneity, energy, correlation) also reveal selective differences, especially between resting and ameboid microglia. Sholl‑derived metrics (AUC, max intersections, decay rate) further distinguish resting and primed cells from ameboid and activated states. Together, these post‑hoc results identify the specific state‑to‑state transitions driving the global statistical effects reported in Table 9, confirming that microglial activation involves coordinated changes across geometric, textural, multiscale, and topological dimensions.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


### **Module 10 — Dimensionality Reduction & Feature Importance**

This module provides an exploratory and machine-learning–oriented view of the morphometric feature space.
While the previous statistical module evaluates individual descriptors one by one, this module examines the dataset at a more global level by asking:

- Do samples separate in low-dimensional morphometric space?
- Which features explain the largest fraction of overall variability?
- Which descriptors are most informative for distinguishing groups?

To address these questions, the module integrates three complementary components:

- Principal Component Analysis (PCA)
- UMAP embedding
- Random Forest feature importance

---

### **Automatic dataset and grouping detection**

The module begins by automatically detecting the master morphometric dataframe from a set of possible names:

- df_master
- df_all
- df_features
- df_ordered

It then automatically identifies the grouping variable from common candidate columns:

- cell_state
- condition
- group
- stage
- label

This makes the module flexible across different versions of the pipeline, provided the input dataframe follows the expected general structure.

---

### **Feature selection and preprocessing**

Only numeric morphometric features are used for dimensionality reduction and classification.

Identifier columns such as:

- cell_id
- image_id

are excluded from the feature matrix.

Before analysis:

- missing values are imputed using the median of each feature
- all numeric features are standardized using StandardScaler

This ensures that metrics with different scales contribute comparably to PCA, UMAP, and Random Forest.

---

## **Principal Component Analysis (PCA)**

The module applies PCA with two components to the standardized feature matrix.

PCA provides:

- a linear low-dimensional summary of the data
- the dominant axes of variation
- an interpretable representation of global morphometric structure

The module generates a 2D scatter plot of:

- PC1
- PC2

with points colored by the detected grouping variable.

Axis labels include the percentage of variance explained by each component.

In addition, the module computes PCA loadings for all numeric features on:

- PC1
- PC2

These loadings provide an interpretable measure of how strongly each feature contributes to the main axes of variation.

The PCA figure is exported as:

'Module 10 - pca.png'

---

### **UMAP**

To complement the linear PCA embedding, the module computes a UMAP representation of the same standardized feature matrix.

UMAP is a nonlinear dimensionality-reduction method that can reveal structure not captured by PCA, including:

- local neighborhood organization
- potential clustering patterns
- nonlinear relationships among morphometric descriptors

The embedding is computed using fixed parameters:

- n_neighbors = 15
- min_dist = 0.1
- metric = "euclidean"
- random_state = 42

The module generates a 2D UMAP scatter plot with points colored by group.

The UMAP figure is exported as:

'Module 10 - umap.png'

---

### **Random Forest Feature Importance**

The module then trains a Random Forest classifier to estimate how informative each morphometric feature is for predicting the group label.

The classifier is configured with:

- n_estimators = 500
- random_state = 42
- class_weight = "balanced"

This provides a supervised ranking of features based on their contribution to classification performance.

The resulting feature importances are stored in a table with:

- metric
- importance

and sorted in descending order.

For visualization, the module plots the top 20 most important features.

This figure is exported as:

'Module 10 - rf_top20.png'

---

### **Outputs generated**

This module produces the following main outputs:

- PCA scatter plot (PC1 vs PC2)
- PCA loadings for all numeric features (computed in memory)
- UMAP embedding plot
- Random Forest feature-importance ranking (computed in memory)
- Top-20 Random Forest importance plot

Exported figures:

'Module 10 - pca.png'
'Module 10 - umap.png'
'Module 10 - rf_top20.png'

---

### **Role in the pipeline**

This module complements the earlier inferential statistics by providing a global view of morphometric organization.

PCA summarizes dominant linear variation
UMAP reveals nonlinear structure in the feature space
Random Forest importance identifies the descriptors that are most useful for distinguishing groups

Together, these analyses provide an interpretable overview of how microglial states are organized in multidimensional morphometric space and which features contribute most strongly to that organization.

In [ ]:
# ============================================================
# Dimensionality Reduction & Feature Importance
# ============================================================

print("\n[INFO] ===== Module 10 - Dimensionality Reduction & Feature Importance =====")

# ------------------------------------------------------------
# 1. Auto-detect master dataframe
# ------------------------------------------------------------
possible_master_names = ["df_master", "df_all", "df_features", "df_ordered"]

df_ml = None
for name in possible_master_names:
    if name in globals():
        df_ml = globals()[name].copy()
        break

if df_ml is None:
    raise NameError("No master dataframe found. Please create df_master.")

# ------------------------------------------------------------
# 2. Auto-detect grouping column
# ------------------------------------------------------------
possible_group_cols = ["cell_state", "condition", "group", "stage", "label"]

GROUP_COL = None
for col in possible_group_cols:
    if col in df_ml.columns:
        GROUP_COL = col
        break

if GROUP_COL is None:
    raise KeyError("No grouping column found.")

# ------------------------------------------------------------
# 3. Select numeric morphometric metrics
# ------------------------------------------------------------
EXCLUDE_COLS = ["cell_id", "image_id"]

numeric_cols = [
    c for c in df_ml.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE_COLS
]

X_df = df_ml[numeric_cols].copy()
medians = X_df.median(numeric_only=True)
X_df = X_df.fillna(medians)
X = X_df.values
y = df_ml[GROUP_COL].values

# ------------------------------------------------------------
# 4. Standardize features
# ------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================================
# 10.1 PCA
# ============================================================

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_pca[:,0], y=X_pca[:,1],
    hue=y, palette="viridis", s=60, alpha=0.9
)
plt.title("PCA — PC1 vs PC2")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.legend(title=GROUP_COL)
plt.tight_layout()

plt.savefig(os.path.join(EXPORT_DIR, "Module 10 - pca.png"), dpi=600, bbox_inches="tight")
plt.show()

# PCA loadings
pca_loadings = pd.DataFrame(
    pca.components_.T,
    index=numeric_cols,
    columns=["PC1", "PC2"]
).sort_values("PC1", ascending=False)

# ============================================================
# 10.2 UMAP
# ============================================================

umap_model = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42
)

X_umap = umap_model.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_umap[:,0], y=X_umap[:,1],
    hue=y, palette="viridis", s=60, alpha=0.9
)
plt.title("UMAP Embedding")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend(title=GROUP_COL)
plt.tight_layout()

plt.savefig(os.path.join(EXPORT_DIR, "Module 10 - umap.png"), dpi=600, bbox_inches="tight")
plt.show()

# ============================================================
# 10.3 Random Forest Feature Importance
# ============================================================

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced"
)
rf.fit(X_scaled, y)

importances = pd.DataFrame({
    "metric": numeric_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8,10))
top_importances = importances.head(20)

sns.barplot(
    data=top_importances,
    x="importance",
    y="metric",
    hue="metric",
    palette="viridis",
    dodge=False,
    legend=False
)

plt.title("Top 20 Most Important Features (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Metric")
plt.tight_layout()

plt.savefig(os.path.join(EXPORT_DIR, "Module 10 - rf_top20.png"), dpi=600, bbox_inches="tight")
plt.show()

# ============================================================
# SHAP Interpretability
# ============================================================

# SHAP warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Use original feature names
feature_names = X_df.columns.tolist()

# SHAP explainer for tree-based models
explainer = shap.TreeExplainer(rf)

# Compute SHAP values
shap_values = explainer.shap_values(X_scaled)

# ------------------------------------------------------------
# Handle multiclass outputs robustly
# ------------------------------------------------------------

if isinstance(shap_values, list):
    # Convert to array: [n_classes, n_samples, n_features]
    shap_array = np.array(shap_values)

    # Mean absolute SHAP across classes and samples
    mean_abs_shap = np.mean(np.abs(shap_array), axis=(0, 1))

    # For summary plot: concatenate classes
    shap_for_plot = np.mean(np.abs(shap_array), axis=0)

else:
    shap_array = np.array(shap_values)

    if shap_array.ndim == 3:
        # [n_samples, n_features, n_classes]
        mean_abs_shap = np.mean(np.abs(shap_array), axis=(0, 2))
        shap_for_plot = np.mean(np.abs(shap_array), axis=2)
    else:
        # binary/single-output fallback
        mean_abs_shap = np.mean(np.abs(shap_array), axis=0)
        shap_for_plot = shap_array

# ------------------------------------------------------------
# Create SHAP table
# ------------------------------------------------------------
df_shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

# ------------------------------------------------------------
# SHAP bar plot
# ------------------------------------------------------------
plt.figure(figsize=(8, 10))
top_shap = df_shap_importance.head(20).sort_values("mean_abs_shap", ascending=True)

plt.barh(top_shap["feature"], top_shap["mean_abs_shap"])
plt.xlabel("Mean |SHAP value|")
plt.ylabel("Feature")
plt.title("Top 20 Features by SHAP Importance")
plt.tight_layout()
plt.savefig(
    os.path.join(EXPORT_DIR, "Module 10 - shap_top20_barplot.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# SHAP summary plot
# ------------------------------------------------------------
shap.summary_plot(
    shap_for_plot,
    X_df,
    show=False
)
plt.tight_layout()
plt.savefig(
    os.path.join(EXPORT_DIR, "Module 10 - shap_summary_plot.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

**Figure 17. Multidimensional embedding, machine-learning feature ranking, and SHAP interpretability analyses reveal robust separation of neural-cell states and identify the multimodal descriptors driving classification.**  
The first panel shows a PCA projection (PC1 vs. PC2), where samples cluster according to their experimental or phenotypic state, with PC1 capturing the dominant variance associated with global morphometric structure. The second panel displays a UMAP embedding that preserves local and global neighborhood relationships, highlighting separation and transitional continuity among cellular phenotypes. The third panel presents the top 20 most informative features identified by a Random Forest classifier, demonstrating that texture descriptors, graph-theoretic topology, Sholl metrics, and multiscale fractal measures jointly contribute to discrimination among cell states. The fourth panel shows the SHAP summary plot, illustrating how individual feature values increase or decrease classification predictions across samples. The fifth panel presents mean absolute SHAP values, providing a global ranking of feature contributions. Together, these analyses show that neural-cell populations occupy structured regions in morphometric space and that complementary descriptors spanning texture, topology, fractal geometry, and branching organization drive this separability.


In [ ]:
# ------------------------------------------------------------
# SHAP importance table
# ------------------------------------------------------------
df_shap_importance.to_csv(
    os.path.join(EXPORT_DIR, "Module 10 - shap_importance_table.csv"),
    index=False
)

print("\n[INFO] ===== Module 10 Completed =====")

print("\n[INFO] ===== Finished =====")

**Table 12. Global SHAP feature importance ranking for classification of neural-cell states.**  
Mean absolute SHAP values were computed from the Random Forest classifier to quantify the overall contribution of each morphometric descriptor to model predictions. Higher values indicate stronger influence on classification decisions. Features derived from texture, graph topology, Sholl branching organization, and multiscale fractal geometry ranked among the most informative predictors, confirming that cellular-state discrimination arises from complementary structural domains rather than a single metric family.

<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


### **Final Remarks — A Multiscale Microglial Morphometric Atlas in the Spirit of Cajal**

This notebook concludes a journey that echoes the tradition of Santiago Ramón y Cajal:
to observe, quantify, and ultimately understand the cell not as a static object,
but as a living architecture whose form reveals its function.

Across all modules, we have constructed a multiscale morphometric framework for microglia,
capable of capturing their remarkable structural plasticity — from fine-scale branching
to global geometric organization. What begins as raw descriptors evolves into a
coherent, interpretable, and biologically grounded analytical system.

Each component of the pipeline contributes a distinct layer of insight:

- Fractal, lacunarity, and texture metrics quantify internal structural complexity.
- Multiscale fractal spectra reveal how morphology is distributed across spatial scales.
- Graph-based morphometrics capture the topology of branching: endpoints, bifurcations, diameter, and connectivity.
- Sholl analysis characterizes the radial organization of microglial processes.
- Global statistical testing identifies robust differences across activation states.
- Post-hoc comparisons resolve the structure of these differences.
- PCA and UMAP uncover the geometry of morphometric space.
- Random Forest feature importance highlights the most discriminative descriptors.
- Reproducible exports ensure that all figures, tables, and results can be regenerated transparently.

Together, these modules form a reproducible and extensible morphometric ecosystem,
designed to be adaptable across datasets, experimental conditions, and imaging modalities.

At its core, this notebook reflects a central idea inherited from Cajal’s work:
that morphology encodes cellular identity and function.
Microglia — dynamic, plastic, and highly responsive — imprint their physiological state
onto the geometry of their processes, and this pipeline provides a way to quantify and interpret that structure.

This framework is not an endpoint.
It is a living computational system, ready to evolve with new datasets, new hypotheses,
and new questions in neuroimmunology. Every module can be re-executed from scratch,
ensuring full reproducibility and methodological transparency.

Rather than reducing microglia to isolated measurements, this workflow reveals a
structured morphometric landscape — a quantitative map of activation, complexity, and resolution.

<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">

### **Outlook — Beyond This Notebook**

Although this framework already captures multiple dimensions of microglial morphology,
it naturally extends toward several future directions:

- Temporal morphometrics (tracking morphological transitions over time)
- Spatial context integration (linking morphology to tissue microenvironments)
- Transcriptomic alignment (connecting morphometric signatures with gene-expression states)
- Deep learning embeddings (complementing handcrafted features with learned representations)
- Cross-species comparisons (testing conservation of microglial morphotypes across organisms)

These directions extend the framework beyond static analysis,
toward a more integrated understanding of microglial biology.

For now, this notebook stands as a complete, reproducible, and biologically grounded
pipeline for microglial morphometric analysis — a bridge between classical morphological insight and modern computational analysis.